# DME Express — Technician Workload Analytics
**Version v1.34.0 — 2026-08-10**

Changelog v1.34.0 (repository onboarding + credential hardening, session 2026-08-10):
- **Repo-aligned paths:** all outputs now land under `<repo>/outputs/tech_workload/<run-date>/`
  (git-ignored). The repo root is discovered automatically by walking up from the working
  directory to the folder containing `.git`/`CLAUDE.md` — no analyst-specific paths remain,
  so the same notebook runs unmodified for every analyst.
- **Credential file moved out of OneDrive:** `MSAIKey.env` now lives at
  `%USERPROFILE%\.dme-secrets\MSAIKey.env` — a local, non-synced folder. The old location
  (`OneDrive - DME Express\Documents\IT\Python`) synced the SQL password to the cloud on
  every change. Cell 3 detects the old location and prints migration instructions.
- **Entra ID interactive auth is now the default** (`AUTH_MODE='entra'`): the connection uses
  your own corporate sign-in (MFA included) — no stored password at all. SQL auth via the
  .env file is retained as `AUTH_MODE='sql'` fallback for accounts without an Entra grant.
- **Read-only guard in `run_query`:** statements that are not SELECT/CTE, or that contain
  write keywords, are refused in-process. NOTE: this is belt-and-suspenders for agent-driven
  runs — real enforcement is the db_datareader-only role on the account (verify with IT).
- **All cell outputs stripped** for repository commit (PHI hygiene: executed outputs embedded
  `record_id` values and tech names; the repo must carry code, never data).


Changelog v1.33.0 (from full code review, session 2026-07-31):
- **H1 FIX:** `tbl_tech_accountability` (+ monthly variant) merged redelivery tables on
  name only while both sides carried (tech, warehouse) grain — multi-warehouse techs
  fanned out and double-counted redeliveries. Merges now key on warehouse too.
- **H2 FIX:** `redel_per_100_visits` / `redel_pct_of_tickets` denominators now match the
  numerator grain (tech, warehouse) instead of company-wide totals per name.
- **H3 FIX:** SERP TRANSACTIONS WHERE now filters on `TRY_CONVERT(DATE, Completed_Date)`
  (raw-string compare could silently drop the last day / admit unparseable rows);
  PLC WorkDate filter is half-open (`< FILTER_END + 1 day`); NaT guard added.
- **M1 FIX:** Tech top/bottom ranking now ranks on `tickets_per_active_day`
  (denominator-honest for PTO / part-time / cross-coverage). Legacy calendar-weekday
  metric retained one quarter as `avg_daily_tickets_DEPRECATED`.
- **M2:** New `Redel_Unlinked_Monthly` sheet reconciles redeliveries whose originating
  order falls outside the ticket window (early-window undercount is now visible).
- **M4 FIX:** Service tickets sharing a (patient, tech, date) group with an exchange
  pair keep `visit_type='Service'` (were mislabeled 'Exchange'; counts were unaffected).
- **L1:** `affected_techs` counts distinct full names. **L2:** `clean_numbers` keeps
  negative signs. **L3:** payroll day-of-week now `DATEDIFF(day,'1900-01-01',d) % 7`
  (Monday=0) — immune to `@@DATEFIRST`. **L4:** dead `USER_ROOT` line removed.
  **L5:** warehouse trend chart titles renamed to the plotted metric.
- **NEW N1 — Overtime:** FLSA-style weekly OT inferred from PLC daily hours
  (Sun–Sat workweek, >40 hrs). Computed per employee-week on **raw, un-prorated**
  hours (FLSA applies to the person, not the warehouse), attributed to the month of
  the week-ending Saturday. *Caveat: if PLC `Hours` includes PTO, OT is overstated —
  PTO does not count toward the 40-hour threshold; verify with payroll.*
- **NEW N2 — Lost-recovery scorecard:** three definitions as separate columns:
  (a) all ATI.Lost items via last-delivery proximity; (b) lost with **no pickup
  completed before the loss** (attributed to last-delivery tech); (c) lost **after a
  completed pickup** (attributed to the pickup tech). Proximity attribution — not fault.
- **NEW N3 — Tech Scorecard:** per-tech monthly sheet joining tickets, active days,
  redeliveries (originating-tech attribution per locked decision #4), the three lost
  columns, and OT. Charts E1 (top OT%), E2 (company OT trend), E3 (productivity vs OT).

Prior versions: see repository history (v1.32.x and earlier changelogs).


## Cell 1 — INSTALL DEPENDENCIES

In [ ]:
%pip install statsmodels rapidfuzz matplotlib seaborn python-dotenv pypyodbc openpyxl python-dateutil scipy

## Cell 2 — IMPORTS

In [ ]:
import os, re, time, warnings, calendar as _cal
from datetime import datetime, date, timedelta
from collections import defaultdict
from functools import reduce
from dateutil.relativedelta import relativedelta
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from dotenv import load_dotenv
import pypyodbc as odbc
import scipy.stats as stats
try:
    from rapidfuzz.distance import DamerauLevenshtein as _DL
    _RAPIDFUZZ_OK = True
except ImportError:
    _RAPIDFUZZ_OK = False
    print('rapidfuzz not installed — falling back to pure-Python Levenshtein.')
try:
    import statsmodels.api as sm; _SM_OK = True
except ImportError:
    _SM_OK = False
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
CHART_DPI = 150
matplotlib.rcParams['figure.dpi'] = CHART_DPI
print(f'pandas {pd.__version__}  |  numpy {np.__version__}  |  rapidfuzz: {_RAPIDFUZZ_OK}')

## Cell 3 — CONFIGURATION

In [ ]:
RUN_DATE     = datetime.now().strftime('%Y-%m-%d')
FILTER_START = '2025-01-01'
AS_OF_DATE   = datetime.now().date() - timedelta(days=1)
FILTER_END   = AS_OF_DATE.strftime('%Y-%m-%d')
print(f'AS_OF_DATE (last full day of data): {AS_OF_DATE.isoformat()}')
EXCLUDE_CURRENT_MONTH = False
if EXCLUDE_CURRENT_MONTH:
    _last = datetime.now().replace(day=1) - relativedelta(days=1)
    FILTER_END = _last.strftime('%Y-%m-%d')

# ── v1.34.0: repo-aligned paths (no analyst-specific paths anywhere) ─────────
# The repo root is found by walking up from the current working directory until
# a folder containing '.git' or 'CLAUDE.md' appears. Works whether the notebook
# is launched from VS Code (folder open = repo) or from a subfolder.
from pathlib import Path

def _find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / '.git').exists() or (cand / 'CLAUDE.md').exists():
            return cand
    raise RuntimeError(
        'Repo root not found. Open the productivity-agent folder in VS Code '
        '(File > Open Folder) so the notebook runs inside the repository.')

REPO_ROOT   = _find_repo_root()
REPORT_ROOT = REPO_ROOT / 'outputs' / 'tech_workload'   # git-ignored; stays local

# ── v1.34.0: credentials live OUTSIDE the repo and OUTSIDE OneDrive ──────────
# %USERPROFILE%\.dme-secrets is local-only: not synced, not in the repo, and
# deny-listed in .claude/settings.json so the agent can never read it.
# Only needed for AUTH_MODE='sql'; 'entra' mode stores no secret at all.
SECRETS_DIR = Path.home() / '.dme-secrets'
ENV_FILE    = SECRETS_DIR / 'MSAIKey.env'

_OLD_ENV = Path.home() / 'OneDrive - DME Express' / 'Documents' / 'IT' / 'Python' / 'MSAIKey.env'
if _OLD_ENV.exists():
    print('*** MIGRATION NEEDED: MSAIKey.env found in OneDrive (it syncs the password')
    print('    to the cloud). Move it, then delete the OneDrive copy + recycle bin copy:')
    print(f'      mkdir "{SECRETS_DIR}" ; Move-Item "{_OLD_ENV}" "{ENV_FILE}"')
    print('    Then rotate the SQL password with IT (the old one has been synced).')

# ── v1.34.0: authentication mode ─────────────────────────────────────────────
# 'entra' (default): Authentication=ActiveDirectoryInteractive — your own
#     corporate sign-in, MFA included, nothing stored on disk. Requires your
#     Entra account to be granted db_datareader on the database (ask IT).
# 'sql': legacy SQL login from ENV_FILE (SQL_USERNAME / SQL_PASSWORD).
AUTH_MODE = 'entra'

DRIVER_NAME   = 'ODBC Driver 18 for SQL Server'   # Entra interactive needs 17.6+
SERVER_NAME   = 'tcp:dmeexpress.database.windows.net,1433'
DATABASE_NAME = 'DMEEXPRESS'

FUZZY_EDIT_DIST  = 1
OUTLIER_Z_THRESH = 2.5
DATA_FRESHNESS_THRESHOLD = 0.25

MIN_HOURS_MONTH  = 4.0

# ── v1.33.0 (N1): Overtime configuration ────────────────────────────────────
# FLSA-style inference from PLC daily hours: any hours over OT_WEEKLY_THRESHOLD
# in a Sun–Sat workweek count as overtime. Computed on RAW per-person hours
# (before any warehouse pro-rating) because the threshold applies to the
# employee, not the warehouse. CAVEAT: if PLC 'Hours' includes PTO/holiday pay,
# OT is overstated (PTO does not count toward the 40-hr threshold). Confirm the
# feed with payroll; if an earnings-code column exists, prefer it over inference.
OT_WEEKLY_THRESHOLD = 40.0   # hours per workweek before OT begins
MIN_HOURS_FOR_OT_RATE = 80.0 # total-hours floor before a tech appears in OT-rate rankings
MIN_ACTIVE_DAYS_RANK  = 30   # active-day floor for tech top/bottom ranking (M1)
PCT_DEPT = 'Patient Care Technician'

FIELD_TECH_TITLES = {'patient care technician','service technician','lead technician','lead tech',
                     'warehouse technician','warehouse tech','warehouse manager','site manager','area manager',}
FIELD_TECH_DEPTS = {'field operations','warehouse'}
INTERNAL_OPS_TITLES = {'customer service','csr','dispatcher','dispatch','routing','call center','intake','scheduler','scheduling'}
INTERNAL_OPS_DEPTS = {'customer service','dispatch','routing','call center','intake','scheduling'}

INCLUDED_REASONS = [
    'Priority 1 - Hospital Discharge (D)',
    'Priority 1 - Respiratory Distress (D)',
    'Priority 1 - Respiratory Service/Exchange (D)',
    'Priority 1 - Respiratory Service/Exchange (P)',
    'Priority 1 - Respiratory Service/Exchange (S)',
    'Priority 1 - Service Correction (D)',
    'Priority 1 - Service Correction (P)',
    'Priority 2 - Exchange (S)',
    'Priority 2 - New Admit (D)',
    'Priority 2 - Respiratory Equipment (D)',
    'Priority 2 - Service/Exchange (D)',
    'Priority 2 - Service/Exchange (P)',
    'Priority 2 - Service/Repair (S)',
    'Priority 2 - Swap Out (Equipment Provider)',
    'Priority 3 - Additional Equipment (D)',
    'Priority 3 - Change Address (D)',
    'Priority 3 - Change Address (P)',
    'Priority 3 - Customer/Patient Request (P)',
    'Priority 3 - Exchange (S)',
    'Priority 3 - Inservice (S)',
    'Priority 3 - Live Discharge (P)',
    'Priority 3 - O2 Refill (D)',
    'Priority 3 - O2 Refill (P)',
    'Priority 3 - Patient Expired (P)',
    'Priority 3 - Respite Stay (D)',
    'Priority 3 - Respite Stay (P)',
    'Priority 3 - Service/Exchange (D)',
    'Priority 3 - Service/Exchange (P)',
    'Priority 3 - Swap Out (Equipment Provider)',
    'Split Order',
]

STATE_PREFIX_LEN = 2
METRO_GROUPS = {
    'DFW':         ['irving','garland','fort worth','txs garland'],
    'Houston':     ['houston','league city','south houston'],
    'San Antonio': ['san antonio'],
}

PALETTE = {
    'attributed'   : '#1f77b4',
    'dark_unattrib': '#d62728',
    'unmatched'    : '#ff7f0e',
    'blank'        : '#9467bd',
    'redelivery'   : '#e377c2',
    'internal_ops' : '#8c564b',
    'company_avg'  : '#000000',
}

OUT_DIR = str(REPORT_ROOT / RUN_DATE)   # str() so os.path.join elsewhere keeps working
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Repo root: {REPO_ROOT}')
print(f'Output:    {OUT_DIR}')
print(f'Window: {FILTER_START} -> {FILTER_END}')
print(f'Metro groups: {list(METRO_GROUPS.keys())}')

# ─────────────────────────────────────────────────────────────────────────────
# WAREHOUSE_STATE_OVERRIDES
# Manual state mapping for warehouses where SERP_WAREHOUSES.[State Province]
# is NULL AND no sibling warehouse (same city, different region prefix) has a
# populated state. The resolution pipeline in Cell 9 tries (1) master,
# (2) sibling-city lookup from df_hier, (3) this override map, (4) 'Unknown'.
# Add new entries here as new orphan warehouses appear.
# ─────────────────────────────────────────────────────────────────────────────
WAREHOUSE_STATE_OVERRIDES = {
    'Distribution Center - Alabama':   'AL',
    'Distribution Center - Louisiana': 'LA',
    'R03 Hot Springs':       'AR',
    'R04 Camden':            'AR',
    'R04 Fort Smith':        'AR',
    'R05 Natchez Storage':   'MS',
    'R05 Tuscaloosa':        'AL',
    'R08 Alexander City':    'AL',
    'R09 Calhoun':           'GA',
    'R10 Hunt Valley':       'MD',
    'R11 Columbia - SC':     'SC',
    'R12 Garland':           'TX',
    'R13 Akron':             'OH',
    'R14 Hendersonville':    'TN',
    'R14 Jackson, TN':       'TN',
    'R15 Chantilly - VA':    'VA',
    'R15 Fredericksburg':    'VA',
    'R16 H3S':               'TX',
    'R16 Houston':           'TX',
    'RNW Austin':            'TX',
    'RNW Garland':           'TX',
}
print(f'WAREHOUSE_STATE_OVERRIDES: {len(WAREHOUSE_STATE_OVERRIDES)} entries')

## Cell 4 — DATABASE CONNECTION & UTILITY FUNCTIONS

In [ ]:
# ── v1.34.0: auth-mode-aware connection ──────────────────────────────────────
_BASE = (f'DRIVER={{{DRIVER_NAME}}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};'
         f'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
         f'ApplicationIntent=ReadOnly;')   # routing hint only — NOT a security boundary

if AUTH_MODE == 'entra':
    # Your own corporate sign-in (browser/MFA prompt on first connect).
    # No password on disk; access rights = whatever IT granted YOUR account.
    conn_str = _BASE + 'Authentication=ActiveDirectoryInteractive;'
elif AUTH_MODE == 'sql':
    if not ENV_FILE.exists():
        raise FileNotFoundError(
            f'{ENV_FILE} not found. Create it (see repo docs/secrets-setup.md): two lines,\n'
            'SQL_USERNAME=...\nSQL_PASSWORD=...\n'
            'It must live in %USERPROFILE%\\.dme-secrets — never in OneDrive or the repo.')
    load_dotenv(ENV_FILE)
    _uid, _pwd = os.getenv('SQL_USERNAME'), os.getenv('SQL_PASSWORD')
    if not (_uid and _pwd):
        raise RuntimeError(f'{ENV_FILE} exists but SQL_USERNAME / SQL_PASSWORD are missing.')
    conn_str = _BASE + f'UID={_uid};PWD={_pwd};'
    del _pwd
else:
    raise ValueError(f'Unknown AUTH_MODE: {AUTH_MODE!r} (use "entra" or "sql")')

sql_conn = odbc.connect(conn_str)
del conn_str   # don't leave the (possibly credentialed) string in notebook globals
print(f'DB connection established ({AUTH_MODE} auth, read-only intent).')

# ── v1.34.0: read-only guard ─────────────────────────────────────────────────
# Belt-and-suspenders for agent-driven runs: refuse anything that isn't a plain
# SELECT/CTE before it reaches the server. The REAL enforcement is server-side —
# the connecting account must hold db_datareader ONLY (verify with IT).
_WRITE_KEYWORDS = re.compile(
    r'\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|CREATE|TRUNCATE|GRANT|REVOKE|EXEC|EXECUTE|INTO)\b',
    re.IGNORECASE)

def _assert_readonly(query):
    stripped = re.sub(r'--[^\n]*', '', query)                    # line comments
    stripped = re.sub(r'/\*.*?\*/', '', stripped, flags=re.S)   # block comments
    if not re.match(r'^\s*(SELECT|WITH)\b', stripped, re.IGNORECASE):
        raise PermissionError('run_query: only SELECT/WITH statements are allowed.')
    m = _WRITE_KEYWORDS.search(stripped)
    if m:
        raise PermissionError(f'run_query: write keyword {m.group(1).upper()!r} refused.')

def run_query(query, label='', verbose=False):
    _assert_readonly(query)
    if verbose: print(f'Query: {label}\n{query}')
    t0 = time.time()
    cur = sql_conn.cursor(); cur.execute(query)
    rows = cur.fetchall()
    cols = [c[0].lower() for c in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    if label: print(f'  {label}: {len(df):,} rows  ({time.time()-t0:.1f}s)')
    return df

def clean_numbers(val):
    # v1.33.0 (L2): preserve the negative sign — the old regex stripped '-', silently
    # flipping negative adjustments positive. Also avoid lstrip('0') mangling.
    if val is None: return np.nan
    s = str(val).strip()
    neg = s.startswith('-') or (s.startswith('(') and s.endswith(')'))  # (123) = accounting negative
    c = re.sub(r'[^0-9.]', '', s)
    if not c or c == '.': return np.nan
    try:
        v = float(c)
    except ValueError:
        return np.nan
    return -v if neg else v

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, f'{name}_{RUN_DATE}.png'), bbox_inches='tight', dpi=CHART_DPI)

def _strip_wh_prefix(name):
    """Strip leading region token (R##, RNW) or 'Distribution Center -' to expose
    the underlying city/location. Used by resolve_state_series() to find sibling
    warehouses for the same physical location.
    Examples:
        'R02 Lake Charles'         -> 'Lake Charles'
        'RNW Del Rio'              -> 'Del Rio'
        'Distribution Center - LA' -> 'LA'
    """
    import re as _re
    if not isinstance(name, str): return ''
    s = name.strip()
    s = _re.sub(r'^R\d{2}\s+', '', s)
    s = _re.sub(r'^RNW\s+', '', s, flags=_re.IGNORECASE)
    s = _re.sub(r'^Distribution Center\s*-\s*', '', s, flags=_re.IGNORECASE)
    return s.strip()

def assign_metro(wh_name, metro_groups=METRO_GROUPS):
    """Assign a warehouse to a metro group by substring match (case-insensitive).
    Returns the metro name, or None if no match. First match wins.
    """
    if not wh_name or not isinstance(wh_name, str): return None
    wh_lc = wh_name.lower().strip()
    for metro, patterns in metro_groups.items():
        if any(p in wh_lc for p in patterns):
            return metro
    return None

## Cell 5 — LEVENSHTEIN DISTANCE

In [ ]:
if _RAPIDFUZZ_OK:
    def levenshtein(a, b): return _DL.distance(a, b)
else:
    def levenshtein(a, b):
        if a==b: return 0
        if not a: return len(b)
        if not b: return len(a)
        la,lb=len(a),len(b)
        d=[[0]*(lb+1) for _ in range(la+1)]
        for i in range(la+1): d[i][0]=i
        for j in range(lb+1): d[0][j]=j
        for i in range(1,la+1):
            for j in range(1,lb+1):
                cost=0 if a[i-1]==b[j-1] else 1
                d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+cost)
                if i>1 and j>1 and a[i-1]==b[j-2] and a[i-2]==b[j-1]: d[i][j]=min(d[i][j],d[i-2][j-2]+cost)
        return d[la][lb]
print('levenshtein() ready.')

## Cell 6 — NAME STANDARDIZATION ENGINE

In [ ]:
NICKNAME_TO_CANONICAL_RAW = [
    ('chris',['Christopher','Christian','Christina','Christine']),('kris',['Christopher','Kristopher']),
    ('mike',['Michael']),('mikey',['Michael']),('matt',['Matthew']),('dan',['Daniel']),('danny',['Daniel']),
    ('dave',['David']),('rob',['Robert']),('bob',['Robert']),('bobby',['Robert']),('robbie',['Robert']),
    ('robby',['Robert']),('jim',['James']),('jimmy',['James']),('jamie',['James']),('joe',['Joseph']),
    ('joey',['Joseph']),('tom',['Thomas']),('tommy',['Thomas']),('bill',['William']),('billy',['William']),
    ('will',['William']),('liam',['William']),('rick',['Richard','Ricardo','Frederick']),
    ('ricky',['Richard','Ricardo']),('rich',['Richard']),('steve',['Steven','Stephen']),
    ('ed',['Edward','Eduardo','Edwin']),('eddie',['Edward','Eduardo']),('ted',['Edward','Theodore']),
    ('tony',['Anthony','Antonio']),('nick',['Nicholas','Nicolas']),('pat',['Patrick','Patricia']),
    ('tim',['Timothy']),('timmy',['Timothy']),('jon',['Jonathan','Jonathon']),
    ('johnny',['John','Jonathan']),('jack',['John','Jackson']),('jeff',['Jeffrey','Geoffrey']),
    ('andy',['Andrew','Andres']),('drew',['Andrew']),('ron',['Ronald','Ronaldo']),
    ('ronnie',['Ronald']),('ken',['Kenneth']),('kenny',['Kenneth']),('ben',['Benjamin']),
    ('benny',['Benjamin','Benito']),('greg',['Gregory']),('sam',['Samuel','Samantha']),
    ('josh',['Joshua']),('alex',['Alexander','Alejandro','Alexandra']),
    ('nate',['Nathaniel','Nathan']),('zach',['Zachary','Zachariah']),('zack',['Zachary']),
    ('mitch',['Mitchell']),('ray',['Raymond','Raymundo']),('larry',['Lawrence','Lorenzo']),
    ('terry',['Terrence','Terrell']),('jerry',['Gerald','Jeremiah','Jerome']),
    ('chuck',['Charles']),('charlie',['Charles']),('fred',['Frederick','Fredrick','Alfredo']),
    ('hank',['Henry']),('harry',['Henry','Harold','Harrison']),('phil',['Philip','Phillip']),
    ('wes',['Wesley']),('vince',['Vincent']),('vinny',['Vincent','Vincenzo']),
    ('abe',['Abraham']),('gabe',['Gabriel']),('len',['Leonard']),('walt',['Walter']),
    ('doug',['Douglas']),('al',['Albert','Alan','Alfonso']),('bert',['Albert','Robert','Herbert']),
    ('don',['Donald']),('gene',['Eugene']),('manny',['Manuel','Emmanuel']),('marty',['Martin']),
    ('art',['Arthur']),('curt',['Curtis']),('ernie',['Ernest','Ernesto']),
    ('frank',['Franklin','Francisco','Francis']),('frankie',['Franklin','Francisco','Frank']),
    ('jr',['Junior']),('lupe',['Guadalupe']),('max',['Maximilian','Maxwell','Maximo']),
    ('reggie',['Reginald']),('rudy',['Rudolph','Rodolfo']),
    ('ty',['Tyler','Tyrone','Tyson']),('vic',['Victor']),
    ('liz',['Elizabeth']),('beth',['Elizabeth']),('lisa',['Elizabeth']),
    ('kate',['Katherine','Kathryn','Kaitlyn']),('kathy',['Katherine','Kathryn']),
    ('katie',['Katherine']),('sue',['Susan','Suzanne']),('susie',['Susan']),
    ('jen',['Jennifer']),('jenny',['Jennifer']),('jenn',['Jennifer']),('amy',['Amelia','Amy']),
    ('meg',['Megan','Margaret']),('maggie',['Margaret']),('pam',['Pamela']),('barb',['Barbara']),
    ('deb',['Deborah','Debra']),('debbie',['Deborah','Debra']),('carol',['Caroline','Carolyn']),
    ('tina',['Christina']),('sandy',['Sandra','Alexandra']),('cindy',['Cynthia']),
    ('angie',['Angela','Angelica']),('ang',['Angela']),('steph',['Stephanie']),
    ('stacy',['Stacey','Stacy']),('nikki',['Nicole','Nichole']),('mia',['Maria']),
    ('maria',['Maria','Marie']),('anna',['Annette','Annalisa']),
    ('nicky',['Nichole','Nicole']),('joanie',['Joan']),('diana',['Diane']),('dani',['Danielle']),
    ('trish',['Patricia']),('patty',['Patricia']),('bri',['Brianna','Brittany']),
    ('brit',['Brittany','Britney']),('chrissy',['Christina','Christine']),('vero',['Veronica']),
    ('cass',['Cassandra']),('dot',['Dorothy']),('bev',['Beverly']),
    ('mel',['Melanie','Melissa','Melinda']),('missy',['Melissa']),('mindy',['Melinda']),
    ('pepe',['Jose']),('chuy',['Jesus']),('nacho',['Ignacio']),('chava',['Salvador']),
    ('memo',['Guillermo']),('beto',['Roberto','Alberto']),('pancho',['Francisco']),
    ('paco',['Francisco']),('chela',['Graciela']),('chucho',['Jesus']),('vale',['Valeria']),
    ('lalo',['Eduardo']),('pipe',['Felipe']),('nico',['Nicolas']),
]
NICKNAME_TO_CANONICAL = {}
for nick,canonicals in NICKNAME_TO_CANONICAL_RAW:
    if nick not in NICKNAME_TO_CANONICAL: NICKNAME_TO_CANONICAL[nick]=[]
    for c in canonicals:
        if c not in NICKNAME_TO_CANONICAL[nick]: NICKNAME_TO_CANONICAL[nick].append(c)
CANONICAL_TO_NICKNAMES = defaultdict(list)
for nick,canonicals in NICKNAME_TO_CANONICAL.items():
    for canon in canonicals: CANONICAL_TO_NICKNAMES[canon.lower()].append(nick)

def standardize_first_name(raw):
    if not raw or not isinstance(raw,str): return []
    raw_lc=raw.lower().strip(); seen,results=set(),[raw]; seen.add(raw_lc)
    for cand in NICKNAME_TO_CANONICAL.get(raw_lc,[]):
        if cand.lower() not in seen: results.append(cand); seen.add(cand.lower())
    for nick in CANONICAL_TO_NICKNAMES.get(raw_lc,[]):
        if nick.lower() not in seen: results.append(nick.title()); seen.add(nick.lower())
    return results
print(f'Nickname map: {len(NICKNAME_TO_CANONICAL)} entries')

## Cell 7 — DATA EXTRACTION

In [ ]:
print(f'Extracting data ({FILTER_START} to {FILTER_END})...')
t0 = time.time()
_reasons_sql = ',\n      '.join(f"'{r}'" for r in INCLUDED_REASONS)

df_tx = run_query(f"""
SELECT TRIM(TX.Order_Num) AS order_num, TRIM(TX.Record_ID) AS record_id,
    TRIM(TX.Tech_Warehouse) AS tech_warehouse,
    TRIM(ISNULL(TX.TechFirstName,'')) AS techfirstname,
    TRIM(ISNULL(TX.TechLastName,''))  AS techlastname,
    TX.Reason AS reason,
    TRY_CONVERT(DATE, TX.Completed_Date) AS completed_date
FROM dbo.[SERP TRANSACTIONS] AS TX WITH (NOLOCK)
WHERE TX.Tech_Warehouse NOT LIKE 'Z%' AND TX.[Status] <> 'Canceled'
  AND TX.Order_Num <> ''
  -- v1.33.0 (H3): filter on the CONVERTED date, matching the SELECT. The raw-string
  -- compare could (a) drop the entire last day if values carry a time component
  -- ('2026-07-30 14:22' > '2026-07-30' as strings) and (b) admit malformed strings
  -- that TRY_CONVERT NULLs in the SELECT, creating NaT rows downstream.
  AND TRY_CONVERT(DATE, TX.Completed_Date) >= '{FILTER_START}'
  AND TRY_CONVERT(DATE, TX.Completed_Date) <= '{FILTER_END}'
  AND TX.Reason IN ({_reasons_sql})
OPTION (RECOMPILE, MAXDOP 4)
""", 'Transactions')
display(df_tx.head(3))

df_emp = run_query("""
SELECT count(id) AS emp_duplicate_count,
    TRIM([FirstName]) AS empfirstname, TRIM([LastName]) AS emplastname,
    TRIM([Warehouse Name]) AS [location],
    MAX([Department Name]) AS dept, MAX([Job Title]) AS title,
    MAX([Employee Number]) AS eid, MAX([Username]) AS username
FROM SERP_DME_EMPLOYEES
GROUP BY TRIM([FirstName]),TRIM([LastName]),TRIM([Warehouse Name])
""", 'Employees')

df_hier = run_query("""
SELECT DISTINCT TRIM([Warehouse Name]) AS warehouse, [State Province] as [State],
    LEFT(TRIM([Warehouse Name]),3) AS region, TRIM([Group]) AS vp
FROM SERP_WAREHOUSES WITH (NOLOCK)
""", 'Warehouse hierarchy')

df_apc_snapshot_query = f"""
SELECT TOP 1 [date] FROM SERP_APC_DAILY
WHERE [date] <= '{FILTER_END}'
ORDER BY [date] DESC
"""
_apc_snap_df = run_query(df_apc_snapshot_query, 'APC snapshot-date probe')
if len(_apc_snap_df)==0 or pd.isna(_apc_snap_df.iloc[0,0]):
    raise RuntimeError(f'APC: no rows on or before {FILTER_END}. Cannot compute snapshot.')
_apc_snap_date = pd.to_datetime(_apc_snap_df.iloc[0,0]).date()
_apc_lag_days = (AS_OF_DATE - _apc_snap_date).days
print(f'  APC snapshot date: {_apc_snap_date.isoformat()} (lag: {_apc_lag_days} day(s) behind AS_OF_DATE)')
if _apc_lag_days > 7:
    print(f'  *** WARNING: APC feed is >7 days stale. Lost-cost-per-ADC metrics may be unreliable.')
df_apc = run_query(f"""
SELECT TRIM(APC.warehourse) AS warehouse, SUM(APC.total) AS apc
FROM SERP_APC_DAILY AS APC WITH (NOLOCK)
WHERE APC.[date] = '{_apc_snap_date.isoformat()}'
  AND APC.warehourse NOT LIKE 'Z%'
  AND APC.customer NOT LIKE '(F)%' AND APC.customer NOT LIKE '(IPU)%'
  AND APC.customer NOT LIKE '%Contract Test%'
GROUP BY TRIM(APC.warehourse)
""", 'APC')
df_apc['apc'] = df_apc['apc'].apply(clean_numbers)

df_adc = run_query(f"""
WITH daily AS (SELECT [date],warehourse,SUM(total) AS APC FROM SERP_APC_DAILY
    WHERE [date]>='{FILTER_START}' AND [date]<='{FILTER_END}' AND warehourse NOT LIKE 'Z%' GROUP BY [date],warehourse)  -- v1.30.4
SELECT TRIM(warehourse) AS warehouse,AVG(APC) AS ADC,SUM(APC) AS pt_days,
    YEAR([date]) AS yr,MONTH([date]) AS mo
FROM daily GROUP BY YEAR([date]),MONTH([date]),warehourse ORDER BY yr,mo,warehourse
""", 'ADC')
df_adc[['yr','mo']] = df_adc[['yr','mo']].astype(int)
df_adc['adc']     = df_adc['adc'].apply(clean_numbers)
df_adc['pt_days'] = df_adc['pt_days'].apply(clean_numbers)

df_hours_raw = run_query(f"""
SELECT Employee_Name, CAST(WorkDate AS DATE) AS workdate,
    CAST([Hours] AS FLOAT) AS workhours,
    -- v1.33.0 (L3): deterministic day-of-week. DATEPART(weekday,...) depends on the
    -- session @@DATEFIRST; a driver/server default change would silently swap
    -- Saturday and Sunday hours. 1900-01-01 was a Monday, so this is Monday=0..Sunday=6
    -- regardless of settings.
    (DATEDIFF(day, '1900-01-01', CAST(WorkDate AS DATE)) % 7) AS dow_mon0
FROM PLC_EMPLOYEE_HOURS WITH (NOLOCK)
-- v1.33.0 (H3): half-open upper bound so intra-day timestamps on FILTER_END are kept.
WHERE Department_Name='{PCT_DEPT}' AND WorkDate>='{FILTER_START}' AND WorkDate < DATEADD(day, 1, '{FILTER_END}')
""", 'Payroll hours')

df_redel = run_query(f"""
SELECT TRIM(RD.Orig_Order) AS orig_order_num,
    TRY_CONVERT(DATE,RD.Completion_DateTime) AS rd_date,
    RD.Completion_DateTime AS rd_datetime_raw,
    TRIM(RD.Products) AS rd_products,
    TRIM(ISNULL(SE.[FirstName],'')) AS techfirstname,
    TRIM(ISNULL(SE.[LastName],''))  AS techlastname,
    TRIM(ISNULL(RD.Tech_Warehouse,'')) AS tech_warehouse
FROM dbo.[Re-Delivery Report] RD
LEFT JOIN dbo.SERP_DME_EMPLOYEES SE ON SE.Username=RD.Tech
WHERE (TRY_CONVERT(DATE,RD.Completion_DateTime) BETWEEN '{FILTER_START}' AND '{FILTER_END}')
   OR (TRY_CONVERT(DATE,RD.Completion_DateTime) IS NULL
       AND RD.Completion_DateTime LIKE '202_-%')  -- keep parse-fail rows that look in-range
""", 'Redeliveries')
if 'tech_warehouse' not in df_redel.columns: df_redel['tech_warehouse']=''

df_lost_raw = run_query(f"""
SELECT ATI.Asset_Tag AS asset_tag,
    NULLIF(TRIM(CAST(ATI.Bill_to_ID AS VARCHAR(20))),'0') AS bill_to_id,
    MP.Product_Name AS product_name, WH.[Warehouse Name] AS tech_warehouse,
    LEFT(TRIM(WH.[Warehouse Name]),3) AS region, TRIM(WH.[Group]) AS vp,
    WH.[State Province] AS state,  -- v1.26.0: state from master, not prefix
    ATI.Lost_Date AS lost_date_raw, MP.Unit_Cost_Last_Price AS lost_cost_last_price
FROM SERP_ACTIVE_TAGGED_INV ATI WITH (NOLOCK)
JOIN SERP_WAREHOUSES WH WITH (NOLOCK) ON WH.ID=ATI.Warehouse_ID
JOIN SERP_MASTER_PRODUCTS MP WITH (NOLOCK) ON MP.ID=ATI.Master_ID
WHERE ATI.Lost IS NOT NULL AND WH.[Warehouse Name] NOT LIKE 'Z%'
  AND ATI.Lost_Date >= '{FILTER_START}'
  AND ATI.Lost_Date <= '{FILTER_END}'  -- v1.30.4
""", 'Lost equipment')

df_inventory_total = run_query("""
SELECT TRIM(WH.[Warehouse Name]) AS tech_warehouse, 
    COUNT_BIG(ATI.Asset_Tag) AS total_inventory_count,
    SUM(CONVERT(float, COALESCE(ATI.Unit_Cost, MP.Unit_Cost_Last_Price))) AS total_inventory_amount
FROM SERP_ACTIVE_TAGGED_INV ATI WITH (NOLOCK)
JOIN SERP_WAREHOUSES WH WITH (NOLOCK) ON WH.ID=ATI.Warehouse_ID
LEFT OUTER JOIN SERP_MASTER_PRODUCTS MP ON MP.ID=ATI.Master_ID
WHERE WH.[Warehouse Name] NOT LIKE 'Z%' AND MP.Active='Yes' AND MP.Asset_Tag_Not_Required = 'No'
GROUP BY TRIM(WH.[Warehouse Name])
""", 'Inventory totals')
df_inventory_total['total_inventory_count'] = pd.to_numeric(df_inventory_total['total_inventory_count'], errors='coerce').fillna(0).astype(int)
df_inventory_total['total_inventory_amount'] = pd.to_numeric(df_inventory_total['total_inventory_amount'], errors='coerce').fillna(0).astype(float)

df_master = run_query("SELECT ID AS masterid, TRIM(Product_Name) AS product_name FROM dbo.SERP_MASTER_PRODUCTS", 'Master products')
print(f'All queries complete in {time.time()-t0:.1f}s')

## Cell 8 — TEMPORAL FEATURES & REASON CLASSIFICATION

In [ ]:
df_tx['completed_date'] = pd.to_datetime(df_tx['completed_date'],errors='coerce')
# v1.33.0 (H3): NaT guard. With the WHERE now on TRY_CONVERT this should be zero;
# if it isn't, rows were admitted that can't be dated and would silently form
# NaN month groups (or crash later int casts). Fail loudly, drop, and report.
_n_nat = int(df_tx['completed_date'].isna().sum())
if _n_nat > 0:
    print(f'*** WARNING: {_n_nat:,} transaction rows have unparseable Completed_Date — DROPPED.')
    df_tx = df_tx[df_tx['completed_date'].notna()].copy()
df_tx['delivery_year']  = df_tx['completed_date'].dt.year
df_tx['delivery_month'] = df_tx['completed_date'].dt.month
df_tx['month_date']     = df_tx['completed_date'].values.astype('datetime64[M]')
_dow = df_tx['completed_date'].dt.dayofweek
df_tx['schedule_period'] = np.select([_dow==5,_dow==6],['Saturday','Sunday'],default='Weekday')
df_tx['metro'] = df_tx['tech_warehouse'].apply(assign_metro)

_REASON_MAP = [
    (r'priority 1','Urgent',1),(r'priority 2 - exchange','Exchange/Service',2),
    (r'priority 2 - new admit','New Admission',2),(r'priority 2 - respiratory','Respiratory',2),
    (r'priority 2','Exchange/Service',2),(r'priority 3 - additional','Additional Equip',3),
    (r'priority 3 - change','Admin/Change',3),(r'priority 3 - customer','Patient Request',3),
    (r'priority 3 - live disc','Live Discharge',3),(r'priority 3 - o2','O2 Refill',3),
    (r'priority 3 - patient exp','Patient Expired',3),(r'priority 3 - respite','Respite',3),
    (r'priority 3','Routine P3',3),(r'split order','Split Order',4),
]
def _classify(reason):
    if not reason or not isinstance(reason,str): return ('Other',5)
    r=reason.lower()
    for pat,cat,pri in _REASON_MAP:
        if re.search(pat,r): return (cat,pri)
    return ('Other',5)
def _ticket_type(reason):
    if not reason or not isinstance(reason,str): return 'Other'
    if reason.strip().upper()=='SPLIT ORDER': return 'Split'
    m=re.search(r'\(([DPS])\)\s*$',reason.strip())
    return {'D':'Delivery','P':'Pickup','S':'Service'}[m.group(1)] if m else 'Other'

_unique_reasons    = df_tx['reason'].dropna().unique()
_reason_cat_lookup = {r:_classify(r)[0] for r in _unique_reasons}
_reason_pri_lookup = {r:_classify(r)[1] for r in _unique_reasons}
_type_lookup       = {r:_ticket_type(r) for r in _unique_reasons}
df_tx['reason_category'] = df_tx['reason'].map(_reason_cat_lookup).fillna('Other')
df_tx['priority_level']  = df_tx['reason'].map(_reason_pri_lookup).fillna(5).astype(int)
df_tx['ticket_type']     = df_tx['reason'].map(_type_lookup).fillna('Other')

print(f'Rows: {len(df_tx):,}')
print(df_tx['schedule_period'].value_counts().to_string())
# State diagnostic moved to Cell 9 (v1.26.1) — state isn't merged in yet here.
print('\nMetro distribution:')
print(df_tx['metro'].value_counts(dropna=False).to_string())

def _count_weekdays_elapsed(year, month, ref=None):
    ref = ref or AS_OF_DATE
    yr, mo = int(year), int(month)
    if yr > ref.year or (yr == ref.year and mo > ref.month):
        return 0
    last_d = _cal.monthrange(yr, mo)[1]
    end_d = ref.day if (yr == ref.year and mo == ref.month) else last_d
    return sum(1 for d in range(1, end_d+1) if _cal.weekday(yr, mo, d) < 5)
def _count_weekdays_full(year, month):
    yr, mo = int(year), int(month)
    last_d = _cal.monthrange(yr, mo)[1]
    return sum(1 for d in range(1, last_d+1) if _cal.weekday(yr, mo, d) < 5)

_monthly_tx = (df_tx.groupby(['delivery_year','delivery_month'], dropna=False, as_index=False)
               .agg(ticket_count=('order_num','nunique'))
               .sort_values(['delivery_year','delivery_month']).reset_index(drop=True))
_monthly_tx['trailing_3mo_median'] = (_monthly_tx['ticket_count']
    .shift(1).rolling(window=3, min_periods=1).median())
_today_ref = AS_OF_DATE

def _period_status(y, m, today=_today_ref):
    y, m = int(y), int(m)
    if y > today.year or (y == today.year and m > today.month): return 'future'
    if y == today.year and m == today.month: return 'partial'
    return 'complete'
_monthly_tx['period_status'] = [_period_status(y, m) for y, m in
    zip(_monthly_tx['delivery_year'], _monthly_tx['delivery_month'])]
_monthly_tx['weekday_days_elapsed'] = [_count_weekdays_elapsed(y, m) for y, m in
    zip(_monthly_tx['delivery_year'], _monthly_tx['delivery_month'])]
_monthly_tx['weekday_days_full'] = [_count_weekdays_full(y, m) for y, m in
    zip(_monthly_tx['delivery_year'], _monthly_tx['delivery_month'])]
# MTD-scaled expected: scale trailing median by elapsed/full weekday ratio so  current month isn't compared against a full-month baseline.
_monthly_tx['expected_tickets_mtd_scaled'] = np.where(
    _monthly_tx['weekday_days_full'] > 0,
    _monthly_tx['trailing_3mo_median'] * _monthly_tx['weekday_days_elapsed'] / _monthly_tx['weekday_days_full'],
    np.nan)
_monthly_tx['ratio_actual_vs_expected'] = (
    _monthly_tx['ticket_count'] / _monthly_tx['expected_tickets_mtd_scaled'].replace(0, np.nan)).round(3)
_monthly_tx['is_data_freshness_flag'] = (
    (_monthly_tx['period_status'] != 'future') &
    (_monthly_tx['ratio_actual_vs_expected'].notna()) &
    (_monthly_tx['ratio_actual_vs_expected'] < DATA_FRESHNESS_THRESHOLD))
_monthly_tx['period'] = (_monthly_tx['delivery_year'].astype('Int64').astype(str)
                          + '-' + _monthly_tx['delivery_month'].astype('Int64').astype(str).str.zfill(2))
df_data_freshness = _monthly_tx[['period','delivery_year','delivery_month','period_status',
    'ticket_count','trailing_3mo_median','weekday_days_elapsed','weekday_days_full',
    'expected_tickets_mtd_scaled','ratio_actual_vs_expected','is_data_freshness_flag']].copy()

_flagged = df_data_freshness[df_data_freshness['is_data_freshness_flag']]
if len(_flagged) > 0:
    _bar = '!' * 80
    print(f'\n{_bar}')
    print(f'!!  DATA FRESHNESS WARNING: months below {DATA_FRESHNESS_THRESHOLD:.0%} of trailing 3-mo median')
    print(_bar)
    for _, r in _flagged.iterrows():
        _exp = '—' if pd.isna(r['expected_tickets_mtd_scaled']) else f"~{int(r['expected_tickets_mtd_scaled']):,}"
        _ratio = '—' if pd.isna(r['ratio_actual_vs_expected']) else f"{r['ratio_actual_vs_expected']:.1%}"
        print(f"!!  {r['period']}: actual={int(r['ticket_count']):,}  expected={_exp}  "
              f"ratio={_ratio}  status={r['period_status']}")
    print('!!  Likely cause: upstream data feed issue (SQL load gap, Reason-code change,')
    print('!!  Status filter change). Investigate before sharing analytics externally.')
    print('!!  Details in Excel sheet "Data_Freshness".')
    print(_bar + '\n')
else:
    print(f"\nData freshness check: PASS (all months >= {DATA_FRESHNESS_THRESHOLD:.0%} of trailing median)")

## Cell 9 — NAME MATCHING PIPELINE

In [ ]:
def _clean(s):
    if not s or not isinstance(s,str): return ''
    return re.sub(r"['\.\s\-]",'',s).lower()

MANUAL_NAME_CORRECTIONS = {
    ('damein','combs'):('Damien','Combs'),
    ('damien','combs'):('Damien','Combs'),
}

emp_idx={}; _emp_by_last=defaultdict(list)
for row in df_emp.itertuples(index=False):
    fn,ln=_clean(row.empfirstname),_clean(row.emplastname)
    if fn and ln: emp_idx[(fn,ln)]=row; _emp_by_last[ln].append((fn,row))
print(f'Employee index: {len(emp_idx):,}')

def _classify_role(dept_raw,title_raw):
    d=(dept_raw or '').lower().strip(); t=(title_raw or '').lower().strip()
    if any(x in t for x in FIELD_TECH_TITLES) or any(x in d for x in FIELD_TECH_DEPTS): return 'field_tech'
    if any(x in t for x in INTERNAL_OPS_TITLES) or any(x in d for x in INTERNAL_OPS_DEPTS): return 'internal_ops'
    return 'other'

_dispatcher_last_names=set()
for row in df_emp.itertuples(index=False):
    if any(d in (row.dept or '').lower() for d in INTERNAL_OPS_DEPTS):
        ln_c=_clean(row.emplastname)
        if ln_c: _dispatcher_last_names.add(ln_c)

_tech_by_wh_first={}; _wh_first_collisions=[]
for row in df_emp.itertuples(index=False):
    if _classify_role(row.dept,row.title)!='field_tech': continue
    wh_c,fn_c=_clean(row.location),_clean(row.empfirstname)
    if not (wh_c and fn_c): continue
    key=(wh_c,fn_c)
    if key in _tech_by_wh_first: _wh_first_collisions.append({'wh':row.location,'first':row.empfirstname})
    else: _tech_by_wh_first[key]=row

_field_tech_by_wh_last={}; _wh_last_collisions={}
for row in df_emp.itertuples(index=False):
    if _classify_role(row.dept,row.title)!='field_tech': continue
    wh_c,ln_c=_clean(row.location),_clean(row.emplastname)
    if not (wh_c and ln_c): continue
    key=(wh_c,ln_c)
    if key in _field_tech_by_wh_last: _wh_last_collisions.setdefault(key,[_field_tech_by_wh_last[key]]).append(row)
    else: _field_tech_by_wh_last[key]=row
for key in list(_wh_last_collisions): _field_tech_by_wh_last.pop(key,None)

_internal_ops_first_names=set(); _internal_ops_last_names=set()
for row in df_emp.itertuples(index=False):
    if _classify_role(row.dept,row.title)=='internal_ops':
        fn_c=_clean(row.empfirstname); ln_c=_clean(row.emplastname)
        if fn_c: _internal_ops_first_names.add(fn_c)
        if ln_c: _internal_ops_last_names.add(ln_c)

def _match_name(fn_raw,ln_raw,wh_raw=''):
    """Passes 0-5 name matching. Passes 6-7 run in the caller loop."""
    fn_lc,ln_lc=fn_raw.lower(),ln_raw.lower()
    corrected=MANUAL_NAME_CORRECTIONS.get((fn_lc,ln_lc))
    if corrected: fn_raw,ln_raw=corrected
    fn_c,ln_c=_clean(fn_raw),_clean(ln_raw); wh_c=_clean(wh_raw) if wh_raw else ''
    if (fn_c,ln_c) in emp_idx: return emp_idx[(fn_c,ln_c)],('P0' if corrected else 'P1')
    for cand in standardize_first_name(fn_raw)[1:]:
        key=(_clean(cand),ln_c)
        if key in emp_idx: return emp_idx[key],'P2'
    best_row,best_dist,best_same_wh=None,FUZZY_EDIT_DIST+1,False
    for (emp_fn,emp_row) in _emp_by_last.get(ln_c,[]):
        d=levenshtein(fn_c,emp_fn); same_wh=(wh_c!='' and _clean(getattr(emp_row,'location','') or '')==wh_c)
        if d<best_dist or (d==best_dist and same_wh and not best_same_wh):
            best_row,best_dist,best_same_wh=emp_row,d,same_wh
    if best_row is not None and best_dist<=FUZZY_EDIT_DIST: return best_row,'P3'
    if ln_c in _dispatcher_last_names and wh_c:
        match=_tech_by_wh_first.get((wh_c,fn_c))
        if match is not None: return match,'P4'
    if wh_c and ln_c:
        match=_field_tech_by_wh_last.get((wh_c,ln_c))
        if match is not None: return match,'P5_fallback'
    return None,None

_unique_names=df_tx[['techfirstname','techlastname','tech_warehouse']].drop_duplicates()
_match_results=[]
_pass_counts={'P0':0,'P1':0,'P2':0,'P3':0,'P4':0,'P5_fallback':0,'P5_intops_override':0,
              'P6_intops_first_override':0,'P7_intops_last_override':0,
              'ambiguous_collision':0,'unmatched_blank':0,'unmatched':0}

for row in _unique_names.itertuples(index=False):
    fn,ln,wh=row.techfirstname,row.techlastname,row.tech_warehouse
    is_blank=not fn.strip() and not ln.strip()
    pass_used,matched,collision_candidates=None,None,[]
    if is_blank:
        _pass_counts['unmatched_blank']+=1
    else:
        matched,pass_used=_match_name(fn,ln,wh)
        if matched is not None and _classify_role(matched.dept,matched.title)=='internal_ops' and wh and ln:
            _p5=_field_tech_by_wh_last.get((_clean(wh),_clean(ln)))
            if _p5 is not None: matched,pass_used=_p5,'P5_intops_override'
        if wh and ln and pass_used!='P5_intops_override':
            fn_c,ln_c,wh_c=_clean(fn),_clean(ln),_clean(wh)
            if fn_c in _internal_ops_first_names:
                _p6=_field_tech_by_wh_last.get((wh_c,ln_c))
                _cur_ok=(matched is not None and _classify_role(matched.dept,matched.title)=='field_tech' and _clean(getattr(matched,'location','') or '')==wh_c)
                if _p6 is not None and not _cur_ok: matched,pass_used=_p6,'P6_intops_first_override'
        if wh and fn and pass_used not in ('P5_intops_override','P6_intops_first_override'):
            fn_c7,ln_c7,wh_c7=_clean(fn),_clean(ln),_clean(wh)
            if ln_c7 in _internal_ops_last_names:
                _p7=_tech_by_wh_first.get((wh_c7,fn_c7))
                _cur_ok7=(matched is not None and _classify_role(matched.dept,matched.title)=='field_tech' and _clean(getattr(matched,'location','') or '')==wh_c7)
                if _p7 is not None and not _cur_ok7: matched,pass_used=_p7,'P7_intops_last_override'
        if matched is None and wh and ln:
            key=(_clean(wh),_clean(ln))
            if key in _wh_last_collisions: collision_candidates=_wh_last_collisions[key]; _pass_counts['ambiguous_collision']+=1
        _pass_counts['unmatched' if matched is None else pass_used]=_pass_counts.get('unmatched' if matched is None else pass_used,0)+1
    _rb=_classify_role(matched.dept,matched.title) if matched else None
    _match_results.append({'techfirstname':fn,'techlastname':ln,'tech_warehouse':wh,
        '_is_blank_tech':is_blank,'_is_unmatched':(not is_blank)and(matched is None),
        '_is_ambiguous':(not is_blank)and(matched is None)and len(collision_candidates)>0,
        '_collision_candidates':'; '.join(f'{r.empfirstname} {r.emplastname} ({r.eid})' for r in collision_candidates) if collision_candidates else '',
        '_matched_dept':matched.dept if matched else None,'_matched_title':matched.title if matched else None,
        '_matched_eid':matched.eid if matched else None,'_matched_first':matched.empfirstname if matched else None,
        '_matched_last':matched.emplastname if matched else None,'_matched_location':matched.location if matched else None,
        '_matched_role_bucket':_rb,'_is_internal_ops':_rb=='internal_ops','_pass_used':pass_used})

_df_match=pd.DataFrame(_match_results)
df_tx=df_tx.merge(_df_match,on=['techfirstname','techlastname','tech_warehouse'],how='left')
for col in ['_is_blank_tech','_is_unmatched','_is_ambiguous','_is_internal_ops']: df_tx[col]=df_tx[col].fillna(False)
df_tx['_matched_role_bucket']=df_tx['_matched_role_bucket'].fillna('unknown')
df_tx['_unattributed']=df_tx['_is_blank_tech']|df_tx['_is_unmatched']|df_tx['_is_internal_ops']

_intops_override_passes={'P5_intops_override','P6_intops_first_override','P7_intops_last_override'}
_correction_mask=df_tx['_pass_used'].isin(_intops_override_passes)
df_tx['_raw_techfirstname']=df_tx['techfirstname']; df_tx['_raw_techlastname']=df_tx['techlastname']
df_tx['_name_was_corrected']=_correction_mask
df_tx.loc[_correction_mask,'techfirstname']=df_tx.loc[_correction_mask,'_matched_first'].fillna(df_tx.loc[_correction_mask,'techfirstname'])
df_tx.loc[_correction_mask,'techlastname'] =df_tx.loc[_correction_mask,'_matched_last'].fillna(df_tx.loc[_correction_mask,'techlastname'])
df_tx=df_tx.merge(df_hier.rename(columns={'warehouse':'tech_warehouse'}),on='tech_warehouse',how='left')
# Layer 1 — value already present from SERP_WAREHOUSES.[State Province] merge.
# Layer 2 — sibling-city lookup: another warehouse with the same trimmed city
#           name (prefix stripped) that has a populated state. Self-healing
#           because as ops backfills the master, this layer picks it up.
# Layer 3 — WAREHOUSE_STATE_OVERRIDES (curated in Cell 3) for orphans.
# Layer 4 — 'Unknown' with a LOUD warning listing every warehouse so the data
#           team can fix the source. NO prefix fallback (it fabricated R0/R1/DI/RN).

def resolve_state_series(wh_series, current_state_series, hier_df):
    """Vectorized 3-layer state resolution. Returns (new_state_series, audit_dict).
    hier_df: a DataFrame with columns ['warehouse','state'] from df_hier."""
    out = current_state_series.copy()
    missing_mask = out.isna() | (out.astype(str).str.strip()=='')
    audit = {'sibling':[], 'override':[], 'unknown':[]}
    if not missing_mask.any():
        return out, audit

    # Build city->state map from hier_df where state IS populated
    _hier_clean = hier_df[hier_df['state'].notna() & (hier_df['state'].astype(str).str.strip()!='')].copy()
    _hier_clean['_city'] = _hier_clean['warehouse'].apply(_strip_wh_prefix)
    # If a city maps to >1 distinct state in the master, that's ambiguous — drop it
    _city_states = _hier_clean.groupby('_city')['state'].nunique()
    _unambig_cities = set(_city_states[_city_states==1].index)
    _city_to_state = (_hier_clean[_hier_clean['_city'].isin(_unambig_cities)]
                      .drop_duplicates('_city').set_index('_city')['state'].to_dict())

    for idx in out[missing_mask].index:
        wh = wh_series.loc[idx]
        if not isinstance(wh, str) or not wh.strip():
            audit['unknown'].append(wh); out.loc[idx] = 'Unknown'; continue
        # Layer 2: sibling city
        city = _strip_wh_prefix(wh)
        if city in _city_to_state:
            out.loc[idx] = _city_to_state[city]
            audit['sibling'].append((wh, city, _city_to_state[city])); continue
        # Layer 3: override
        if wh in WAREHOUSE_STATE_OVERRIDES:
            out.loc[idx] = WAREHOUSE_STATE_OVERRIDES[wh]
            audit['override'].append((wh, WAREHOUSE_STATE_OVERRIDES[wh])); continue
        # Layer 4: unknown
        out.loc[idx] = 'Unknown'
        audit['unknown'].append(wh)
    return out, audit

def _print_state_audit(audit, label):
    print(f'\n[{label}] State resolution audit:')
    if audit['sibling']:
        print(f'  Sibling-city resolved ({len(audit["sibling"])}):')
        # dedupe: show distinct (wh, state) pairs only once
        _seen = set()
        for wh, city, st in audit['sibling']:
            if (wh, st) not in _seen:
                print(f"    {wh:<40} (city='{city}') -> {st}")
                _seen.add((wh, st))
    if audit['override']:
        print(f'  Override-resolved ({len(audit["override"])}):')
        _seen = set()
        for wh, st in audit['override']:
            if (wh, st) not in _seen:
                print(f"    {wh:<40} -> {st} [override]")
                _seen.add((wh, st))
    if audit['unknown']:
        _unique_unknown = sorted(set(audit['unknown']))
        print(f'  *** UNRESOLVED ({len(_unique_unknown)} distinct warehouses) — set to state="Unknown" ***')
        for wh in _unique_unknown:
            print(f'    {wh}')
        print('  ACTION: Add to WAREHOUSE_STATE_OVERRIDES (Cell 3) or fix SERP_WAREHOUSES.[State Province].')

df_tx['state'], _state_audit = resolve_state_series(df_tx['tech_warehouse'], df_tx['state'], df_hier)
_print_state_audit(_state_audit, 'df_tx')

# Remove R0/R1/DI/RN/etc region codes for states
_bogus = df_tx['state'].astype(str).str.match(r'^R\d|^DI$|^RN$', na=False)
if _bogus.any():
    _bogus_whs = sorted(df_tx.loc[_bogus,'tech_warehouse'].dropna().unique())
    print(f'\n*** WARNING: {_bogus.sum():,} rows still have region-code states. Warehouses: {_bogus_whs}')

_tot=df_tx['order_num'].nunique(); _n_field=(~df_tx['_unattributed']).sum()
_n_blank=df_tx['_is_blank_tech'].sum(); _n_unmatch=df_tx['_is_unmatched'].sum()
_n_ambig=df_tx['_is_ambiguous'].sum(); _n_intops=df_tx['_is_internal_ops'].sum(); _n_unattr=df_tx['_unattributed'].sum()
print(f'Matching complete. {_tot:,} tickets — field: {_n_field:,} ({_n_field/_tot*100:.1f}%) | dark: {_n_unattr:,} ({_n_unattr/_tot*100:.1f}%)')
print('\nState distribution (top 10):')
print(df_tx['state'].value_counts(dropna=False).head(10).to_string())

## Cell 10 — UNMATCHED NAME RESEARCH & HR EXPORT

In [ ]:
def _find_closest_candidates(fn,ln,top_n=3):
    fn_c,ln_c=_clean(fn),_clean(ln); results=[]
    for (emp_fn,emp_row) in _emp_by_last.get(ln_c,[]):
        results.append((levenshtein(fn_c,emp_fn),999,'fuzzy_first',emp_fn,ln_c,emp_row))
    prefix=ln_c[:4]
    for (fn2,ln2),emp_row2 in emp_idx.items():
        if ln2.startswith(prefix) and ln2!=ln_c: results.append((levenshtein(fn_c,fn2),1,'last_prefix',fn2,ln2,emp_row2))
    results.sort(key=lambda x:(x[0],x[1])); return results[:top_n]

_unmatched=(
    df_tx[df_tx['_is_unmatched']]
    .groupby(['techfirstname','techlastname'],dropna=False)
    .agg(ticket_count=('order_num','nunique'),patient_count=('record_id','nunique'),
         tx_warehouses=('tech_warehouse',lambda x:', '.join(sorted(x.dropna().unique()))),
         is_ambiguous=('_is_ambiguous','any'),collision_cand=('_collision_candidates','first'))
    .reset_index().sort_values('ticket_count',ascending=False)
)
_unmatched['unmatched_full']=_unmatched['techfirstname']+' '+_unmatched['techlastname']

candidate_rows=[]
for row in _unmatched.itertuples(index=False):
    cands=_find_closest_candidates(row.techfirstname,row.techlastname)
    if not cands:
        candidate_rows.append({'unmatched_full':row.unmatched_full,'unmatched_first':row.techfirstname,'unmatched_last':row.techlastname,'ticket_count':row.ticket_count,'patient_count':row.patient_count,'tx_warehouses':row.tx_warehouses,'is_ambiguous':row.is_ambiguous,'collision_cand':row.collision_cand,'rank':1,'candidate_name':'(no match)','candidate_title':None,'candidate_dept':None,'candidate_loc':None,'candidate_eid':None,'strategy':None,'score':None})
    else:
        for rank,(fd,_,strategy,ef,el,er) in enumerate(cands,start=1):
            candidate_rows.append({'unmatched_full':row.unmatched_full,'unmatched_first':row.techfirstname,'unmatched_last':row.techlastname,'ticket_count':row.ticket_count,'patient_count':row.patient_count,'tx_warehouses':row.tx_warehouses,'is_ambiguous':row.is_ambiguous,'collision_cand':row.collision_cand,'rank':rank,'candidate_name':f'{er.empfirstname} {er.emplastname}','candidate_title':er.title,'candidate_dept':er.dept,'candidate_loc':er.location,'candidate_eid':er.eid,'strategy':strategy,'score':fd})
tbl_unmatched_candidates=pd.DataFrame(candidate_rows)

_AUTO_RECOVERY_PASSES={'P4','P5_fallback','P5_intops_override','P6_intops_first_override','P7_intops_last_override'}
tbl_resolved_recoveries=(
    df_tx[df_tx['_pass_used'].isin(_AUTO_RECOVERY_PASSES)&(~df_tx['_is_internal_ops'])]
    .groupby(['techfirstname','techlastname','tech_warehouse','_pass_used','_matched_first','_matched_last','_matched_eid','_matched_title','_matched_dept','_matched_location','_matched_role_bucket'],dropna=False,as_index=False)
    .agg(ticket_count=('order_num','nunique'),patient_count=('record_id','nunique'),first_seen=('completed_date','min'),last_seen=('completed_date','max'))
    .rename(columns={'techfirstname':'ticket_first_name','techlastname':'ticket_last_name','_pass_used':'resolution_pass','_matched_first':'resolved_first_name','_matched_last':'resolved_last_name','_matched_eid':'resolved_employee_id','_matched_title':'resolved_title','_matched_dept':'resolved_dept','_matched_location':'resolved_emp_warehouse','_matched_role_bucket':'resolved_role_bucket'})
    .sort_values(['resolution_pass','ticket_count'],ascending=[True,False]).reset_index(drop=True)
)
tbl_ambiguous_collisions=(
    df_tx[df_tx['_is_ambiguous']]
    .groupby(['techfirstname','techlastname','tech_warehouse','_collision_candidates'],dropna=False,as_index=False)
    .agg(ticket_count=('order_num','nunique'),patient_count=('record_id','nunique'))
    .rename(columns={'techfirstname':'ticket_first_name','techlastname':'ticket_last_name','tech_warehouse':'warehouse','_collision_candidates':'candidates_emp_id'})
    .sort_values('ticket_count',ascending=False).reset_index(drop=True)
)
_unmatched_ticket_detail=(
    df_tx[df_tx['_is_unmatched']]
    [['order_num','record_id','tech_warehouse','region','vp','techfirstname','techlastname','reason','reason_category','priority_level','ticket_type','schedule_period','completed_date','delivery_year','delivery_month','_is_ambiguous','_collision_candidates']]
    .drop_duplicates(subset=['order_num']).sort_values(['techlastname','techfirstname','completed_date']).copy().reset_index(drop=True)
)
print(f'Unmatched: {len(_unmatched)} unique names | Auto-recovered: {tbl_resolved_recoveries["ticket_count"].sum():,} tickets')

_hr_xlsx=os.path.join(OUT_DIR,f'Unmatched_Techs_HR_{RUN_DATE}.xlsx')
_summary_rows=[('Total unique tickets',_tot),('Attributed to field tech',_n_field),('Internal ops (excluded)',_n_intops),('Blank tech name (dark)',_n_blank),('Unmatched tech name (dark)',_n_unmatch),('of which AMBIGUOUS',_n_ambig),('Total excluded',_n_unattr),('',''),('--- Per-pass attribution ---','')]
for k in ['P0','P1','P2','P3','P4','P5_fallback','P5_intops_override','P6_intops_first_override','P7_intops_last_override','ambiguous_collision','unmatched','unmatched_blank']:
    _summary_rows.append((k,_pass_counts.get(k,0)))
tbl_match_summary=pd.DataFrame(_summary_rows,columns=['Metric','Value'])
with pd.ExcelWriter(_hr_xlsx,engine='openpyxl') as xw:
    tbl_match_summary.to_excel(xw,sheet_name='Summary',index=False)
    _unmatched[['unmatched_full','techfirstname','techlastname','ticket_count','patient_count','tx_warehouses','is_ambiguous','collision_cand']].rename(columns={'unmatched_full':'name','techfirstname':'first_name','techlastname':'last_name','tx_warehouses':'warehouses_seen','collision_cand':'collision_candidates'}).to_excel(xw,sheet_name='Unmatched_Names',index=False)
    _unmatched_ticket_detail.to_excel(xw,sheet_name='Unmatched_Ticket_Det',index=False)
    tbl_resolved_recoveries.to_excel(xw,sheet_name='Resolved_Recoveries',index=False)
    tbl_ambiguous_collisions.to_excel(xw,sheet_name='Ambiguous_Collisions',index=False)
print(f'HR export: {_hr_xlsx}')

## Cell 11 — VISIT DEDUPLICATION (EXCHANGE CONSOLIDATION)

In [ ]:
_KEY=['record_id','techfirstname','techlastname','completed_date']
# 'exchange pair' = same patient + tech + date with BOTH a Pickup and a Delivery row. Only the Pickup is the redundant half; 
# the Delivery represents the visit. Service tickets (and anything else) in the same group are independent work and must be kept.
_tt_set=df_tx.groupby(_KEY)['ticket_type'].transform(lambda s:('Pickup' in s.values)and('Delivery' in s.values))
df_tx['_is_exchange_pair']=_tt_set.fillna(False).astype(bool)
# Drop ONLY the Pickup row(s) inside an exchange pair; keep every other row.
df_tx['_keep'] = ~(df_tx['_is_exchange_pair'] & (df_tx['ticket_type']=='Pickup'))
df_visits=df_tx[df_tx['_keep']].copy()
# v1.33.0 (M4): only the Delivery half of an exchange pair becomes 'Exchange'.
# Service (or other) tickets sharing the (patient, tech, date) group are
# independent work and keep their own type — they were previously mislabeled
# 'Exchange' (row counts were unaffected; the visit-type MIX was wrong).
df_visits['visit_type']=np.where(df_visits['_is_exchange_pair']&(df_visits['ticket_type']=='Delivery'),'Exchange',df_visits['ticket_type'])
n_raw,n_visits=len(df_tx),len(df_visits); n_ex=int(df_visits['_is_exchange_pair'].sum())
print(f'Raw: {n_raw:,}  -> Visits: {n_visits:,}  ({(1-n_visits/n_raw)*100:.1f}% reduction)  Exchange pairs: {n_ex:,}')
print(df_visits['visit_type'].value_counts().to_string())
_ex_kept=df_visits[df_visits['_is_exchange_pair']].groupby(_KEY)['order_num'].count()
print('PASS: Exchange dedup validated.' if (_ex_kept>1).sum()==0 else f'WARNING: {(_ex_kept>1).sum():,} groups >1 kept row.')

## Cell 13 — PRODUCTIVITY METRICS & HIERARCHY ROLLUP

In [ ]:
def parse_plc(raw):
    if not raw or not isinstance(raw,str): return ('','')
    p=raw.split(',',1); last=p[0].strip().title()
    first=p[1].strip().split()[0].title() if len(p)>1 else ''
    return (first,last)
df_hours_raw[['h_first','h_last']]=pd.DataFrame(df_hours_raw['employee_name'].apply(parse_plc).tolist(),index=df_hours_raw.index)
df_hours_raw['workdate']=pd.to_datetime(df_hours_raw['workdate'],errors='coerce')
# v1.33.0 (L3): dow_mon0 convention — Monday=0 .. Saturday=5, Sunday=6.
df_hours_monthly=(df_hours_raw.assign(delivery_year=df_hours_raw['workdate'].dt.year,delivery_month=df_hours_raw['workdate'].dt.month,schedule_period=np.where(df_hours_raw['dow_mon0']==5,'Saturday',np.where(df_hours_raw['dow_mon0']==6,'Sunday','Weekday'))).groupby(['h_first','h_last','delivery_year','delivery_month','schedule_period'],as_index=False).agg(total_workhours=('workhours','sum')))

# ── v1.33.0 (N1): OVERTIME — FLSA-style weekly inference ────────────────────
# Sun–Sat workweek. week_start = the Sunday on/before workdate. Computed on RAW
# per-person daily hours (NOT the visit-share pro-rated hours used elsewhere):
# the 40-hour threshold applies to the employee, not the warehouse. The week is
# attributed to the calendar month containing its Saturday (week end) — a week
# spanning a month boundary lands wholly in the later month. This is a
# documented simplification; do not sum OT across a month boundary and expect
# it to reconcile day-by-day with payroll.
# CAVEAT (also in Cell 3): if PLC Hours includes PTO, OT is overstated.
_hrs = df_hours_raw.copy()
# Days since Sunday: dow_mon0 Monday=0..Sunday=6  ->  Sunday offset = (dow_mon0 + 1) % 7
_hrs['week_start'] = _hrs['workdate'] - pd.to_timedelta((_hrs['dow_mon0'] + 1) % 7, unit='D')
_hrs['week_end']   = _hrs['week_start'] + pd.Timedelta(days=6)  # Saturday
df_ot_weekly = (_hrs.groupby(['h_first','h_last','week_start','week_end'], as_index=False)
                .agg(week_hours=('workhours','sum'), days_worked=('workdate','nunique')))
df_ot_weekly['ot_hours']  = (df_ot_weekly['week_hours'] - OT_WEEKLY_THRESHOLD).clip(lower=0).round(2)
df_ot_weekly['reg_hours'] = df_ot_weekly['week_hours'].clip(upper=OT_WEEKLY_THRESHOLD).round(2)
df_ot_weekly['delivery_year']  = df_ot_weekly['week_end'].dt.year
df_ot_weekly['delivery_month'] = df_ot_weekly['week_end'].dt.month
# Guard: partial FIRST week of the window can only UNDERstate OT (missing days
# lower the weekly sum) — flag it so trend readers know the edge is soft.
_first_wk = df_ot_weekly['week_start'].min()
if pd.notna(_first_wk) and _first_wk < pd.Timestamp(FILTER_START):
    print(f'  OT note: first workweek ({_first_wk.date()}) starts before FILTER_START — '
          f'that week\'s OT is understated (pre-window days not pulled).')
df_ot_monthly = (df_ot_weekly.groupby(['h_first','h_last','delivery_year','delivery_month'], as_index=False)
                 .agg(total_hours=('week_hours','sum'), ot_hours=('ot_hours','sum'),
                      reg_hours=('reg_hours','sum'), weeks_worked=('week_start','nunique'),
                      max_week_hours=('week_hours','max')))
df_ot_monthly['ot_pct'] = (df_ot_monthly['ot_hours'] / df_ot_monthly['total_hours'].replace(0,np.nan) * 100).round(2)
# Company-level monthly OT (ALL PCT-department payroll, matched or not — this is
# the labor-cost lens; the per-tech scorecard uses only name-matched techs).
tbl_ot_company_monthly = (df_ot_monthly.groupby(['delivery_year','delivery_month'], as_index=False)
                          .agg(total_hours=('total_hours','sum'), ot_hours=('ot_hours','sum'),
                               employees=('h_first','size')))
tbl_ot_company_monthly['ot_pct'] = (tbl_ot_company_monthly['ot_hours']/tbl_ot_company_monthly['total_hours'].replace(0,np.nan)*100).round(2)
tbl_ot_company_monthly['period'] = (tbl_ot_company_monthly['delivery_year'].astype(int).astype(str)+'-'+tbl_ot_company_monthly['delivery_month'].astype(int).astype(str).str.zfill(2))
print(f'OT weekly rows: {len(df_ot_weekly):,} | OT monthly rows: {len(df_ot_monthly):,} | '
      f'company OT hours in window: {df_ot_monthly["ot_hours"].sum():,.1f}')

_plc_idx={}; _plc_by_last=defaultdict(list)
for r in df_hours_raw[['h_first','h_last']].drop_duplicates().itertuples():
    fn,ln=_clean(r.h_first),_clean(r.h_last)
    if fn and ln: _plc_idx[(fn,ln)]=(r.h_first,r.h_last); _plc_by_last[ln].append((fn,(r.h_first,r.h_last)))
_plc_map_rows=[]
for row in df_tx[~df_tx['_unattributed']][['techfirstname','techlastname']].drop_duplicates().itertuples():
    fn,ln=row.techfirstname,row.techlastname; ln_c=_clean(ln); matched=None
    k=(_clean(fn),ln_c)
    if k in _plc_idx: matched=k
    if not matched:
        for cand in standardize_first_name(fn):
            k2=(_clean(cand),ln_c)
            if k2 in _plc_idx: matched=k2; break
    if not matched:
        best_key,best_d=None,FUZZY_EDIT_DIST+1
        for (i_fn,_) in _plc_by_last.get(ln_c,[]):
            d=levenshtein(_clean(fn),i_fn)
            if d<best_d: best_key,best_d=(_clean(i_fn),ln_c),d
        if best_key and best_d<=FUZZY_EDIT_DIST: matched=best_key
    if matched and matched in _plc_idx:
        pf,pl=_plc_idx[matched]; _plc_map_rows.append({'techfirstname':fn,'techlastname':ln,'plc_first':pf,'plc_last':pl})
df_plc_map=pd.DataFrame(_plc_map_rows)
print(f'Payroll matched: {len(df_plc_map):,} techs')

_PERIOD=['delivery_year','delivery_month','schedule_period']
df_va=df_visits[~df_visits['_unattributed']].copy()
tech_monthly=(df_va.groupby(['techfirstname','techlastname','tech_warehouse','region','vp','state','metro']+_PERIOD,as_index=False,dropna=False).agg(total_visits=('order_num','nunique'),unique_patients=('record_id','nunique')))
_raw_cts=(df_tx[~df_tx['_unattributed']].groupby(['techfirstname','techlastname','tech_warehouse']+_PERIOD,as_index=False).agg(total_tickets=('order_num','nunique')))
tech_monthly=tech_monthly.merge(_raw_cts,on=['techfirstname','techlastname','tech_warehouse']+_PERIOD,how='left')

_audit_total = tech_monthly['total_tickets'].fillna(0).sum()
_audit_dedup = df_tx[~df_tx['_unattributed']]['order_num'].nunique()
_audit_inflation = _audit_total - _audit_dedup
if abs(_audit_inflation) > _audit_dedup * 0.02:  # tolerance: 2%
    print(f'  *** WARNING: ticket count audit failed. '
          f'sum-across-rows={_audit_total:,.0f}, dedup={_audit_dedup:,}, '
          f'delta={_audit_inflation:+,.0f}. Investigate _raw_cts merge.')
else:
    print(f'  ✓ Ticket audit: sum-across-rows={_audit_total:,.0f}, dedup={_audit_dedup:,}, '
          f'delta={_audit_inflation:+,.0f} (within 2% tolerance).')
df_prod=tech_monthly.merge(df_plc_map,on=['techfirstname','techlastname'],how='left')
df_prod=df_prod.merge(df_hours_monthly.rename(columns={'h_first':'plc_first','h_last':'plc_last'}),on=['plc_first','plc_last','delivery_year','delivery_month','schedule_period'],how='left')
df_prod['total_workhours']=df_prod['total_workhours'].fillna(0)

_period_key=['techfirstname','techlastname','delivery_year','delivery_month','schedule_period']
_tech_period_visits=df_prod.groupby(_period_key,dropna=False)['total_visits'].transform('sum')
df_prod['_wh_count']=df_prod.groupby(_period_key,dropna=False)['tech_warehouse'].transform('nunique')
_share_when_no_visits=(1.0/df_prod['_wh_count'].replace(0,np.nan)).fillna(0.0)
_visit_share=(df_prod['total_visits']/_tech_period_visits.replace(0,np.nan)).fillna(_share_when_no_visits)
df_prod['total_workhours']=(df_prod['total_workhours']*_visit_share).round(4)
_n_zero_visit_periods = int((_tech_period_visits == 0).sum())
if _n_zero_visit_periods > 0:
    print(f'  *** WARNING: {_n_zero_visit_periods:,} tech-period rows had _tech_period_visits==0;')
    print(f'      workhour pro-rating fell back to even-split across warehouses. Investigate upstream.')

# active_days = distinct dates each tech had >=1 attributed ticket, at the
# (tech, warehouse, year, month, schedule_period) grain. This is the unit
# used by the new metric `tickets_per_active_day` and its rollup variant
# `tickets_per_active_day_per_tech`. It self-corrects for:
#   - PTO / sick days (tech isn't counted when not working)
#   - Mid-month hires & departures (only counted on days they actually ran tickets)
#   - Partial current month (no need for a separate "exclude current month" rule)
#   - Multi-warehouse fan-out (each tech-warehouse-day counted once at its warehouse)
# It is computed from df_tx (raw tickets) — NOT df_visits — so it reflects all
# attributed activity, not just deduplicated visits.
_active_days = (df_tx[~df_tx['_unattributed']]
    .groupby(['techfirstname','techlastname','tech_warehouse',
              'delivery_year','delivery_month','schedule_period'],
             dropna=False, as_index=False)
    .agg(active_days=('completed_date','nunique')))
_pre_merge_len = len(df_prod)  # v1.30.5 — audit C4
df_prod = df_prod.merge(
    _active_days,
    on=['techfirstname','techlastname','tech_warehouse',
        'delivery_year','delivery_month','schedule_period'],
    how='left')
if len(df_prod) != _pre_merge_len:
    raise AssertionError(
        f'_active_days merge multiplied rows: pre={_pre_merge_len:,} post={len(df_prod):,}. '
        f'Duplicate keys on right side. Investigate _active_days groupby.')
df_prod['active_days'] = df_prod['active_days'].fillna(0).astype(int)

def _count_weekdays(year,month):
    yr,mo=int(year),int(month)
    return sum(1 for d in range(1,_cal.monthrange(yr,mo)[1]+1) if _cal.weekday(yr,mo,d)<5)
_ym_pairs=(df_prod[['delivery_year','delivery_month']].dropna().drop_duplicates().astype(int).reset_index(drop=True))
_ym_pairs['weekday_days']=_ym_pairs.apply(lambda r:_count_weekdays(r['delivery_year'],r['delivery_month']),axis=1)
_ym_pairs['weekday_days_elapsed']=_ym_pairs.apply(lambda r:_count_weekdays_elapsed(r['delivery_year'],r['delivery_month']),axis=1)
df_prod=df_prod.merge(_ym_pairs,on=['delivery_year','delivery_month'],how='left')

_h=df_prod['total_workhours'].fillna(0)>=MIN_HOURS_MONTH
df_prod['visits_per_workhour'] =np.where(_h,df_prod['total_visits']/df_prod['total_workhours'],np.nan)
df_prod['tickets_per_workhour']=np.where(_h,df_prod['total_tickets']/df_prod['total_workhours'],np.nan)
df_prod['avg_daily_tickets'] =np.where((df_prod['schedule_period']=='Weekday')&(df_prod['weekday_days']>0),df_prod['total_tickets']/df_prod['weekday_days'],np.nan)
df_prod['mtd_avg_daily_tickets']=np.where((df_prod['schedule_period']=='Weekday')&(df_prod['weekday_days_elapsed']>0),df_prod['total_tickets']/df_prod['weekday_days_elapsed'],np.nan)
df_prod['tickets_per_active_day'] =np.where(df_prod['active_days']>0,df_prod['total_tickets']/df_prod['active_days'],np.nan)
df_prod['visits_per_active_day']  =np.where(df_prod['active_days']>0,df_prod['total_visits']/df_prod['active_days'],np.nan)
df_prod['_tech_key']=df_prod['techfirstname'].str.strip()+'|'+df_prod['techlastname'].str.strip()

def rollup(df,gcols):
    """Pooled-sum rollup. unique_patients recomputed from df_va to avoid double-counting.

    v1.29.0: now also aggregates `total_active_days` (sum of distinct tech-worked-days
    at this grain) and emits `tickets_per_active_day_per_tech` — total tickets divided
    by total tech-days actually worked. This metric is robust to PTO, mid-month hires,
    partial current month, and multi-warehouse fan-out — the four drivers that depress
    the legacy `avg_daily_tickets_per_tech` metric. Both metrics are returned so
    consumers can compare.
    """
    agg=df.groupby(gcols,dropna=False,as_index=False).agg(
        total_visits=('total_visits','sum'),total_tickets=('total_tickets','sum'),
        total_workhours=('total_workhours','sum'),tech_count=('_tech_key','nunique'),
        total_active_days=('active_days','sum'),  # NEW v1.29.0
        weekday_days=('weekday_days','max'),
        weekday_days_elapsed=('weekday_days_elapsed','max'))  # NEW v1.30.0
    _va_gcols=[c for c in gcols if c in df_va.columns]
    if _va_gcols:
        _pts=df_va.groupby(_va_gcols,dropna=False,as_index=False).agg(unique_patients=('record_id','nunique'))
        agg=agg.merge(_pts,on=_va_gcols,how='left')
    else:
        agg['unique_patients']=df_va['record_id'].nunique()
    _h2=agg['total_workhours']>=MIN_HOURS_MONTH
    agg['visits_per_workhour'] =np.where(_h2,agg['total_visits']/agg['total_workhours'],np.nan)
    agg['tickets_per_workhour']=np.where(_h2,agg['total_tickets']/agg['total_workhours'],np.nan)
    _wkday=(agg.get('schedule_period',pd.Series(['Weekday']*len(agg),index=agg.index))=='Weekday') if 'schedule_period' in agg.columns else pd.Series(True,index=agg.index)
    # Legacy metric: assumes every tech works every calendar weekday.
    agg['avg_daily_tickets_per_tech']=np.where(_wkday&(agg['tech_count']>0)&(agg['weekday_days'].fillna(0)>0),agg['total_tickets']/(agg['tech_count']*agg['weekday_days']),np.nan)
    # For complete past months this equals avg_daily_tickets_per_tech exactly.
    agg['mtd_avg_daily_tickets_per_tech']=np.where(_wkday&(agg['tech_count']>0)&(agg['weekday_days_elapsed'].fillna(0)>0),agg['total_tickets']/(agg['tech_count']*agg['weekday_days_elapsed']),np.nan)
    agg['tickets_per_active_day_per_tech']=np.where(agg['total_active_days']>0,agg['total_tickets']/agg['total_active_days'],np.nan)
    agg['visits_per_active_day_per_tech'] =np.where(agg['total_active_days']>0,agg['total_visits']/agg['total_active_days'],np.nan)
    # Utilization: fraction of expected tech-weekdays actually worked. <100% in any period with PTO, partial month, mid-month hires, or part-time techs.
    _exp=(agg['tech_count']*agg['weekday_days']).replace(0,np.nan)
    agg['utilization_pct']=(agg['total_active_days']/_exp*100).round(1) if 'schedule_period' not in agg.columns else np.where(_wkday,(agg['total_active_days']/_exp*100).round(1),np.nan)
    return agg

rollup_tech     =rollup(df_prod,['techfirstname','techlastname','tech_warehouse','region','vp','state','metro']+_PERIOD)
rollup_warehouse=rollup(df_prod,['tech_warehouse','region','vp','state','metro']+_PERIOD)
rollup_region   =rollup(df_prod,['region','vp','state']+_PERIOD)
rollup_vp       =rollup(df_prod,['vp']+_PERIOD)
rollup_total    =rollup(df_prod,_PERIOD)

_wh_rate=(rollup_warehouse[rollup_warehouse['schedule_period']=='Weekday'][['tech_warehouse','delivery_year','delivery_month','visits_per_workhour']].rename(columns={'visits_per_workhour':'_wh_rate'}))
df_prod=df_prod.merge(_wh_rate,on=['tech_warehouse','delivery_year','delivery_month'],how='left')
df_prod['workload_index']=df_prod['visits_per_workhour']/df_prod['_wh_rate']
df_prod.drop(columns=['_wh_rate'],inplace=True)
print(f'Rollup rows: tech={len(rollup_tech):,} wh={len(rollup_warehouse):,} region={len(rollup_region):,} vp={len(rollup_vp):,}')

_intops_tickets=df_tx[df_tx['_is_internal_ops']].copy()
if len(_intops_tickets)>0:
    _intops_tickets['_tech_full']=(_intops_tickets['techfirstname'].fillna('')+' '+_intops_tickets['techlastname'].fillna('')).str.strip()
    tbl_intops_monthly=(_intops_tickets.groupby(['tech_warehouse','region','vp','state','metro','delivery_year','delivery_month','schedule_period','ticket_type','reason_category'],dropna=False,as_index=False).agg(ticket_count=('order_num','nunique'),patient_count=('record_id','nunique'),roles_involved=('_matched_title',lambda x:' | '.join(sorted(x.dropna().unique()))),tech_names=('_tech_full',lambda x:', '.join(sorted(x.dropna().unique())))).sort_values(['delivery_year','delivery_month','ticket_count'],ascending=[True,True,False]))
    tbl_intops_monthly['period']=(tbl_intops_monthly['delivery_year'].astype(str)+'-'+tbl_intops_monthly['delivery_month'].astype(str).str.zfill(2))
    tbl_intops_by_wh=(tbl_intops_monthly.groupby(['tech_warehouse','region','vp','state','metro'],as_index=False).agg(total_tickets=('ticket_count','sum'),months_active=('period','nunique'),ticket_types=('ticket_type',lambda x:', '.join(sorted(x.dropna().unique()))),roles_involved=('roles_involved',lambda x:' | '.join(sorted(set(r for v in x.dropna() for r in v.split(' | ')))))).sort_values('total_tickets',ascending=False))
    print(f'Internal Ops: {len(_intops_tickets):,} tickets'); display(tbl_intops_by_wh.head(15))
else:
    tbl_intops_monthly=pd.DataFrame(); tbl_intops_by_wh=pd.DataFrame(); print('No internal-ops tickets.')

## Cell 13b — STATE & METRO ROLLUPS (v1.25.0)

Builds `rollup_state` and `rollup_metro` productivity tables that parallel the
warehouse and VP rollups.  Also prints metro membership diagnostics so you can
verify that the METRO_GROUPS patterns in Cell 3 are matching the right warehouses.

In [ ]:
# ── Diagnostic: show which warehouses mapped to which metro ──────────────────
_wh_metro_map = (
    df_tx[['tech_warehouse','state','metro']].drop_duplicates()
    .sort_values(['state','metro','tech_warehouse'])
)
print('=== Warehouse -> State / Metro mapping ===')
print(_wh_metro_map.to_string(index=False))
print(f'\nWarehouses with no metro assignment: {_wh_metro_map["metro"].isna().sum():,}')
print(f'Warehouses mapped to a metro:        {_wh_metro_map["metro"].notna().sum():,}')
for metro, grp in _wh_metro_map[_wh_metro_map['metro'].notna()].groupby('metro'):
    print(f'  {metro}: {sorted(grp["tech_warehouse"].tolist())}')

# ── State rollup ──────────────────────────────────────────────────────────────
# Groups all warehouses in the same state. schedule_period is included so the
# same Weekday/Saturday/Sunday split available at warehouse level is preserved.
rollup_state = rollup(df_prod, ['state'] + _PERIOD)

# ── Metro rollup ──────────────────────────────────────────────────────────────
# Only warehouses with a non-null metro assignment are included.
_prod_metro = df_prod[df_prod['metro'].notna()].copy()
if len(_prod_metro) > 0:
    rollup_metro = rollup(_prod_metro, ['metro', 'state'] + _PERIOD)
else:
    rollup_metro = pd.DataFrame()
    print('WARNING: No warehouses matched any metro pattern. Check METRO_GROUPS in Cell 3.')

print(f'\nState rollup rows:  {len(rollup_state):,}')
print(f'Metro rollup rows:  {len(rollup_metro):,}')

# ── _build_tpt and _add_mom helpers (needed for state/metro trend tables) ────
def _count_weekdays(year, month):
    yr, mo = int(year), int(month)
    return sum(1 for d in range(1, _cal.monthrange(yr,mo)[1]+1) if _cal.weekday(yr,mo,d) < 5)

_today = AS_OF_DATE
# Past months: legacy avg_daily_tickets_per_tech equals MTD (full denominator).
# Current month: legacy column nulled (would otherwise divide MTD numerator by
# full-month denominator, depressing the value). MTD column is the trend value.
_wkday_all = df_prod[df_prod['schedule_period']=='Weekday'].copy()
_wkday_all[['delivery_year','delivery_month']] = _wkday_all[['delivery_year','delivery_month']].astype(int)
_wkday_all['_tech_key'] = _wkday_all['techfirstname'].str.strip()+'|'+_wkday_all['techlastname'].str.strip()

def _build_tpt(df, group_cols):
    agg = df.groupby(group_cols, as_index=False, dropna=False).agg(
        tech_count=('_tech_key','nunique'),
        total_tickets=('total_tickets','sum'),
        total_visits=('total_visits','sum'),
        weekday_days=('weekday_days','max'),
        weekday_days_elapsed=('weekday_days_elapsed','max'),  # NEW v1.30.0
        total_active_days=('active_days','sum'))  # NEW v1.29.0
    denom_cal = agg['tech_count'].replace(0,np.nan) * agg['weekday_days'].replace(0,np.nan)
    denom_act = agg['total_active_days'].replace(0,np.nan)
    agg['avg_daily_tickets_per_tech']      = (agg['total_tickets']/denom_cal).round(3)
    agg['avg_daily_visits_per_tech']       = (agg['total_visits']/denom_cal).round(3)

    if 'delivery_year' in agg.columns and 'delivery_month' in agg.columns:
        _today_l = AS_OF_DATE  # v1.30.4
        _is_cur = ((agg['delivery_year'].astype(int)==_today_l.year) &
                   (agg['delivery_month'].astype(int)==_today_l.month))
        agg.loc[_is_cur, 'avg_daily_tickets_per_tech'] = np.nan
        agg.loc[_is_cur, 'avg_daily_visits_per_tech']  = np.nan
    denom_mtd = agg['tech_count'].replace(0,np.nan) * agg['weekday_days_elapsed'].replace(0,np.nan)
    agg['mtd_avg_daily_tickets_per_tech']  = (agg['total_tickets']/denom_mtd).round(3)
    agg['mtd_avg_daily_visits_per_tech']   = (agg['total_visits']/denom_mtd).round(3)
    agg['tickets_per_active_day_per_tech'] = (agg['total_tickets']/denom_act).round(3)
    agg['visits_per_active_day_per_tech']  = (agg['total_visits']/denom_act).round(3)
    agg['utilization_pct'] = (agg['total_active_days']/denom_cal*100).round(1)
    _one_day = (df[df['active_days'] < 2]
                .groupby(group_cols, dropna=False, as_index=False)
                .size()
                .rename(columns={'size': 'one_day_tech_rows'}))
    agg = agg.merge(_one_day, on=group_cols, how='left')
    agg['one_day_tech_rows'] = agg['one_day_tech_rows'].fillna(0).astype(int)
    return agg.sort_values(group_cols)

def _add_mom(tbl, key_cols, metric='tickets_per_active_day_per_tech'): 
    tbl = tbl.sort_values((key_cols if key_cols else [])+['delivery_year','delivery_month']).copy()
    if key_cols:
        tbl['mom_delta'] = tbl.groupby(key_cols, dropna=False)[metric].diff().round(3)
        tbl['mom_pct']   = tbl.groupby(key_cols, dropna=False)[metric].pct_change().mul(100).round(1)
    else:
        tbl['mom_delta'] = tbl[metric].diff().round(3)
        tbl['mom_pct']   = tbl[metric].pct_change().mul(100).round(1)
    tbl['trend_flag'] = np.select([tbl['mom_pct'].isna(),tbl['mom_pct']>5.0,tbl['mom_pct']<-5.0],['baseline','improving','declining'],default='stable')
    if 'one_day_tech_rows' in tbl.columns:
        if key_cols:
            _mean = tbl.groupby(key_cols, dropna=False)['one_day_tech_rows'].transform(lambda s: s.rolling(6, min_periods=3).mean())
            _std  = tbl.groupby(key_cols, dropna=False)['one_day_tech_rows'].transform(lambda s: s.rolling(6, min_periods=3).std())
        else:
            _mean = tbl['one_day_tech_rows'].rolling(6, min_periods=3).mean()
            _std  = tbl['one_day_tech_rows'].rolling(6, min_periods=3).std()
        tbl['coverage_anomaly_flag'] = (tbl['one_day_tech_rows'] > (_mean + 2 * _std)).fillna(False)
    tbl['period'] = tbl['delivery_year'].astype(str)+'-'+tbl['delivery_month'].astype(str).str.zfill(2)
    _today_l2 = AS_OF_DATE
    tbl['is_partial_month'] = ((tbl['delivery_year'].astype(int)==_today_l2.year) &
                               (tbl['delivery_month'].astype(int)==_today_l2.month))
    return tbl

# State daily-tickets-per-tech trend
tbl_state_tpt  = _add_mom(_build_tpt(_wkday_all, ['state','delivery_year','delivery_month']), ['state'])

# Metro daily-tickets-per-tech trend (non-null metro only)
_wkday_metro = _wkday_all[_wkday_all['metro'].notna()].copy()
tbl_metro_tpt  = (_add_mom(_build_tpt(_wkday_metro, ['metro','delivery_year','delivery_month']), ['metro'])
                  if len(_wkday_metro) > 0 else pd.DataFrame())

# VP / WH tables (used by charts in Cell 15 and Cell 20)
_vp_w  = _wkday_all[_wkday_all['vp'].notna()&(_wkday_all['vp'].astype(str).str.strip()!='')]
tbl_vp_tpt    = _add_mom(_build_tpt(_vp_w, ['vp','delivery_year','delivery_month']), ['vp'])
tbl_wh_tpt    = _add_mom(_build_tpt(_wkday_all, ['tech_warehouse','region','vp','state','metro','delivery_year','delivery_month']), ['tech_warehouse'])
tbl_grand_tpt = _add_mom(_build_tpt(_wkday_all, ['delivery_year','delivery_month']), [])

print(f'State tpt: {len(tbl_state_tpt):,} rows  |  Metro tpt: {len(tbl_metro_tpt):,} rows')
print(f'VP tpt: {len(tbl_vp_tpt):,}  WH tpt: {len(tbl_wh_tpt):,}  Grand tpt: {len(tbl_grand_tpt):,}')

# Intops overlay tables for charts
tbl_intops_tpt_wh=pd.DataFrame(columns=['tech_warehouse','region','vp','state','metro','delivery_year','delivery_month','total_intops_tickets','weekday_days','intops_daily_tickets','period'])
tbl_intops_tpt_vp=pd.DataFrame(columns=['vp','delivery_year','delivery_month','total_intops_tickets','weekday_days','intops_daily_tickets','period'])
if len(tbl_intops_monthly)>0:
    _io_wkday=tbl_intops_monthly[tbl_intops_monthly['schedule_period']=='Weekday'].copy()
    _io_wkday[['delivery_year','delivery_month']]=_io_wkday[['delivery_year','delivery_month']].astype(int)
    _io_wh=_io_wkday.groupby(['tech_warehouse','region','vp','state','metro','delivery_year','delivery_month'],as_index=False).agg(total_intops_tickets=('ticket_count','sum'))
    _io_wh=_io_wh.merge(_ym_pairs,on=['delivery_year','delivery_month'],how='left')
    _io_wh['intops_daily_tickets']=(_io_wh['total_intops_tickets']/_io_wh['weekday_days_elapsed'].replace(0,np.nan)).round(3)
    _io_wh['period']=_io_wh['delivery_year'].astype(str)+'-'+_io_wh['delivery_month'].astype(str).str.zfill(2)
    tbl_intops_tpt_wh=_io_wh.sort_values(['tech_warehouse','delivery_year','delivery_month']).reset_index(drop=True)
    _io_vp=_io_wkday.groupby(['vp','delivery_year','delivery_month'],as_index=False).agg(total_intops_tickets=('ticket_count','sum'))
    _io_vp=_io_vp.merge(_ym_pairs,on=['delivery_year','delivery_month'],how='left')
    _io_vp['intops_daily_tickets']=(_io_vp['total_intops_tickets']/_io_vp['weekday_days_elapsed'].replace(0,np.nan)).round(3)
    _io_vp['period']=_io_vp['delivery_year'].astype(str)+'-'+_io_vp['delivery_month'].astype(str).str.zfill(2)
    tbl_intops_tpt_vp=_io_vp.sort_values(['vp','delivery_year','delivery_month']).reset_index(drop=True)

## Cell 14 — PAYROLL HOURS DEBUG

In [ ]:
_multi=df_prod[df_prod['_wh_count']>1].copy()
print(f'Tech-months with multiple warehouses: {len(_multi):,}')
if not _multi.empty:
    display(_multi.groupby(['techfirstname','techlastname','delivery_year','delivery_month','schedule_period'],dropna=False).agg(warehouses=('tech_warehouse',lambda x:', '.join(sorted(x.dropna().unique()))),wh_count=('_wh_count','first'),total_visits=('total_visits','sum'),allocated_hours=('total_workhours','sum')).reset_index().sort_values('allocated_hours',ascending=False).head(20))
_hours_check=(df_prod.groupby(['techfirstname','techlastname','delivery_year','delivery_month','schedule_period'],dropna=False)['total_workhours'].sum().reset_index(name='total_hrs'))
_still_bad=_hours_check[_hours_check['total_hrs']>250].sort_values('total_hrs',ascending=False)
print(f'Tech-period-months >250 hrs: {len(_still_bad):,}')
if len(_still_bad)>0: display(_still_bad)

## Cell 15 — RANKINGS (TOP & BOTTOM PERFORMERS)

In [ ]:
TOP_N=10; MIN_MONTHS_RANK=3

def _disjoint_top_bottom(ranked_df, n=TOP_N, value_col=None):
    """Returns (top, bottom) DataFrames with no row overlap.

    ranked_df must already be sorted descending by the ranking metric.
    value_col is used only to sort bottom ascending for display."""
    if len(ranked_df) == 0:
        return ranked_df.iloc[:0].copy(), ranked_df.iloc[:0].copy()
    top = ranked_df.head(n).copy()
    # Bottom: rows starting at index n (i.e. strictly after the top slice).
    # If len < n, bottom is empty. If len in [n, 2n), bottom < n. If len >= 2n, bottom == n.
    bottom = ranked_df.iloc[n:].tail(n).copy()
    if value_col is not None and len(bottom) > 0:
        bottom = bottom.sort_values(value_col, ascending=True).reset_index(drop=True)
    return top, bottom

# ── Tech ranking ────────────────────────────────────────────────────────────
_prod_complete = df_prod[df_prod['schedule_period']=='Weekday'].copy()
_prod_complete['_denom_days'] = _prod_complete['weekday_days_elapsed'].fillna(_prod_complete['weekday_days'])
# v1.33.0 (M1): rank on tickets_per_active_day, the adopted headline denominator.
# The legacy calendar-weekday metric penalized part-timers, PTO-heavy months,
# mid-month hires and cross-coverage techs (each half-entity looked half as
# productive) — bottom-10 lists built on it generate false accusations. Legacy
# retained one quarter as avg_daily_tickets_DEPRECATED per deprecation convention.
_tech_rank = (_prod_complete
    .groupby(['techfirstname','techlastname','tech_warehouse','vp','state','metro'], dropna=False)
    .agg(total_tickets=('total_tickets','sum'),
         total_days=('_denom_days','sum'),
         total_active_days=('active_days','sum'),
         months_active=('delivery_month','nunique'))
    .reset_index())
_tech_rank['avg_daily_tickets_DEPRECATED'] = (_tech_rank['total_tickets']
    / _tech_rank['total_days'].replace(0,np.nan)).round(3)
_tech_rank['tickets_per_active_day'] = (_tech_rank['total_tickets']
    / _tech_rank['total_active_days'].replace(0,np.nan)).round(3)
_tech_rank = (_tech_rank[(_tech_rank['months_active']>=MIN_MONTHS_RANK)
                         & (_tech_rank['total_active_days']>=MIN_ACTIVE_DAYS_RANK)]
    .sort_values('tickets_per_active_day', ascending=False).reset_index(drop=True))
_tech_rank['rank'] = _tech_rank.index + 1
tbl_top_techs, tbl_bot_techs = _disjoint_top_bottom(_tech_rank, n=TOP_N, value_col='tickets_per_active_day')
# Bottom rank values reflect position in full ranking (e.g. ranks 20-11 if universe = 20)
if len(tbl_bot_techs) > 0:
    _start_rank = len(_tech_rank); _end_rank = _start_rank - len(tbl_bot_techs)
    tbl_bot_techs['rank'] = list(range(_start_rank, _end_rank, -1))

# ── Warehouse ranking ───────────────────────────────────────────────────────
_wh_rank_src = tbl_wh_tpt.copy()
_wh_rank_src['_tech_days_month']        = _wh_rank_src['total_active_days'].fillna(0)
_wh_rank_src['_legacy_tech_days_month'] = (_wh_rank_src['tech_count'].fillna(0)
    * _wh_rank_src['weekday_days_elapsed'].fillna(_wh_rank_src['weekday_days']).fillna(0))

_wh_latest = (_wh_rank_src
    .sort_values(['tech_warehouse','delivery_year','delivery_month'])
    .groupby('tech_warehouse', as_index=False, dropna=False)
    .agg(region=('region','last'), vp=('vp','last'),
         state=('state','last'), metro=('metro','last')))

# Sum-of-pooled rate: one row per warehouse regardless of VP changes
_wh_agg = (_wh_rank_src
    .groupby('tech_warehouse', dropna=False, as_index=False)
    .agg(total_tickets=('total_tickets','sum'),
         total_tech_days=('_tech_days_month','sum'),
         _legacy_total_tech_days=('_legacy_tech_days_month','sum'),
         months_active=('period','nunique'),
         avg_tech_count=('tech_count','mean')))

_wh_rank = _wh_agg.merge(_wh_latest, on='tech_warehouse', how='left')
_wh_rank['tickets_per_active_day_per_tech'] = (_wh_rank['total_tickets']
    / _wh_rank['total_tech_days'].replace(0,np.nan)).round(3)
_wh_rank['avg_daily_tickets_per_tech'] = _wh_rank['tickets_per_active_day_per_tech']
_wh_rank = (_wh_rank[_wh_rank['months_active']>=MIN_MONTHS_RANK]
    .sort_values('tickets_per_active_day_per_tech', ascending=False).reset_index(drop=True))
_wh_rank['rank'] = _wh_rank.index + 1

tbl_top_wh, tbl_bot_wh = _disjoint_top_bottom(_wh_rank, n=TOP_N, value_col='tickets_per_active_day_per_tech')
if len(tbl_bot_wh) > 0:
    _start_rank = len(_wh_rank); _end_rank = _start_rank - len(tbl_bot_wh)
    tbl_bot_wh['rank'] = list(range(_start_rank, _end_rank, -1))

print(f'Tech ranked: {len(_tech_rank):,} (top={len(tbl_top_techs)}, bottom={len(tbl_bot_techs)})')
print(f'WH ranked:   {len(_wh_rank):,} (top={len(tbl_top_wh)}, bottom={len(tbl_bot_wh)})')
if len(_wh_rank) < 2*TOP_N:
    print(f'  NOTE: WH universe ({len(_wh_rank)}) < 2*TOP_N ({2*TOP_N}); bottom list is {len(tbl_bot_wh)} entries.')
_state_dist = _wh_rank['state'].fillna('(Unknown)').value_counts().to_dict()
print(f'  Ranking universe state distribution: {_state_dist}')
_top_states = tbl_top_wh['state'].fillna('(Unknown)').value_counts().to_dict() if len(tbl_top_wh)>0 else {}
_bot_states = tbl_bot_wh['state'].fillna('(Unknown)').value_counts().to_dict() if len(tbl_bot_wh)>0 else {}
print(f'  Top-{TOP_N} states: {_top_states}')
print(f'  Bottom-{TOP_N} states: {_bot_states}')

if len(tbl_top_wh) > 0 and len(tbl_bot_wh) > 0:
    _overlap = set(tbl_top_wh['tech_warehouse']) & set(tbl_bot_wh['tech_warehouse'])
    assert not _overlap, f'BUG: WH top/bottom overlap = {_overlap}'

display(tbl_top_techs[['rank','techfirstname','techlastname','tech_warehouse','vp','state','metro','tickets_per_active_day','avg_daily_tickets_DEPRECATED','months_active']])

## Cell 16 — REDELIVERY ANALYSIS

In [ ]:
print(f'df_redel raw rows (post-SQL filter): {len(df_redel):,}')
_df_redel_orig=df_redel.copy()
_rd_from_convert=pd.to_datetime(_df_redel_orig['rd_date'],errors='coerce')
_rd_from_raw    =pd.to_datetime(_df_redel_orig['rd_datetime_raw'],errors='coerce')
_best_date=_rd_from_convert.where(_rd_from_convert.notna(),_rd_from_raw)
_df_redel_orig['_best_rd_date']=_best_date
_n_rescued=(_rd_from_convert.isna()&_rd_from_raw.notna()).sum()
if _n_rescued>0: print(f'  Rescued via raw parse: {_n_rescued:,}')
_n_unparseable = (_rd_from_convert.isna() & _rd_from_raw.isna()).sum()
if _n_unparseable>0:
    print(f'  *** WARNING: {_n_unparseable:,} redelivery rows have UNPARSEABLE Completion_DateTime')
    print(f'      These rows are DROPPED from all redelivery analytics.')
    print(f'      Sample unparseable raw values: {_df_redel_orig.loc[(_rd_from_convert.isna() & _rd_from_raw.isna()),"rd_datetime_raw"].dropna().astype(str).head(5).tolist()}')
_mask=(_df_redel_orig['_best_rd_date']>=pd.Timestamp(FILTER_START))&(_df_redel_orig['_best_rd_date']<=pd.Timestamp(FILTER_END))
df_redel=_df_redel_orig[_mask].copy()
print(f'  Window filter: {len(df_redel):,} kept / {(~_mask).sum():,} dropped (incl. {_n_unparseable:,} unparseable)')
df_redel['rd_date']=df_redel['_best_rd_date'].dt.date; df_redel.drop(columns=['_best_rd_date'],inplace=True)
df_redel['event_key'] = (df_redel['orig_order_num'].astype(str).str.strip()
                          + '|' + df_redel['rd_date'].astype(str))
print(f'After filter: {len(df_redel):,} rows  |  {df_redel["event_key"].nunique():,} unique (orig_order, rd_date) events')

def _parse_products(raw):
    if not raw or not isinstance(raw,str) or raw.strip()=='': return ['Unknown / Blank']
    items=[p.strip() for p in raw.replace('\r\n','\n').replace('\r','\n').split('\n') if p.strip()]
    return items or ['Unknown / Blank']

df_redel_exploded=(df_redel.copy().assign(product_list=lambda d:d['rd_products'].apply(_parse_products)).explode('product_list').rename(columns={'product_list':'product'}).reset_index(drop=True))
df_redel_exploded['product']=df_redel_exploded['product'].str.strip()
df_redel_exploded.loc[df_redel_exploded['product'].isna()|(df_redel_exploded['product']==''),'product']='Unknown / Blank'

df_tx['order_num']=df_tx['order_num'].astype(str).str.strip()
df_redel_exploded['orig_order_num']=df_redel_exploded['orig_order_num'].astype(str).str.strip()
_order_wh_counts=df_tx.groupby(['order_num','tech_warehouse'],dropna=False).size().reset_index(name='_n')
_modal_wh=(_order_wh_counts.sort_values(['order_num','_n'],ascending=[True,False]).drop_duplicates(subset=['order_num'],keep='first')[['order_num','tech_warehouse']])
_tx_keys=(df_tx[['order_num','tech_warehouse','region','vp','state','metro','techfirstname','techlastname','delivery_year','delivery_month','schedule_period']].merge(_modal_wh,on=['order_num','tech_warehouse'],how='inner').drop_duplicates(subset=['order_num']).rename(columns={'tech_warehouse':'tx_warehouse','region':'tx_region','vp':'tx_vp','state':'tx_state','metro':'tx_metro','techfirstname':'tx_techfirstname','techlastname':'tx_techlastname','delivery_year':'tx_delivery_year','delivery_month':'tx_delivery_month','schedule_period':'tx_schedule_period'}))
redel_linked=df_redel_exploded.merge(_tx_keys,left_on='orig_order_num',right_on='order_num',how='inner')
print(f'Unlinked: {len(df_redel_exploded)-len(redel_linked):,}  Linked: {len(redel_linked):,}')
# v1.33.0 (M2): reconcile UNLINKED redeliveries by month (of the redelivery date).
# These are events whose originating order is outside the ticket window (e.g. a
# Jan-2025 redelivery of a Dec-2024 original) or under a non-included Reason.
# They are EXCLUDED from every linked redelivery metric — the early months of the
# window undercount by design. This table makes the exclusion auditable in Excel
# (sheet 'Redel_Unlinked_Monthly'); footnote any board chart that spans the edge.
_unlinked = df_redel_exploded[~df_redel_exploded['orig_order_num'].isin(set(_tx_keys['order_num']))].copy()
_unlinked['_rd_dt'] = pd.to_datetime(_unlinked['rd_date'], errors='coerce')
tbl_redel_unlinked_monthly = (_unlinked
    .assign(delivery_year=_unlinked['_rd_dt'].dt.year, delivery_month=_unlinked['_rd_dt'].dt.month)
    .groupby(['delivery_year','delivery_month'], dropna=False, as_index=False)
    .agg(unlinked_events=('event_key','nunique'), unlinked_items=('product','count')))
tbl_redel_unlinked_monthly['period'] = (tbl_redel_unlinked_monthly['delivery_year'].astype('Int64').astype(str)
    + '-' + tbl_redel_unlinked_monthly['delivery_month'].astype('Int64').astype(str).str.zfill(2))
print(f'  Unlinked reconciliation: {tbl_redel_unlinked_monthly["unlinked_events"].sum():,} events across '
      f'{len(tbl_redel_unlinked_monthly)} month(s) — exported to Redel_Unlinked_Monthly.')
for _src,_dst in [('tx_warehouse','tech_warehouse'),('tx_region','region'),('tx_vp','vp'),('tx_state','state'),('tx_metro','metro'),('tx_techfirstname','techfirstname'),('tx_techlastname','techlastname'),('tx_delivery_year','delivery_year'),('tx_delivery_month','delivery_month'),('tx_schedule_period','schedule_period')]:
    redel_linked[_dst]=redel_linked[_src]
redel_linked.drop(columns=[c for c in redel_linked.columns if c.startswith('tx_')],inplace=True,errors='ignore')

redel_linked['period']=(redel_linked['delivery_year'].astype(int).astype(str)+'-'+redel_linked['delivery_month'].astype(int).astype(str).str.zfill(2))
tbl_redel_product=(redel_linked.groupby('product').agg(
    redelivery_count=('event_key','nunique'),
    warehouses=('tech_warehouse',lambda x:x.nunique())
).reset_index().sort_values('redelivery_count',ascending=False))
tbl_redel_monthly=(redel_linked.groupby(['delivery_year','delivery_month']).agg(
    redelivery_count=('event_key','nunique'),
    redelivery_items=('product','count')   # count of product rows (items returned)
).reset_index().sort_values(['delivery_year','delivery_month']))
tbl_redel_monthly['period']=tbl_redel_monthly['delivery_year'].astype(str)+'-'+tbl_redel_monthly['delivery_month'].astype(str).str.zfill(2)
_tx_monthly_total = df_tx.groupby(['delivery_year','delivery_month'],dropna=False,as_index=False).agg(total_tickets=('order_num','nunique'))
tbl_redel_monthly = tbl_redel_monthly.merge(_tx_monthly_total,on=['delivery_year','delivery_month'],how='left')
tbl_redel_monthly['redel_pct_of_tickets'] = (tbl_redel_monthly['redelivery_count']/tbl_redel_monthly['total_tickets'].replace(0,np.nan)*100).round(2)
display(tbl_redel_product.head(10))
print(f'Redelivery exploded rows: {len(redel_linked):,}')

## Cell 16b — MONTHLY REDELIVERIES BY WAREHOUSE & STATE/METRO (v1.25.0)

Produces three monthly redelivery trend tables:
- `tbl_redel_monthly_wh`    — one row per (warehouse, year, month)
- `tbl_redel_monthly_state` — one row per (state, year, month)
- `tbl_redel_monthly_metro` — one row per (metro, year, month), metro-assigned only

In [ ]:
# ── Monthly redeliveries by warehouse ────────────────────────────────────────
tbl_redel_monthly_wh = (
    redel_linked
    .groupby(['tech_warehouse','region','vp','state','metro','delivery_year','delivery_month'], dropna=False, as_index=False)
    .agg(
        redelivery_count = ('event_key', 'nunique'),
        redelivery_items = ('product',   'count'),
        distinct_products= ('product',   'nunique'),
    )
    .sort_values(['tech_warehouse','delivery_year','delivery_month'])
)
tbl_redel_monthly_wh['period'] = (
    tbl_redel_monthly_wh['delivery_year'].astype(int).astype(str) + '-' +
    tbl_redel_monthly_wh['delivery_month'].astype(int).astype(str).str.zfill(2)
)

# ── Monthly redeliveries by state ────────────────────────────────────────────
tbl_redel_monthly_state = (
    redel_linked
    .groupby(['state','delivery_year','delivery_month'], dropna=False, as_index=False)
    .agg(
        redelivery_count    = ('event_key',      'nunique'),
        redelivery_items    = ('product',        'count'),
        distinct_warehouses = ('tech_warehouse', 'nunique'),
        distinct_products   = ('product',        'nunique'),
    )
    .sort_values(['state','delivery_year','delivery_month'])
)
tbl_redel_monthly_state['period'] = (
    tbl_redel_monthly_state['delivery_year'].astype(int).astype(str) + '-' +
    tbl_redel_monthly_state['delivery_month'].astype(int).astype(str).str.zfill(2)
)

# ── Monthly redeliveries by metro (non-null only) ─────────────────────────────
_redel_metro = redel_linked[redel_linked['metro'].notna()].copy()
if len(_redel_metro) > 0:
    tbl_redel_monthly_metro = (
        _redel_metro
        .groupby(['metro','state','delivery_year','delivery_month'], dropna=False, as_index=False)
        .agg(
            redelivery_count    = ('event_key',      'nunique'),
            redelivery_items    = ('product',        'count'),
            distinct_warehouses = ('tech_warehouse', 'nunique'),
            distinct_products   = ('product',        'nunique'),
        )
        .sort_values(['metro','delivery_year','delivery_month'])
    )
    tbl_redel_monthly_metro['period'] = (
        tbl_redel_monthly_metro['delivery_year'].astype(int).astype(str) + '-' +
        tbl_redel_monthly_metro['delivery_month'].astype(int).astype(str).str.zfill(2)
    )
else:
    tbl_redel_monthly_metro = pd.DataFrame()

print(f'Redelivery monthly by WH:    {len(tbl_redel_monthly_wh):,} rows')
print(f'Redelivery monthly by state: {len(tbl_redel_monthly_state):,} rows')
print(f'Redelivery monthly by metro: {len(tbl_redel_monthly_metro):,} rows')
print('\nState totals:')
print(tbl_redel_monthly_state.groupby('state')['redelivery_count'].sum().sort_values(ascending=False).to_string())
if len(tbl_redel_monthly_metro) > 0:
    print('\nMetro totals:')
    print(tbl_redel_monthly_metro.groupby('metro')['redelivery_count'].sum().sort_values(ascending=False).to_string())

# Monthly tickets by warehouse
_tx_monthly_wh = (df_tx.groupby(['tech_warehouse','delivery_year','delivery_month'],dropna=False,as_index=False)
                  .agg(total_tickets=('order_num','nunique')))
tbl_redel_monthly_wh = tbl_redel_monthly_wh.merge(_tx_monthly_wh, on=['tech_warehouse','delivery_year','delivery_month'], how='left')
tbl_redel_monthly_wh['redel_pct_of_tickets'] = (tbl_redel_monthly_wh['redelivery_count']/tbl_redel_monthly_wh['total_tickets'].replace(0,np.nan)*100).round(2)

# Monthly tickets by state
_tx_monthly_state = (df_tx.groupby(['state','delivery_year','delivery_month'],dropna=False,as_index=False)
                     .agg(total_tickets=('order_num','nunique')))
tbl_redel_monthly_state = tbl_redel_monthly_state.merge(_tx_monthly_state, on=['state','delivery_year','delivery_month'], how='left')
tbl_redel_monthly_state['redel_pct_of_tickets'] = (tbl_redel_monthly_state['redelivery_count']/tbl_redel_monthly_state['total_tickets'].replace(0,np.nan)*100).round(2)

# Monthly tickets by metro
if len(tbl_redel_monthly_metro) > 0:
    _tx_monthly_metro = (df_tx[df_tx['metro'].notna()].groupby(['metro','delivery_year','delivery_month'],dropna=False,as_index=False)
                         .agg(total_tickets=('order_num','nunique')))
    tbl_redel_monthly_metro = tbl_redel_monthly_metro.merge(_tx_monthly_metro, on=['metro','delivery_year','delivery_month'], how='left')
    tbl_redel_monthly_metro['redel_pct_of_tickets'] = (tbl_redel_monthly_metro['redelivery_count']/tbl_redel_monthly_metro['total_tickets'].replace(0,np.nan)*100).round(2)

print(f'\nMonthly redel rates merged. Sample (state, latest period):')
if len(tbl_redel_monthly_state) > 0:
    _latest = tbl_redel_monthly_state.sort_values(['delivery_year','delivery_month']).iloc[-3:]
    print(_latest[['state','period','redelivery_count','total_tickets','redel_pct_of_tickets']].to_string(index=False))

## Cell 17 — REDELIVERY & LOST EQUIPMENT ACCOUNTABILITY

In [ ]:
# v1.33.0 (L1): full-name key so distinct techs sharing a first name don't collapse.
redel_linked['_tech_full']=(redel_linked['techfirstname'].fillna('')+' '+redel_linked['techlastname'].fillna('')).str.strip()
_redel_attrib=redel_linked[redel_linked['techfirstname'].fillna('').str.strip().ne('')&redel_linked['techlastname'].fillna('').str.strip().ne('')].copy()
tbl_tech_redeliveries=(_redel_attrib.groupby(['techfirstname','techlastname','tech_warehouse','region','vp','state','metro'],dropna=False).agg(redelivery_count=('event_key','nunique'),redelivery_items=('product','count')).reset_index().sort_values('redelivery_count',ascending=False))
# v1.33.0 (H2): denominator grain must match the numerator — visits at
# (tech, warehouse), not company-wide per name. The old name-only total
# understated every multi-warehouse tech's rate and repeated the same
# denominator on each warehouse row.
_tech_visit_totals=(df_visits[~df_visits['_unattributed']].groupby(['techfirstname','techlastname','tech_warehouse'],dropna=False).agg(total_visits=('order_num','nunique')).reset_index())
tbl_tech_redeliveries=tbl_tech_redeliveries.merge(_tech_visit_totals,on=['techfirstname','techlastname','tech_warehouse'],how='left')
tbl_tech_redeliveries['redel_per_100_visits']=(tbl_tech_redeliveries['redelivery_count']/tbl_tech_redeliveries['total_visits'].replace(0,np.nan)*100).round(2)
tbl_wh_redeliveries=(redel_linked.groupby(['tech_warehouse','region','vp','state','metro'],dropna=False).agg(redelivery_count=('event_key','nunique'),redelivery_items=('product','count'),affected_techs=('_tech_full',lambda x:x[x!=''].nunique())).reset_index().sort_values('redelivery_count',ascending=False))

# State-level redelivery summary
tbl_state_redeliveries = (
    redel_linked.groupby(['state'], dropna=False, as_index=False)
    .agg(redelivery_count=('event_key','nunique'), redelivery_items=('product','count'),
         distinct_warehouses=('tech_warehouse','nunique'), affected_techs=('_tech_full','nunique'))
    .sort_values('redelivery_count', ascending=False)
)
# Metro-level redelivery summary
tbl_metro_redeliveries = pd.DataFrame()
if redel_linked['metro'].notna().any():
    tbl_metro_redeliveries = (
        redel_linked[redel_linked['metro'].notna()]
        .groupby(['metro','state'], dropna=False, as_index=False)
        .agg(redelivery_count=('event_key','nunique'), redelivery_items=('product','count'),
             distinct_warehouses=('tech_warehouse','nunique'), affected_techs=('_tech_full','nunique'))
        .sort_values('redelivery_count', ascending=False)
    )

print('WH redeliveries (top 10):'); display(tbl_wh_redeliveries.head(10))
print('\nState redeliveries:'); display(tbl_state_redeliveries)
if len(tbl_metro_redeliveries)>0:
    print('\nMetro redeliveries:'); display(tbl_metro_redeliveries)

# v1.33.0 (H2): tickets denominator at (tech, warehouse) grain to match numerator.
_tx_by_tech = (df_tx[~df_tx['_unattributed']]
               .groupby(['techfirstname','techlastname','tech_warehouse'],dropna=False,as_index=False)
               .agg(total_tickets=('order_num','nunique')))
tbl_tech_redeliveries = tbl_tech_redeliveries.merge(_tx_by_tech, on=['techfirstname','techlastname','tech_warehouse'], how='left')
tbl_tech_redeliveries['redel_pct_of_tickets'] = (tbl_tech_redeliveries['redelivery_count']/tbl_tech_redeliveries['total_tickets'].replace(0,np.nan)*100).round(2)

# Warehouse scope
_tx_by_wh = df_tx.groupby('tech_warehouse',dropna=False,as_index=False).agg(total_tickets=('order_num','nunique'))
tbl_wh_redeliveries = tbl_wh_redeliveries.merge(_tx_by_wh, on='tech_warehouse', how='left')
tbl_wh_redeliveries['redel_pct_of_tickets'] = (tbl_wh_redeliveries['redelivery_count']/tbl_wh_redeliveries['total_tickets'].replace(0,np.nan)*100).round(2)

# State scope
_tx_by_state = df_tx.groupby('state',dropna=False,as_index=False).agg(total_tickets=('order_num','nunique'))
tbl_state_redeliveries = tbl_state_redeliveries.merge(_tx_by_state, on='state', how='left')
tbl_state_redeliveries['redel_pct_of_tickets'] = (tbl_state_redeliveries['redelivery_count']/tbl_state_redeliveries['total_tickets'].replace(0,np.nan)*100).round(2)

# Metro scope (only when metro is non-null)
if len(tbl_metro_redeliveries) > 0:
    _tx_by_metro = df_tx[df_tx['metro'].notna()].groupby('metro',dropna=False,as_index=False).agg(total_tickets=('order_num','nunique'))
    tbl_metro_redeliveries = tbl_metro_redeliveries.merge(_tx_by_metro, on='metro', how='left')
    tbl_metro_redeliveries['redel_pct_of_tickets'] = (tbl_metro_redeliveries['redelivery_count']/tbl_metro_redeliveries['total_tickets'].replace(0,np.nan)*100).round(2)

# VP scope
_tx_by_vp = df_tx.groupby('vp',dropna=False,as_index=False).agg(total_tickets=('order_num','nunique'))
_redel_by_vp = (redel_linked.groupby('vp',dropna=False,as_index=False)
                .agg(redelivery_count=('event_key','nunique'),
                     redelivery_items=('product','count'),
                     distinct_warehouses=('tech_warehouse','nunique'),
                     affected_techs=('_tech_full','nunique')))
tbl_vp_redeliveries = _redel_by_vp.merge(_tx_by_vp, on='vp', how='left').sort_values('redelivery_count',ascending=False)
tbl_vp_redeliveries['redel_pct_of_tickets'] = (tbl_vp_redeliveries['redelivery_count']/tbl_vp_redeliveries['total_tickets'].replace(0,np.nan)*100).round(2)

# Total scope
_total_redel  = int(redel_linked['event_key'].nunique())
_total_items  = int(redel_linked['product'].count())
_total_whs    = int(redel_linked['tech_warehouse'].nunique())
_total_techs  = int(redel_linked.loc[redel_linked['_tech_full']!='','_tech_full'].nunique())  # v1.33.0 (L1)
_total_tx_all = int(df_tx['order_num'].nunique())
tbl_total_redeliveries = pd.DataFrame([{
    'scope':'COMPANY TOTAL','redelivery_count':_total_redel,'redelivery_items':_total_items,
    'distinct_warehouses':_total_whs,'affected_techs':_total_techs,'total_tickets':_total_tx_all,
    'redel_pct_of_tickets':round(_total_redel/max(_total_tx_all,1)*100,2),
}])
print('\nCompany-total redelivery rate:')
print(tbl_total_redeliveries[['scope','redelivery_count','total_tickets','redel_pct_of_tickets']].to_string(index=False))
print('\nVP redelivery rates:')
print(tbl_vp_redeliveries[['vp','redelivery_count','total_tickets','redel_pct_of_tickets']].to_string(index=False))

# ── Lost equipment ────────────────────────────────────────────────────────────
df_lost_raw['lost_date']=pd.to_datetime(df_lost_raw['lost_date_raw'],errors='coerce')
df_lost_raw['asset_tag']=pd.to_numeric(df_lost_raw['asset_tag'],errors='coerce').astype('Int64')
df_lost_raw['asset_tag_str']=df_lost_raw['asset_tag'].astype(str).replace('<NA>',pd.NA)
df_lost_raw['state'], _lost_state_audit = resolve_state_series(
    df_lost_raw['tech_warehouse'], df_lost_raw['state'], df_hier)
_print_state_audit(_lost_state_audit, 'df_lost_raw')
df_lost_raw['metro']=df_lost_raw['tech_warehouse'].apply(assign_metro)

_lost_filt=df_lost_raw[
    df_lost_raw['lost_date'].notna()&
    (df_lost_raw['lost_date']>=pd.Timestamp(FILTER_START))&
    (df_lost_raw['lost_date']<=pd.Timestamp(FILTER_END))
].copy()
print(f'\nLost equipment in window ({FILTER_START} to {FILTER_END}): {len(_lost_filt):,}')
if len(_lost_filt)==0:
    print('  *** WARNING: zero lost-equipment rows in window.')
    print(f'      df_lost_raw has {len(df_lost_raw):,} rows total but NONE fall between')
    print(f'      {FILTER_START} and {FILTER_END}. Lost-equipment sheets will be empty.')
    print(f'      Likely cause: ATI.Lost_Date feed has not been updated, or FILTER_START')
    print(f'      is set later than any lost event in the table.')
_lost_filt['lost_year'] =_lost_filt['lost_date'].dt.year.fillna(0).astype(int)
_lost_filt['lost_month']=_lost_filt['lost_date'].dt.month.fillna(0).astype(int)
_lost_filt['lost_cost_last_price']=pd.to_numeric(_lost_filt['lost_cost_last_price'],errors='coerce').fillna(0.0)

tbl_lost_by_wh=(_lost_filt.groupby(['tech_warehouse','region','vp','state','metro','lost_year','lost_month'],dropna=False).agg(lost_asset_count=('asset_tag','nunique'),lost_asset_cost=('lost_cost_last_price','sum'),lost_product_types=('product_name','nunique'),lost_products_list=('product_name',lambda x:' | '.join(sorted(x.dropna().unique())))).reset_index().sort_values('lost_asset_cost',ascending=False))
tbl_lost_by_wh=tbl_lost_by_wh.merge(df_adc[['warehouse','yr','mo','adc','pt_days']],left_on=['tech_warehouse','lost_year','lost_month'],right_on=['warehouse','yr','mo'],how='left').drop(columns=['warehouse','yr','mo'],errors='ignore')
tbl_lost_by_wh['lost_cost_per_adc']   =(tbl_lost_by_wh['lost_asset_cost']/tbl_lost_by_wh['adc'].replace(0,np.nan)).round(2)
tbl_lost_by_wh['lost_cost_per_pt_day']=(tbl_lost_by_wh['lost_asset_cost']/tbl_lost_by_wh['pt_days'].replace(0,np.nan)).round(4)

_lost_filt['_bid_str']=_lost_filt['bill_to_id'].astype(str).str.strip()
_lost_patient_ids=set(_lost_filt['_bid_str'].replace('',pd.NA).replace('nan',pd.NA).replace('None',pd.NA).replace('<NA>',pd.NA).replace('0',pd.NA).dropna().unique())
print(f'Unique patients in lost-asset set: {len(_lost_patient_ids):,}')

tbl_lost_tech_attribution=pd.DataFrame()
if len(_lost_patient_ids)>0:
    _tx_for_lost=df_tx[df_tx['record_id'].astype(str).str.strip().isin(_lost_patient_ids)].copy()
    _tx_for_lost['completed_date']=pd.to_datetime(_tx_for_lost['completed_date'],errors='coerce')
    _delivery_tx=_tx_for_lost[_tx_for_lost['ticket_type']=='Delivery'].copy()
    _last_delivery=(_delivery_tx.sort_values('completed_date',ascending=False).groupby('record_id',as_index=False).first()[['record_id','order_num','completed_date','tech_warehouse','techfirstname','techlastname','reason']].rename(columns={'order_num':'last_delivery_order','completed_date':'last_delivery_date','tech_warehouse':'last_delivery_warehouse','techfirstname':'last_delivery_first','techlastname':'last_delivery_last','reason':'last_delivery_reason'})) if len(_delivery_tx)>0 else pd.DataFrame(columns=['record_id','last_delivery_order','last_delivery_date','last_delivery_warehouse','last_delivery_first','last_delivery_last','last_delivery_reason'])
    _pickup_tx_base=_tx_for_lost[_tx_for_lost['ticket_type'].isin(['Pickup','Exchange'])].copy()
    if len(_pickup_tx_base)>0:
        _pickup_tx_base['tech_full']=(_pickup_tx_base['techfirstname'].fillna('')+' '+_pickup_tx_base['techlastname'].fillna('')).str.strip()
        _pickup_summary=_pickup_tx_base.groupby('record_id',as_index=False).agg(pickup_ticket_count=('order_num','nunique'),pickup_orders=('order_num',lambda x:' | '.join(sorted(x.dropna().astype(str).unique()))),pickup_techs=('tech_full',lambda x:', '.join(sorted(x.dropna().unique()))),earliest_pickup_date=('completed_date','min'),latest_pickup_date=('completed_date','max'))
    else:
        _pickup_summary=pd.DataFrame(columns=['record_id','pickup_ticket_count','pickup_orders','pickup_techs','earliest_pickup_date','latest_pickup_date'])
    tbl_lost_tech_attribution=(_lost_filt.assign(record_id=lambda d:d['_bid_str']).merge(_last_delivery,on='record_id',how='left').merge(_pickup_summary,on='record_id',how='left'))
    tbl_lost_tech_attribution['no_pickup_found']=(tbl_lost_tech_attribution['pickup_ticket_count'].isna()|(tbl_lost_tech_attribution['pickup_ticket_count']==0))
    _lost_for_pickup_join = _lost_filt[['asset_tag','_bid_str','lost_date']].rename(
        columns={'_bid_str':'record_id'}).copy()
    _lost_for_pickup_join['record_id'] = _lost_for_pickup_join['record_id'].astype(str).str.strip()
    _pk_only = _pickup_tx_base[_pickup_tx_base['ticket_type']=='Pickup'].copy() if len(_pickup_tx_base)>0 else _pickup_tx_base.copy()
    if len(_pk_only) > 0 and len(_lost_for_pickup_join) > 0:
        _pk_only = _pk_only[['order_num','record_id','techfirstname','techlastname',
                             'tech_warehouse','completed_date','reason']].copy()
        _pk_only['record_id'] = _pk_only['record_id'].astype(str).str.strip()
        _pk_xjoin = _lost_for_pickup_join.merge(_pk_only, on='record_id', how='inner')
        _pk_xjoin = _pk_xjoin[_pk_xjoin['completed_date'] <= _pk_xjoin['lost_date']].copy()
        if len(_pk_xjoin) > 0:
            _last_expected_pickup = (_pk_xjoin.sort_values('completed_date', ascending=False)
                                     .drop_duplicates(subset=['asset_tag'], keep='first')
                                     [['asset_tag','order_num','completed_date',
                                       'techfirstname','techlastname','tech_warehouse','reason','lost_date']]
                                     .rename(columns={
                                         'order_num':'pickup_last_order',
                                         'completed_date':'pickup_last_date',
                                         'techfirstname':'pickup_last_first',
                                         'techlastname':'pickup_last_last',
                                         'tech_warehouse':'pickup_last_warehouse',
                                         'reason':'pickup_last_reason'}))
            _last_expected_pickup['days_pickup_to_lost'] = (
                _last_expected_pickup['lost_date'] - _last_expected_pickup['pickup_last_date']).dt.days
            _last_expected_pickup = _last_expected_pickup.drop(columns=['lost_date'])
            tbl_lost_tech_attribution = tbl_lost_tech_attribution.merge(
                _last_expected_pickup, on='asset_tag', how='left')
            print(f'Lost-asset → last expected pickup: {_last_expected_pickup["pickup_last_order"].notna().sum():,} '
                  f'assets linked ({_last_expected_pickup["pickup_last_order"].notna().sum()/max(1,len(tbl_lost_tech_attribution))*100:.1f}% of lost assets)')
        else:
            print('Lost-asset → last expected pickup: 0 assets linked (no pickups precede any lost_date).')
    else:
        print('Lost-asset → last expected pickup: skipped (no pickup tickets in scope).')
    for _col in ['pickup_last_order','pickup_last_date','pickup_last_first','pickup_last_last',
                 'pickup_last_warehouse','pickup_last_reason','days_pickup_to_lost']:
        if _col not in tbl_lost_tech_attribution.columns:
            tbl_lost_tech_attribution[_col] = pd.NA
    tbl_lost_tech_attribution['no_prior_pickup_found'] = tbl_lost_tech_attribution['pickup_last_order'].isna()
    _lta_cols=['asset_tag_str','asset_tag','product_name','tech_warehouse','region','vp','state','metro','lost_date_raw','lost_cost_last_price','record_id','last_delivery_order','last_delivery_date','last_delivery_first','last_delivery_last','last_delivery_warehouse','last_delivery_reason','pickup_ticket_count','no_pickup_found','pickup_techs','pickup_orders','earliest_pickup_date','latest_pickup_date',
        'pickup_last_order','pickup_last_date','pickup_last_first','pickup_last_last',
        'pickup_last_warehouse','pickup_last_reason','days_pickup_to_lost','no_prior_pickup_found']
    tbl_lost_tech_attribution=tbl_lost_tech_attribution[[c for c in _lta_cols if c in tbl_lost_tech_attribution.columns]].sort_values(['tech_warehouse','lost_cost_last_price'],ascending=[True,False]).reset_index(drop=True)
    print(f'Lost attribution: {len(tbl_lost_tech_attribution):,} rows')

_prod_wkday_summary=(rollup_tech[rollup_tech['schedule_period']=='Weekday'].groupby(['techfirstname','techlastname','tech_warehouse','region','vp','state','metro'],dropna=False).agg(total_visits=('total_visits','sum'),total_tickets=('total_tickets','sum'),total_workhours=('total_workhours','sum'),total_weekday_days=('weekday_days','sum')).reset_index())
_prod_wkday_summary['avg_daily_tickets']=(_prod_wkday_summary['total_tickets']/_prod_wkday_summary['total_weekday_days'].replace(0,np.nan)).round(3)
# v1.33.0 (H1): merge keyed on (tech, warehouse). The name-only merge fanned out
# multi-warehouse techs (duplicate name keys on the right) and double-counted
# redeliveries — same fan-out class as the _raw_cts bug fixed in v1.31.
tbl_tech_accountability=_prod_wkday_summary.merge(tbl_tech_redeliveries[['techfirstname','techlastname','tech_warehouse','redelivery_count','redelivery_items','redel_per_100_visits']],on=['techfirstname','techlastname','tech_warehouse'],how='left')
assert len(tbl_tech_accountability)==len(_prod_wkday_summary), 'H1 regression: accountability merge multiplied rows'
tbl_tech_accountability[['redelivery_count','redelivery_items']]=tbl_tech_accountability[['redelivery_count','redelivery_items']].fillna(0).astype(int)
tbl_tech_accountability['redel_per_100_visits']=tbl_tech_accountability['redel_per_100_visits'].fillna(0)
tbl_tech_accountability=tbl_tech_accountability.sort_values(['avg_daily_tickets','redelivery_count'],ascending=[False,False]).reset_index(drop=True)
print(f'Tech accountability: {len(tbl_tech_accountability):,} rows')

_inv = df_inventory_total.merge(df_hier.rename(columns={'warehouse':'tech_warehouse'}),
                                on='tech_warehouse', how='left')
_inv['metro'] = _inv['tech_warehouse'].apply(assign_metro)
_inv['state'], _inv_state_audit = resolve_state_series(_inv['tech_warehouse'], _inv['state'], df_hier)
_print_state_audit(_inv_state_audit, '_inv')

tbl_inventory_by_wh    = _inv[['tech_warehouse','region','vp','state','metro','total_inventory_count','total_inventory_amount']].copy()
tbl_inventory_by_state = _inv.groupby('state',dropna=False,as_index=False).agg(
    total_inventory_count=('total_inventory_count','sum'),
    total_inventory_amount=('total_inventory_amount','sum'))
tbl_inventory_by_metro = (_inv[_inv['metro'].notna()].groupby('metro',dropna=False,as_index=False).agg(
    total_inventory_count=('total_inventory_count','sum'),
    total_inventory_amount=('total_inventory_amount','sum'))
    if _inv['metro'].notna().any()
    else pd.DataFrame(columns=['metro','total_inventory_count','total_inventory_amount']))
tbl_inventory_by_vp    = _inv.groupby('vp',dropna=False,as_index=False).agg(
    total_inventory_count=('total_inventory_count','sum'),
    total_inventory_amount=('total_inventory_amount','sum'))
_inv_total_count  = int(_inv['total_inventory_count'].sum())
_inv_total_amount = float(_inv['total_inventory_amount'].sum())

tbl_lost_by_wh = tbl_lost_by_wh.merge(
    tbl_inventory_by_wh[['tech_warehouse','total_inventory_count','total_inventory_amount']],
    on='tech_warehouse', how='left')
tbl_lost_by_wh['lost_amount_pct_of_inventory'] = (
    tbl_lost_by_wh['lost_asset_cost'] / tbl_lost_by_wh['total_inventory_amount'].replace(0,np.nan) * 100).round(2)
tbl_lost_by_wh['lost_count_pct_of_inventory'] = (
    tbl_lost_by_wh['lost_asset_count'] / tbl_lost_by_wh['total_inventory_count'].replace(0,np.nan) * 100).round(2)

print(f'Inventory denominator: {_inv_total_count:,} tagged assets / ${_inv_total_amount:,.0f} value across {len(tbl_inventory_by_wh):,} warehouse(s).')

## Cell 17b — MONTHLY LOST EQUIPMENT BY WAREHOUSE & STATE/METRO (v1.25.0)

Produces three monthly lost-equipment tables parallel to the redelivery tables in Cell 16b:
- `tbl_lost_monthly_wh`    — by warehouse × month
- `tbl_lost_monthly_state` — by state × month
- `tbl_lost_monthly_metro` — by metro × month (metro-matched only)

In [ ]:
# ── Monthly lost equipment by warehouse ──────────────────────────────────────
tbl_lost_monthly_wh = (
    _lost_filt
    .groupby(['tech_warehouse','region','vp','state','metro','lost_year','lost_month'], dropna=False, as_index=False)
    .agg(
        lost_asset_count  = ('asset_tag',            'nunique'),
        lost_asset_cost   = ('lost_cost_last_price',  'sum'),
        lost_product_types= ('product_name',          'nunique'),
    )
    .rename(columns={'lost_year':'delivery_year','lost_month':'delivery_month'})
    .sort_values(['tech_warehouse','delivery_year','delivery_month'])
)
tbl_lost_monthly_wh['period'] = (
    tbl_lost_monthly_wh['delivery_year'].astype(int).astype(str) + '-' +
    tbl_lost_monthly_wh['delivery_month'].astype(int).astype(str).str.zfill(2)
)

# ── Monthly lost equipment by state ─────────────────────────────────────────
tbl_lost_monthly_state = (
    _lost_filt
    .groupby(['state','lost_year','lost_month'], dropna=False, as_index=False)
    .agg(
        lost_asset_count    = ('asset_tag',            'nunique'),
        lost_asset_cost     = ('lost_cost_last_price',  'sum'),
        distinct_warehouses = ('tech_warehouse',         'nunique'),
        lost_product_types  = ('product_name',           'nunique'),
    )
    .rename(columns={'lost_year':'delivery_year','lost_month':'delivery_month'})
    .sort_values(['state','delivery_year','delivery_month'])
)
tbl_lost_monthly_state['period'] = (
    tbl_lost_monthly_state['delivery_year'].astype(int).astype(str) + '-' +
    tbl_lost_monthly_state['delivery_month'].astype(int).astype(str).str.zfill(2)
)

# ── Monthly lost equipment by metro ─────────────────────────────────────────
_lost_filt_metro = _lost_filt[_lost_filt['metro'].notna()].copy()
if len(_lost_filt_metro) > 0:
    tbl_lost_monthly_metro = (
        _lost_filt_metro
        .groupby(['metro','state','lost_year','lost_month'], dropna=False, as_index=False)
        .agg(
            lost_asset_count    = ('asset_tag',            'nunique'),
            lost_asset_cost     = ('lost_cost_last_price',  'sum'),
            distinct_warehouses = ('tech_warehouse',         'nunique'),
            lost_product_types  = ('product_name',           'nunique'),
        )
        .rename(columns={'lost_year':'delivery_year','lost_month':'delivery_month'})
        .sort_values(['metro','delivery_year','delivery_month'])
    )
    tbl_lost_monthly_metro['period'] = (
        tbl_lost_monthly_metro['delivery_year'].astype(int).astype(str) + '-' +
        tbl_lost_monthly_metro['delivery_month'].astype(int).astype(str).str.zfill(2)
    )
else:
    tbl_lost_monthly_metro = pd.DataFrame()

# ── State and metro summary totals ───────────────────────────────────────────
tbl_lost_by_state = (
    _lost_filt.groupby('state', dropna=False, as_index=False)
    .agg(lost_asset_count=('asset_tag','nunique'), lost_asset_cost=('lost_cost_last_price','sum'),
         distinct_warehouses=('tech_warehouse','nunique'), distinct_products=('product_name','nunique'))
    .sort_values('lost_asset_cost', ascending=False)
)
tbl_lost_by_metro = pd.DataFrame()
if len(_lost_filt_metro) > 0:
    tbl_lost_by_metro = (
        _lost_filt_metro.groupby(['metro','state'], dropna=False, as_index=False)
        .agg(lost_asset_count=('asset_tag','nunique'), lost_asset_cost=('lost_cost_last_price','sum'),
             distinct_warehouses=('tech_warehouse','nunique'), distinct_products=('product_name','nunique'))
        .sort_values('lost_asset_cost', ascending=False)
    )

print(f'Lost monthly by WH:    {len(tbl_lost_monthly_wh):,} rows')
print(f'Lost monthly by state: {len(tbl_lost_monthly_state):,} rows')
print(f'Lost monthly by metro: {len(tbl_lost_monthly_metro):,} rows')
print('\nState lost cost totals:')
print(tbl_lost_by_state[['state','lost_asset_count','lost_asset_cost']].to_string(index=False))
if len(tbl_lost_by_metro) > 0:
    print('\nMetro lost cost totals:')
    print(tbl_lost_by_metro.to_string(index=False))

def _add_lost_rates(tbl, inv_tbl, on_cols):
    """Merge inventory denominators and compute both lost-rate metrics.

    Pattern is identical at every scope, so factored out. Keeps the rate
    columns consistent and the merge logic in one place — easier to audit
    and harder to introduce a typo in one variant only.
    """
    tbl = tbl.merge(inv_tbl[on_cols + ['total_inventory_count','total_inventory_amount']],
                    on=on_cols, how='left')
    tbl['lost_amount_pct_of_inventory'] = (
        tbl['lost_asset_cost'] / tbl['total_inventory_amount'].replace(0,np.nan) * 100).round(2)
    tbl['lost_count_pct_of_inventory'] = (
        tbl['lost_asset_count'] / tbl['total_inventory_count'].replace(0,np.nan) * 100).round(2)
    return tbl

# Monthly tables
tbl_lost_monthly_wh    = _add_lost_rates(tbl_lost_monthly_wh,    tbl_inventory_by_wh,    ['tech_warehouse'])
tbl_lost_monthly_state = _add_lost_rates(tbl_lost_monthly_state, tbl_inventory_by_state, ['state'])
if len(tbl_lost_monthly_metro) > 0 and len(tbl_inventory_by_metro) > 0:
    tbl_lost_monthly_metro = _add_lost_rates(tbl_lost_monthly_metro, tbl_inventory_by_metro, ['metro'])

# Period-totals (state, metro) — already exist; add rates
tbl_lost_by_state = _add_lost_rates(tbl_lost_by_state, tbl_inventory_by_state, ['state'])
if len(tbl_lost_by_metro) > 0 and len(tbl_inventory_by_metro) > 0:
    tbl_lost_by_metro = _add_lost_rates(tbl_lost_by_metro, tbl_inventory_by_metro, ['metro'])

tbl_lost_by_vp = (_lost_filt.groupby('vp',dropna=False,as_index=False)
                  .agg(lost_asset_count=('asset_tag','nunique'),
                       lost_asset_cost=('lost_cost_last_price','sum'),
                       distinct_warehouses=('tech_warehouse','nunique'),
                       distinct_products=('product_name','nunique')))
tbl_lost_by_vp = _add_lost_rates(tbl_lost_by_vp, tbl_inventory_by_vp, ['vp']).sort_values('lost_asset_cost', ascending=False)

# company total — both denominators, both rates
_lost_total_count = int(_lost_filt['asset_tag'].nunique())
_lost_total_cost  = float(_lost_filt['lost_cost_last_price'].sum())
tbl_lost_total = pd.DataFrame([{
    'scope':'COMPANY TOTAL',
    'lost_asset_count':_lost_total_count,
    'lost_asset_cost':_lost_total_cost,
    'distinct_warehouses':int(_lost_filt['tech_warehouse'].nunique()),
    'distinct_products':int(_lost_filt['product_name'].nunique()),
    'total_inventory_count':_inv_total_count,
    'total_inventory_amount':_inv_total_amount,
    'lost_amount_pct_of_inventory':round(_lost_total_cost / max(_inv_total_amount,1.0) * 100, 2),
    'lost_count_pct_of_inventory': round(_lost_total_count / max(_inv_total_count,1) * 100, 2),
}])

print('\nCompany-total lost rates:')
print(tbl_lost_total[['scope','lost_asset_count','lost_asset_cost',
                      'total_inventory_count','total_inventory_amount',
                      'lost_amount_pct_of_inventory','lost_count_pct_of_inventory']].to_string(index=False))
print('\nVP lost rates:')
print(tbl_lost_by_vp[['vp','lost_asset_count','lost_asset_cost',
                      'total_inventory_count','total_inventory_amount',
                      'lost_amount_pct_of_inventory','lost_count_pct_of_inventory']].to_string(index=False))

## Cell 17c — LOST-RECOVERY SCORECARD: THREE DEFINITIONS (v1.33.0)

Per confirmed design decision (2026-07-31), lost items are counted against a tech
under **three separate definitions**, exported side by side:

| Column | Definition | Attributed to |
|---|---|---|
| `lost_all_items` | Every ATI.Lost item in window | Last-delivery tech (proximity) |
| `lost_no_pickup_items` | Lost with **no completed pickup before the loss date** — the recovery never happened | Last-delivery tech |
| `lost_after_pickup_items` | Lost **despite** a completed pickup before the loss date — picked up but still written off | Tech who performed that pickup |

All three are **proximity attribution, not evidence of fault** — a tech serving
high-turnover patients accrues attributions mechanically. Use for pattern
detection and coaching conversations, not discipline, without ticket-level review.


In [ ]:
# v1.33.0 (N2) — three lost-recovery definitions per tech (summary + monthly)
tbl_tech_lost_scorecard   = pd.DataFrame()
tbl_tech_lost_monthly_defs = pd.DataFrame()

if len(tbl_lost_tech_attribution) > 0:
    _lsc = tbl_lost_tech_attribution.copy()
    _lsc['_ldt'] = pd.to_datetime(_lsc['lost_date_raw'], errors='coerce')
    _lsc['delivery_year']  = _lsc['_ldt'].dt.year.astype('Int64')   # NOTE: month the item
    _lsc['delivery_month'] = _lsc['_ldt'].dt.month.astype('Int64')  # was MARKED lost
    _lsc['lost_cost_last_price'] = pd.to_numeric(_lsc['lost_cost_last_price'], errors='coerce').fillna(0.0)
    # no_prior_pickup_found may be object dtype (pd.NA path) — normalize to bool.
    _lsc['no_prior_pickup_found'] = _lsc['no_prior_pickup_found'].fillna(True).astype(bool)

    def _agg_def(df, fn_col, ln_col, item_col, cost_col, monthly):
        """One definition's aggregation. monthly=False -> all-period grain."""
        keys = [fn_col, ln_col] + (['delivery_year','delivery_month'] if monthly else [])
        out = (df[df[fn_col].notna() & df[fn_col].astype(str).str.strip().ne('')]
               .groupby(keys, dropna=False, as_index=False)
               .agg(**{item_col: ('asset_tag','nunique'), cost_col: ('lost_cost_last_price','sum')})
               .rename(columns={fn_col:'techfirstname', ln_col:'techlastname'}))
        return out

    def _build_lost_defs(monthly):
        # A: all lost -> last-delivery tech
        _a = _agg_def(_lsc, 'last_delivery_first','last_delivery_last',
                      'lost_all_items','lost_all_cost', monthly)
        # B: no completed pickup before loss -> last-delivery tech (recovery never happened)
        _b = _agg_def(_lsc[_lsc['no_prior_pickup_found']], 'last_delivery_first','last_delivery_last',
                      'lost_no_pickup_items','lost_no_pickup_cost', monthly)
        # C: pickup completed before loss, item still lost -> pickup tech
        _c = _agg_def(_lsc[~_lsc['no_prior_pickup_found']], 'pickup_last_first','pickup_last_last',
                      'lost_after_pickup_items','lost_after_pickup_cost', monthly)
        keys = ['techfirstname','techlastname'] + (['delivery_year','delivery_month'] if monthly else [])
        out = _a.merge(_b, on=keys, how='outer').merge(_c, on=keys, how='outer')
        for c in ['lost_all_items','lost_no_pickup_items','lost_after_pickup_items']:
            out[c] = out[c].fillna(0).astype(int)
        for c in ['lost_all_cost','lost_no_pickup_cost','lost_after_pickup_cost']:
            out[c] = out[c].fillna(0.0).round(2)
        return out

    tbl_tech_lost_scorecard    = _build_lost_defs(monthly=False).sort_values('lost_all_items', ascending=False).reset_index(drop=True)
    tbl_tech_lost_monthly_defs = _build_lost_defs(monthly=True)

    # Reconciliation: definitions B and C partition definition A ONLY within the
    # attributed subset (items with a last-delivery tech). Items lost with no
    # attributable delivery at all appear in none of the three per-tech columns.
    _n_all   = int(_lsc['asset_tag'].nunique())
    _n_attrA = int(_lsc.loc[_lsc['last_delivery_first'].notna(),'asset_tag'].nunique())
    print(f'Lost scorecard: {len(tbl_tech_lost_scorecard):,} techs | '
          f'{_n_attrA:,} of {_n_all:,} lost assets have a last-delivery attribution '
          f'({_n_all-_n_attrA:,} unattributable — visible in Lost_Tech_Attribution, not per-tech columns).')
else:
    print('Lost scorecard: skipped — tbl_lost_tech_attribution is empty.')


## Cell 18 — LOST EQUIPMENT VISUALIZATIONS (D1/D2/D3)

In [ ]:
tbl_lost_tech_summary = pd.DataFrame()

MIN_INVENTORY_FOR_RATE = 5000  # $ floor: avoid tiny warehouses dominating the rate ranking
_d1 = (tbl_lost_by_wh.groupby(['tech_warehouse','region','vp','state','metro'], as_index=False)
       .agg(lost_asset_count=('lost_asset_count','sum'),
            lost_asset_cost=('lost_asset_cost','sum')))
_d1 = _d1.merge(tbl_inventory_by_wh[['tech_warehouse','total_inventory_amount','total_inventory_count']],
                on='tech_warehouse', how='left')
_d1['lost_amount_pct_of_inventory'] = (_d1['lost_asset_cost']
    / _d1['total_inventory_amount'].replace(0,np.nan) * 100).round(3)
_d1_ranked = _d1[_d1['total_inventory_amount'].fillna(0) >= MIN_INVENTORY_FOR_RATE].copy()
_d1_ranked = _d1_ranked.dropna(subset=['lost_amount_pct_of_inventory']).sort_values(
    'lost_amount_pct_of_inventory', ascending=False)
if len(_d1_ranked)>0:
    _d1p = _d1_ranked.head(20)
    fig,ax = plt.subplots(figsize=(12,max(5,len(_d1p)*0.45)))
    ax.barh(_d1p['tech_warehouse'][::-1], _d1p['lost_amount_pct_of_inventory'][::-1],
            color=PALETTE['redelivery'], alpha=0.85)
    _xmax = float(_d1p['lost_amount_pct_of_inventory'].max())
    for bar,(_,row) in zip(ax.patches, _d1p[::-1].iterrows()):
        ax.text(bar.get_width()+_xmax*0.01, bar.get_y()+bar.get_height()/2,
                f"{row['lost_amount_pct_of_inventory']:.2f}%  (${row['lost_asset_cost']:,.0f} of ${row['total_inventory_amount']:,.0f} | {int(row['lost_asset_count'])} items)",
                va='center', ha='left', fontsize=8)
    ax.set_title(f'D1: Top 20 Warehouses by Lost Cost as % of Inventory Value (min ${MIN_INVENTORY_FOR_RATE:,} inventory)',
                 fontsize=11, fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.2f}%'))
    ax.set_xlim(right=_xmax*1.55); ax.grid(axis='x',linestyle='--',alpha=0.4); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'D1_Lost_Cost_By_WH'); plt.show(); plt.close(fig)

# D1b: Lost cost by state — ranked by % of inventory value
if len(tbl_lost_by_state)>0 and 'lost_amount_pct_of_inventory' in tbl_lost_by_state.columns:
    _d1b = tbl_lost_by_state.dropna(subset=['lost_amount_pct_of_inventory']).sort_values(
        'lost_amount_pct_of_inventory', ascending=True)
    fig,ax = plt.subplots(figsize=(11,max(4,len(_d1b)*0.5)))
    ax.barh(_d1b['state'], _d1b['lost_amount_pct_of_inventory'], color='#17becf', alpha=0.85)
    _xmax = float(_d1b['lost_amount_pct_of_inventory'].max())
    for bar,(_,row) in zip(ax.patches, _d1b.iterrows()):
        _inv_amt = row.get('total_inventory_amount', np.nan)
        _inv_str = f"${_inv_amt:,.0f}" if pd.notna(_inv_amt) else 'n/a'
        ax.text(bar.get_width()+_xmax*0.01, bar.get_y()+bar.get_height()/2,
                f"{row['lost_amount_pct_of_inventory']:.2f}%  (${row['lost_asset_cost']:,.0f} of {_inv_str} | {int(row['lost_asset_count'])} items)",
                va='center', ha='left', fontsize=9)
    ax.set_title('D1b: State Lost Cost as % of Inventory Value', fontsize=12, fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.2f}%'))
    ax.set_xlim(right=_xmax*1.55 if _xmax>0 else 1); ax.grid(axis='x',linestyle='--',alpha=0.4); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'D1b_Lost_Cost_By_State'); plt.show(); plt.close(fig)

# D1c: Lost cost by metro — ranked by % of inventory value
if len(tbl_lost_by_metro)>0 and 'lost_amount_pct_of_inventory' in tbl_lost_by_metro.columns:
    _d1c = tbl_lost_by_metro.dropna(subset=['lost_amount_pct_of_inventory']).sort_values(
        'lost_amount_pct_of_inventory', ascending=True)
    fig,ax = plt.subplots(figsize=(10,max(3,len(_d1c)*0.6)))
    ax.barh(_d1c['metro'], _d1c['lost_amount_pct_of_inventory'], color='#bcbd22', alpha=0.85)
    _xmax = float(_d1c['lost_amount_pct_of_inventory'].max())
    for bar,(_,row) in zip(ax.patches, _d1c.iterrows()):
        _inv_amt = row.get('total_inventory_amount', np.nan)
        _inv_str = f"${_inv_amt:,.0f}" if pd.notna(_inv_amt) else 'n/a'
        ax.text(bar.get_width()+_xmax*0.01, bar.get_y()+bar.get_height()/2,
                f"{row['lost_amount_pct_of_inventory']:.2f}%  (${row['lost_asset_cost']:,.0f} of {_inv_str} | {int(row['lost_asset_count'])} items)",
                va='center', ha='left', fontsize=9)
    ax.set_title('D1c: Metro Lost Cost as % of Inventory Value', fontsize=12, fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.2f}%'))
    ax.set_xlim(right=_xmax*1.55 if _xmax>0 else 1); ax.grid(axis='x',linestyle='--',alpha=0.4); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'D1c_Lost_Cost_By_Metro'); plt.show(); plt.close(fig)

# D2: Lost items by product type
_d2=_lost_filt.groupby('product_name',as_index=False).agg(lost_count=('asset_tag','nunique'),lost_cost=('lost_cost_last_price','sum')).sort_values('lost_count',ascending=False).head(20)
if len(_d2)>0:
    fig,ax=plt.subplots(figsize=(10,max(5,len(_d2)*0.45)))
    ax.barh(_d2['product_name'][::-1],_d2['lost_count'][::-1],color=PALETTE['unmatched'],alpha=0.85)
    for i,(_,row) in enumerate(_d2[::-1].iterrows()):
        ax.text(row['lost_count']+_d2['lost_count'].max()*0.01,i,f"${row['lost_cost']:,.0f}",va='center',ha='left',fontsize=8)
    ax.set_title('D2: Lost Items by Product Type (Top 20)',fontsize=12,fontweight='bold')
    ax.set_xlim(right=_d2['lost_count'].max()*1.30); ax.grid(axis='x',linestyle='--',alpha=0.4); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'D2_Lost_Items_By_Product'); plt.show(); plt.close(fig)

# D3: Top techs by attribution
if len(tbl_lost_tech_attribution)>0:
    _lta_attrib=tbl_lost_tech_attribution[tbl_lost_tech_attribution['last_delivery_first'].notna()].copy()
    if len(_lta_attrib)>0:
        _lta_attrib['tech_name']=(_lta_attrib['last_delivery_first'].fillna('')+' '+_lta_attrib['last_delivery_last'].fillna('')).str.strip()
        tbl_lost_tech_summary=(_lta_attrib.groupby(['tech_name','last_delivery_warehouse'],as_index=False).agg(attributed_lost_items=('asset_tag','nunique'),attributed_lost_cost=('lost_cost_last_price','sum'),distinct_products=('product_name','nunique'),latest_attribution=('last_delivery_date','max')).rename(columns={'last_delivery_warehouse':'tech_warehouse'}).sort_values('attributed_lost_items',ascending=False).reset_index(drop=True))
        _d3=tbl_lost_tech_summary.head(20)
        fig,ax=plt.subplots(figsize=(10,max(5,len(_d3)*0.48)))
        _lbl=(_d3['tech_name']+' ('+_d3['tech_warehouse'].fillna('?')+')').tolist()
        ax.barh(_lbl[::-1],_d3['attributed_lost_items'][::-1],color=PALETTE['internal_ops'],alpha=0.85)
        for i,(_,row) in enumerate(_d3[::-1].iterrows()):
            ax.text(row['attributed_lost_items']+_d3['attributed_lost_items'].max()*0.01,i,f"${row['attributed_lost_cost']:,.0f}",va='center',ha='left',fontsize=8)
        ax.set_title('D3: Top Technicians by Lost Asset Attribution\n(last-delivery proximity — not evidence of fault)',fontsize=11,fontweight='bold')
        ax.set_xlim(right=_d3['attributed_lost_items'].max()*1.30); ax.grid(axis='x',linestyle='--',alpha=0.4); ax.set_axisbelow(True)
        fig.tight_layout(); save_fig(fig,'D3_Lost_Attribution_By_Tech'); plt.show(); plt.close(fig)
        print(f'tbl_lost_tech_summary: {len(tbl_lost_tech_summary):,} rows')

# ──  Tech-scope lost rate (per 100 deliveries) ──────────────────────
# PROXIMITY-based attribution, NOT causation: lost items are attributed to the
# tech who made the most recent delivery to that patient. A tech with high
# delivery volume to high-turnover patients will accrue more attributions
# without necessarily losing more equipment. Interpret with caution.
if len(tbl_lost_tech_summary) > 0:
    _tech_deliveries = (df_tx[(~df_tx['_unattributed']) & (df_tx['ticket_type']=='Delivery')]
                        .assign(tech_name=lambda d:(d['techfirstname'].fillna('')+' '+d['techlastname'].fillna('')).str.strip())
                        .groupby(['tech_name','tech_warehouse'],dropna=False,as_index=False)
                        .agg(total_deliveries=('order_num','nunique')))
    tbl_lost_tech_summary = tbl_lost_tech_summary.merge(_tech_deliveries,
                                                        on=['tech_name','tech_warehouse'], how='left')
    tbl_lost_tech_summary['lost_per_100_deliveries'] = (tbl_lost_tech_summary['attributed_lost_items']/tbl_lost_tech_summary['total_deliveries'].replace(0,np.nan)*100).round(2)
    print(f'Tech-scope lost rate added (per-100-deliveries, proximity attribution).')

## Cell 19 — STATE & METRO TREND CHARTS

- **B2:** State avg daily tickets per tech (monthly trend)
- **B3:** Metro avg daily tickets per tech (monthly trend)
- Both include the company average black reference line for benchmark context.

In [ ]:
def _plot_series_with_partial(ax, xs, ys, partial_flags, color, label, lw=2, ms=6, marker_solid='o', marker_partial='o', zorder=2):
    """v1.30.3 — Plot a series where partial-month points get an open (hollow) marker
    and a '*' annotation suffix. Solid points use filled markers."""
    if len(xs)==0: return
    # Connected line over everything
    ax.plot(xs, ys, color=color, linewidth=lw, label=label, zorder=zorder)
    # Solid markers for complete months
    _solid_mask = ~partial_flags
    if _solid_mask.any():
        ax.plot(xs[_solid_mask], ys[_solid_mask], linestyle='None', marker=marker_solid,
                markersize=ms, color=color, zorder=zorder+1)
    # Open markers for partial month
    _partial_mask = partial_flags
    if _partial_mask.any():
        ax.plot(xs[_partial_mask], ys[_partial_mask], linestyle='None', marker=marker_partial,
                markersize=ms+1, markerfacecolor='white', markeredgecolor=color,
                markeredgewidth=1.8, zorder=zorder+1)
        
def _plot_geo_trend(tbl, key_col, title, save_name, cmap_name='tab10'):
    """Generic trend chart for any geographic grouping (state, metro, VP).

    v1.30.3 — Plots mtd_avg_daily_tickets_per_tech (equals legacy for past months,
    uses weekday_days_elapsed denominator for current partial month). Partial-month
    points use open (hollow) markers + asterisk label.
    """
    if tbl is None or len(tbl) == 0:
        print(f'  {title}: no data — chart skipped.'); return
    _tbl = tbl.copy()  # v1.30.3 — include current month
    if len(_tbl) == 0:
        print(f'  {title}: no periods — skipped.'); return
    _keys   = sorted(_tbl[key_col].dropna().unique())
    _cmap   = plt.get_cmap(cmap_name)
    _colors = {k: _cmap(i % 10) for i, k in enumerate(_keys)}
    fig, ax = plt.subplots(figsize=(14, 6))
    _has_partial_any = False
    for k in _keys:
        _d = _tbl[_tbl[key_col] == k].sort_values('period')
        if _d.empty: continue
        _xs = pd.to_datetime(_d['period']+'-01').values
        _ys = _d['tickets_per_active_day_per_tech'].values  # v1.32.0
        _pf = _d['is_partial_month'].values if 'is_partial_month' in _d.columns else np.zeros(len(_d),dtype=bool)
        if _pf.any(): _has_partial_any = True
        _plot_series_with_partial(ax, _xs, _ys, _pf, _colors[k], str(k))
        for _px, _py, _pp in zip(_xs, _ys, _pf):
            if pd.notna(_py):
                _lbl = f'{_py:.2f}*' if _pp else f'{_py:.2f}'
                ax.annotate(_lbl, xy=(_px,_py), xytext=(0,6),
                    textcoords='offset points', fontsize=6.5, color=_colors[k], ha='center', va='bottom')
    # Company average reference line
    _grand = tbl_grand_tpt.sort_values('period').copy()  # v1.30.3
    _grand['_pd'] = pd.to_datetime(_grand['period']+'-01')
    if len(_grand) > 0 and _grand['tickets_per_active_day_per_tech'].notna().any():  # v1.32.0
        _gxs = _grand['_pd'].values
        _gys = _grand['tickets_per_active_day_per_tech'].values  # v1.32.0
        _gpf = _grand['is_partial_month'].values if 'is_partial_month' in _grand.columns else np.zeros(len(_grand),dtype=bool)
        if _gpf.any(): _has_partial_any = True
        _plot_series_with_partial(ax, _gxs, _gys, _gpf, PALETTE['company_avg'], 'Company Average',
                                  lw=2.5, ms=7, marker_solid='s', marker_partial='s', zorder=10)
        for _px, _py, _pp in zip(_gxs, _gys, _gpf):
            if pd.notna(_py):
                _lbl = f'{_py:.2f}*' if _pp else f'{_py:.2f}'
                ax.annotate(_lbl, xy=(_px,_py), xytext=(0,-12),
                    textcoords='offset points', fontsize=6.5, color=PALETTE['company_avg'],
                    ha='center', va='top', fontweight='bold')
    if _has_partial_any:
        _today_l4=AS_OF_DATE  # v1.30.4
        _ewd=_count_weekdays_elapsed(_today_l4.year,_today_l4.month); _fwd=_count_weekdays_full(_today_l4.year,_today_l4.month)
        ax.text(0.01,-0.18,f'* As of {_today_l4.isoformat()} (last full day); {_ewd} of {_fwd} weekdays elapsed. Open markers = partial month.',
                transform=ax.transAxes,fontsize=7.5,style='italic',color='#555')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Tickets per Active Tech-Day (v1.32.0)')
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.legend(loc='upper left', fontsize=8, ncol=2, framealpha=0.85)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
    fig.tight_layout(); save_fig(fig, save_name); plt.show(); plt.close(fig)

# B2: State trend
_plot_geo_trend(
    tbl_state_tpt, 'state',
    'B2: Tickets per Active Tech-Day by State (v1.32.0)',
    'B2_State_Daily_Tickets_Per_Tech'
)

# B3: Metro trend
_plot_geo_trend(
    tbl_metro_tpt if len(tbl_metro_tpt) > 0 else None,
    'metro',
    'B3: Tickets per Active Tech-Day by Metro — DFW / Houston / San Antonio (v1.32.0)',
    'B3_Metro_Daily_Tickets_Per_Tech'
)

# Warehouse chart helper (used in Cell 20)
def plot_wh_trends(wh_list, title, save_name):
    # v1.30.3 — MTD-aware; partial month gets open marker + asterisk
    _sub = tbl_wh_tpt[tbl_wh_tpt['tech_warehouse'].isin(wh_list)].sort_values(['tech_warehouse','delivery_year','delivery_month']).copy()
    if _sub.empty: print(f'No data: {wh_list}'); return
    _sub['plot_date'] = pd.to_datetime(_sub['period']+'-01')
    _whs   = _sub['tech_warehouse'].unique()
    _colors= {wh:plt.get_cmap('tab10')(i%10) for i,wh in enumerate(_whs)}
    _has_io= (len(tbl_intops_tpt_wh)>0 and tbl_intops_tpt_wh['tech_warehouse'].isin(wh_list).any())
    _has_partial_any = False
    fig,ax = plt.subplots(figsize=(13,5))
    for wh in _whs:
        _d = _sub[_sub['tech_warehouse']==wh].sort_values('plot_date').dropna(subset=['tickets_per_active_day_per_tech'])  # v1.32.0
        if _d.empty: continue
        _xs=_d['plot_date'].values; _ys=_d['tickets_per_active_day_per_tech'].values  # v1.32.0
        _pf=_d['is_partial_month'].values if 'is_partial_month' in _d.columns else np.zeros(len(_d),dtype=bool)
        if _pf.any(): _has_partial_any=True
        _plot_series_with_partial(ax,_xs,_ys,_pf,_colors[wh],wh,lw=2,ms=5)
        _last=_d.iloc[-1]; _lbl=f"{_last['tickets_per_active_day_per_tech']:.2f}{'*' if _last['is_partial_month'] else ''}"  # v1.32.0
        ax.annotate(_lbl,xy=(_last['plot_date'],_last['tickets_per_active_day_per_tech']),xytext=(5,2),textcoords='offset points',fontsize=7,color=_colors[wh])
    if _has_io:
        ax2=ax.twinx(); ax2.set_ylabel('Intops Daily (R)',color=PALETTE['internal_ops'],fontsize=9); ax2.tick_params(axis='y',labelcolor=PALETTE['internal_ops'])
        _io_sub=tbl_intops_tpt_wh[tbl_intops_tpt_wh['tech_warehouse'].isin(wh_list)].copy(); _io_sub['plot_date']=pd.to_datetime(_io_sub['period']+'-01')
        _io_max=_io_sub['intops_daily_tickets'].dropna().max(); _io_max=_io_max if (pd.notna(_io_max) and _io_max>0) else 1.0; ax2.set_ylim(0,_io_max*1.20)
        for wh in _whs:
            _io=_io_sub[_io_sub['tech_warehouse']==wh].sort_values('plot_date').dropna(subset=['intops_daily_tickets'])
            if _io.empty: continue
            ax2.plot(_io['plot_date'],_io['intops_daily_tickets'],marker='^',linewidth=1.5,markersize=5,linestyle='--',color=_colors[wh],alpha=0.65)
        h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1,l1,loc='upper left',fontsize=7,ncol=2); ax2.legend(h2,l2,loc='upper right',fontsize=7)
    else:
        ax.legend(loc='upper left',fontsize=7,ncol=2)
    ax.set_title(title,fontsize=13,fontweight='bold',pad=10); ax.set_xlabel('Month'); ax.set_ylabel('Tickets per Active Tech-Day (v1.32.0)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m')); ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(),rotation=45,ha='right',fontsize=8)
    ax.grid(axis='y',linestyle='--',alpha=0.3)
    if _has_partial_any:  # v1.30.3 footnote
        _today_l5=AS_OF_DATE  # v1.30.4
        _ewd5=_count_weekdays_elapsed(_today_l5.year,_today_l5.month); _fwd5=_count_weekdays_full(_today_l5.year,_today_l5.month)
        ax.text(0.01,-0.22,f'* As of {_today_l5.isoformat()} (last full day); {_ewd5} of {_fwd5} weekdays elapsed. Open markers = partial month.',transform=ax.transAxes,fontsize=7.5,style='italic',color='#555')
    fig.tight_layout(); save_fig(fig,save_name); plt.show(); plt.close(fig)

## Cell 20 — WAREHOUSE TREND CHARTS: TOP & BOTTOM PERFORMERS

In [ ]:
_top_whs=tbl_top_wh['tech_warehouse'].tolist(); _bot_whs=tbl_bot_wh['tech_warehouse'].tolist()
plot_wh_trends(_top_whs,f'Top {TOP_N} Warehouses — Avg Daily Tickets per Tech',f'wh_top{TOP_N}_trend')
plot_wh_trends(_bot_whs,f'Bottom {TOP_N} Warehouses — Avg Daily Tickets per Tech',f'wh_bot{TOP_N}_trend')

# Metro warehouse-level drill-downs
for _metro_name, _metro_patterns in METRO_GROUPS.items():
    _metro_whs = [
        wh for wh in tbl_wh_tpt['tech_warehouse'].unique()
        if any(p in str(wh).lower() for p in _metro_patterns)
    ]
    if _metro_whs:
        plot_wh_trends(
            _metro_whs,
            f'{_metro_name} Metro — Avg Daily Tickets per Tech',
            f'wh_{_metro_name.replace(" ","_")}_trend'
        )
    else:
        print(f'  {_metro_name}: no warehouses matched — chart skipped.')

_latest_period = (tbl_wh_tpt.dropna(subset=['mom_pct']).sort_values(['delivery_year','delivery_month'])['period'].iloc[-1] if not tbl_wh_tpt['mom_pct'].dropna().empty else None)
if _latest_period:
    # v1.30.6 — Two bugs fixed here:
    # (1) The merge with _wh_rank on tech_warehouse used to fan-out rows when _wh_rank
    #     had a warehouse listed multiple times (one per VP). Now _wh_rank is one
    #     row per warehouse, so the merge is 1:1. drop_duplicates is belt-and-suspenders.
    # (2) Gainers and Decliners both used .head(TOP_N) after opposite sorts. With a
    #     small universe (e.g. only 10 warehouses with non-null mom_pct in the latest
    #     period), top 10 gainers and top 10 decliners contained the same warehouses
    #     in opposite order. Now: sort descending once, take top and bottom disjoint.
    _latest_slice = (tbl_wh_tpt[tbl_wh_tpt['period']==_latest_period]
        .dropna(subset=['mom_pct'])
        .drop_duplicates(subset=['tech_warehouse'])  # defensive
        .merge(_wh_rank[['tech_warehouse']], on='tech_warehouse', how='inner')
        .sort_values('mom_pct', ascending=False)
        .reset_index(drop=True))
    _gain_df, _lose_df = _disjoint_top_bottom(_latest_slice, n=TOP_N, value_col='mom_pct')
    _gainers = _gain_df['tech_warehouse'].tolist()
    _losers  = _lose_df['tech_warehouse'].tolist()  # already sorted ascending by value_col
    print(f'  Gainers/decliners universe: {len(_latest_slice)} WH | gainers={len(_gainers)} | decliners={len(_losers)}')
    if _gainers and _losers:
        _overlap_gd = set(_gainers) & set(_losers)
        assert not _overlap_gd, f'BUG: gainers/decliners overlap = {_overlap_gd}'
    if _gainers: plot_wh_trends(_gainers, f'Top {len(_gainers)} MoM Gainers ({_latest_period})', f'wh_gainers_{_latest_period}')
    if _losers:  plot_wh_trends(_losers,  f'Top {len(_losers)} MoM Decliners ({_latest_period})', f'wh_decliners_{_latest_period}')

## Cell 21 — REDELIVERY VISUALIZATIONS (C1/C2/C3)

In [ ]:
# C1: Top products
_c1=tbl_redel_product.head(20)
fig,ax=plt.subplots(figsize=(10,max(6,len(_c1)*0.45)))
ax.barh(_c1['product'][::-1],_c1['redelivery_count'][::-1],color=PALETTE['redelivery'],alpha=0.85)
ax.set_xlabel('Redelivery Count'); ax.set_title('C1: Top 20 Products by Redelivery Count',fontsize=12,fontweight='bold')
ax.grid(axis='x',linestyle='--',alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout(); save_fig(fig,'C1_Redelivery_Top_Products'); plt.show(); plt.close(fig)

# C2: Top techs by redelivery % of tickets sorted by rate
# Excel table tbl_tech_redeliveries retains absolute-count sort; this chart only changes.
# MIN_TICKETS_FOR_RATE prevents techs with tiny denominators from dominating the rate ranking.
MIN_TICKETS_FOR_RATE = 50
_c2_src = tbl_tech_redeliveries[tbl_tech_redeliveries['total_tickets'].fillna(0) >= MIN_TICKETS_FOR_RATE].copy()
_c2 = _c2_src.sort_values('redel_pct_of_tickets', ascending=False).head(20).copy()
_c2['tech_name'] = _c2['techfirstname']+' '+_c2['techlastname']+' ('+_c2['tech_warehouse'].fillna('?')+')'
fig,ax = plt.subplots(figsize=(11,max(6,len(_c2)*0.45)))
ax.barh(_c2['tech_name'][::-1], _c2['redel_pct_of_tickets'][::-1], color=PALETTE['redelivery'], alpha=0.85)
# Annotate each bar with both % and absolute count
_xmax = float(_c2['redel_pct_of_tickets'].max()) if len(_c2)>0 else 0
for bar, (_, row) in zip(ax.patches, _c2[::-1].iterrows()):
    ax.text(bar.get_width()+_xmax*0.01, bar.get_y()+bar.get_height()/2,
            f"{row['redel_pct_of_tickets']:.2f}%  ({int(row['redelivery_count']):,} of {int(row['total_tickets']):,})",
            va='center', ha='left', fontsize=8)
ax.set_xlabel(f'Redeliveries as % of Total Tickets (min {MIN_TICKETS_FOR_RATE} tickets)')
ax.set_title('C2: Top 20 Technicians by Redelivery Rate', fontsize=12, fontweight='bold')
ax.set_xlim(right=_xmax*1.35 if _xmax>0 else 1)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.2f}%'))
ax.grid(axis='x', linestyle='--', alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout(); save_fig(fig,'C2_Redelivery_Top_Techs'); plt.show(); plt.close(fig)

# C3: Warehouse redelivery rate as % of total tickets
# Excel sheet WH_Redeliveries retains absolute-count sort; this chart sorts by rate.
_c3 = tbl_wh_redeliveries[tbl_wh_redeliveries['total_tickets'].fillna(0) >= MIN_TICKETS_FOR_RATE].copy()
_c3 = _c3.dropna(subset=['redel_pct_of_tickets']).sort_values('redel_pct_of_tickets', ascending=True)
fig,ax = plt.subplots(figsize=(11,max(6,len(_c3)*0.4)))
ax.barh(_c3['tech_warehouse'], _c3['redel_pct_of_tickets'], color=PALETTE['redelivery'], alpha=0.85)
_xmax = float(_c3['redel_pct_of_tickets'].max()) if len(_c3)>0 else 0
for bar, (_, row) in zip(ax.patches, _c3.iterrows()):
    ax.text(bar.get_width()+_xmax*0.01, bar.get_y()+bar.get_height()/2,
            f"{row['redel_pct_of_tickets']:.2f}%  ({int(row['redelivery_count']):,})",
            va='center', ha='left', fontsize=8)
ax.set_xlabel(f'Redeliveries as % of Total Tickets (min {MIN_TICKETS_FOR_RATE} tickets)')
ax.set_title('C3: Warehouse Redelivery Rate (% of Tickets)', fontsize=12, fontweight='bold')
ax.set_xlim(right=_xmax*1.30 if _xmax>0 else 1)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.2f}%'))
ax.grid(axis='x', linestyle='--', alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout(); save_fig(fig,'C3_Redelivery_Rate_By_WH'); plt.show(); plt.close(fig)

## Cell 22 — MONTHLY REDELIVERIES STACKED BY PRODUCT + STATE/METRO CHARTS

- **C4:** Monthly stacked bar by product (Top 10 + Other) — restored from prior version
- **C5:** Total redeliveries by state (bar chart)
- **C6:** Total redeliveries by metro (bar chart)
- **C7:** Monthly redeliveries trend by state (line)
- **C8:** Monthly redeliveries trend by metro (line)

In [ ]:
# ── C4: Monthly stacked by product ───────────────────────────────────────────
TOP_PRODUCTS = 10
if len(redel_linked) > 0:
    _periods = sorted(redel_linked['period'].dropna().unique())
    _product_totals = redel_linked.groupby('product',dropna=False)['orig_order_num'].count().sort_values(ascending=False)
    _top_products   = _product_totals.head(TOP_PRODUCTS).index.tolist()
    redel_linked['_product_bucket'] = redel_linked['product'].where(redel_linked['product'].isin(_top_products), other='Other')
    _pivot = (redel_linked.groupby(['period','_product_bucket'],dropna=False).size().unstack(fill_value=0).reindex(index=_periods,fill_value=0))
    for _p in _top_products+['Other']:
        if _p not in _pivot.columns: _pivot[_p]=0
    _pivot = _pivot[_top_products+(['Other'] if 'Other' in _pivot.columns else [])]
    _tab10=plt.get_cmap('tab10'); _tab20b=plt.get_cmap('tab20b')
    _pcolors={}
    for _i,_p in enumerate(_top_products): _pcolors[_p]=_tab10(_i) if _i<10 else _tab20b((_i-10)%20)
    _pcolors['Other']='#BFBFBF'
    _x=np.arange(len(_periods)); _bottom=np.zeros(len(_periods))
    fig,ax=plt.subplots(figsize=(max(14,len(_periods)*0.95),7))
    for _p in _pivot.columns:
        _vals=_pivot[_p].values.astype(float)
        _tot_lbl=int(_product_totals.get(_p,_pivot[_p].sum()))
        ax.bar(_x,_vals,width=0.72,bottom=_bottom,color=_pcolors[_p],label=f'{_p}({_tot_lbl})',edgecolor='white',linewidth=0.3)
        _bottom+=_vals
    ax.set_xticks(_x); ax.set_xticklabels(_periods,rotation=45,ha='right',fontsize=9)
    ax.set_ylabel('Redelivery Orders',fontsize=11)
    ax.set_title(f'C4: Monthly Redeliveries — Stacked by Product (Top {TOP_PRODUCTS} + Other)',fontsize=13,fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{int(x):,}'))
    ax.grid(axis='y',linestyle='--',alpha=0.35); ax.set_axisbelow(True); ax.set_xlim(-0.6,len(_periods)-0.4)
    ax.legend(loc='upper left',bbox_to_anchor=(1.01,1),fontsize=8,framealpha=0.9,borderpad=0.8)
    fig.tight_layout(rect=[0,0,0.82,1]); save_fig(fig,'C4_Redeliveries_Monthly_By_Product'); plt.show(); plt.close(fig)
    print(f'C4: {len(_periods)} months | {len(_top_products)} named products')

# ── C5_VP: VP redelivery rate (% of tickets) —──────────────────
# Sort by rate; annotate with rate + absolute count.
if 'tbl_vp_redeliveries' in dir() and len(tbl_vp_redeliveries)>0:
    _c5vp = tbl_vp_redeliveries.dropna(subset=['redel_pct_of_tickets']).sort_values('redel_pct_of_tickets', ascending=True)
    fig,ax = plt.subplots(figsize=(10,max(3.5,len(_c5vp)*0.6)))
    ax.barh(_c5vp['vp'].fillna('(no VP)').astype(str), _c5vp['redel_pct_of_tickets'], color='#9467bd', alpha=0.85)
    _xmax = float(_c5vp['redel_pct_of_tickets'].max()) if len(_c5vp)>0 else 0
    for bar,(_,row) in zip(ax.patches, _c5vp.iterrows()):
        ax.text(bar.get_width()+_xmax*0.01, bar.get_y()+bar.get_height()/2,
                f"{row['redel_pct_of_tickets']:.2f}%  ({int(row['redelivery_count']):,} of {int(row['total_tickets']):,})",
                va='center', ha='left', fontsize=9)
    ax.set_xlabel('Redeliveries as % of Total Tickets')
    ax.set_title('C5_VP: VP Redelivery Rate (% of Tickets)', fontsize=12, fontweight='bold')
    ax.set_xlim(right=_xmax*1.40 if _xmax>0 else 1)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.2f}%'))
    ax.grid(axis='x', linestyle='--', alpha=0.4); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'C5_VP_Redelivery_Rate'); plt.show(); plt.close(fig)

# ── C5: State redelivery rate (% of tickets) —──────────────────────
# Sort by rate; annotate with rate, absolute count, and warehouse count.
if len(tbl_state_redeliveries)>0:
    _c5 = tbl_state_redeliveries.dropna(subset=['redel_pct_of_tickets']).sort_values('redel_pct_of_tickets', ascending=True)
    fig,ax = plt.subplots(figsize=(10,max(4,len(_c5)*0.5)))
    ax.barh(_c5['state'], _c5['redel_pct_of_tickets'], color='#17becf', alpha=0.85)
    _xmax = float(_c5['redel_pct_of_tickets'].max()) if len(_c5)>0 else 0
    for bar,(_,row) in zip(ax.patches, _c5.iterrows()):
        ax.text(bar.get_width()+_xmax*0.01, bar.get_y()+bar.get_height()/2,
                f"{row['redel_pct_of_tickets']:.2f}%  ({int(row['redelivery_count']):,} | {int(row['distinct_warehouses'])} WH)",
                va='center', ha='left', fontsize=9)
    ax.set_xlabel('Redeliveries as % of Total Tickets')
    ax.set_title('C5: State Redelivery Rate (% of Tickets)', fontsize=12, fontweight='bold')
    ax.set_xlim(right=_xmax*1.40 if _xmax>0 else 1)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.2f}%'))
    ax.grid(axis='x', linestyle='--', alpha=0.4); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'C5_Redeliveries_By_State'); plt.show(); plt.close(fig)

# ── C6: Total redeliveries by metro 
if redel_linked['metro'].notna().any():
    _c6 = (redel_linked[redel_linked['metro'].notna()]
           .groupby('metro', as_index=False)
           .agg(redelivery_count=('event_key','nunique'),
                distinct_warehouses=('tech_warehouse','nunique'),
                states_in_metro=('state', lambda x: ', '.join(sorted(x.dropna().astype(str).unique())))))
    # Bring in the rate from tbl_metro_redeliveries if multiple state-rows exist, 
    # sum redelivery_count and total_tickets directly. Otherwise pull from the existing table.
    _tx_metro = (df_tx[df_tx['metro'].notna()].groupby('metro',as_index=False)
                 .agg(total_tickets=('order_num','nunique')))
    _c6 = _c6.merge(_tx_metro, on='metro', how='left')
    _c6['redel_pct_of_tickets'] = (_c6['redelivery_count']/_c6['total_tickets'].replace(0,np.nan)*100).round(2)
    _c6 = _c6.sort_values('redel_pct_of_tickets', ascending=True).reset_index(drop=True)  # v1.30.5

    # Warn on state inconsistency so the user can clean SERP_WAREHOUSES
    _inconsistent = _c6[_c6['states_in_metro'].str.contains(',', na=False)]
    if len(_inconsistent) > 0:
        print('  C6 WARNING: state inconsistency in SERP_WAREHOUSES for these metros:')
        for _, r in _inconsistent.iterrows():
            print(f"    {r['metro']}: states = [{r['states_in_metro']}]")

    fig, ax = plt.subplots(figsize=(10, max(3.5, len(_c6)*0.85)))
    ax.barh(_c6['metro'], _c6['redelivery_count'], color='#bcbd22', alpha=0.85, height=0.6)
    _xmax = _c6['redelivery_count'].max()
    for bar, (_, row) in zip(ax.patches, _c6.iterrows()):
        ax.text(bar.get_width() + _xmax*0.01,
                bar.get_y() + bar.get_height()/2,
                f"{int(row['redelivery_count']):,}  |  {row['redel_pct_of_tickets']:.2f}% of tickets  |  {int(row['distinct_warehouses'])} WH",
                va='center', ha='left', fontsize=9)
    ax.set_xlabel('Total Redelivery Orders')
    ax.set_title('C6: Total Redeliveries by Metro Area', fontsize=12, fontweight='bold')
    ax.set_xlim(right=_xmax*1.45)
    ax.grid(axis='x', linestyle='--', alpha=0.4); ax.set_axisbelow(True)
    ax.margins(y=0.15)  # extra vertical breathing room between bars
    fig.tight_layout()
    save_fig(fig, 'C6_Redeliveries_By_Metro'); plt.show(); plt.close(fig)

# ── C7: Monthly redeliveries by state (line chart) ───────────────────────────
if len(tbl_redel_monthly_state)>0:
    _states=sorted(tbl_redel_monthly_state['state'].dropna().unique())
    _cmap_s=plt.get_cmap('tab20'); _sc={s:_cmap_s(i%20) for i,s in enumerate(_states)}
    fig,ax=plt.subplots(figsize=(14,6))
    for s in _states:
        _d=tbl_redel_monthly_state[tbl_redel_monthly_state['state']==s].sort_values('period')
        if _d.empty: continue
        _xs=pd.to_datetime(_d['period']+'-01'); _ys=_d['redelivery_count'].values
        ax.plot(_xs,_ys,marker='o',linewidth=2,markersize=5,label=s,color=_sc[s])
    ax.set_ylabel('Redelivery Orders'); ax.set_title('C7: Monthly Redeliveries by State',fontsize=12,fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{int(x):,}'))
    ax.legend(loc='upper left',fontsize=8,ncol=3,framealpha=0.85)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m')); ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(),rotation=45,ha='right',fontsize=8)
    ax.grid(axis='y',linestyle='--',alpha=0.35); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'C7_Redeliveries_Monthly_By_State'); plt.show(); plt.close(fig)

# ── C8: Monthly redeliveries by metro (line chart) ───────────────────────────
if len(tbl_redel_monthly_metro)>0:
    _metro_states = (tbl_redel_monthly_metro.groupby('metro')['state']
                     .nunique().reset_index(name='n_states'))
    _multi_state = _metro_states[_metro_states['n_states']>1]
    if len(_multi_state)>0:
        print(f'INFO C8: {len(_multi_state)} metro(s) span >1 state; summing across states for the line chart:')
        for _, r in _multi_state.iterrows():
            _sts = sorted(tbl_redel_monthly_metro.loc[tbl_redel_monthly_metro['metro']==r['metro'],'state'].dropna().unique())
            print(f"  {r['metro']}: {_sts}")
    _c8_plot = (tbl_redel_monthly_metro
                .groupby(['metro','period'], as_index=False)
                .agg(redelivery_count=('redelivery_count','sum')))
    _metros=sorted(_c8_plot['metro'].dropna().unique())
    _mc={m:plt.get_cmap('tab10')(i%10) for i,m in enumerate(_metros)}
    fig,ax=plt.subplots(figsize=(14,5))
    for m in _metros:
        _d=_c8_plot[_c8_plot['metro']==m].sort_values('period')
        if _d.empty: continue
        _xs=pd.to_datetime(_d['period']+'-01'); _ys=_d['redelivery_count'].values
        ax.plot(_xs,_ys,marker='o',linewidth=2,markersize=6,label=m,color=_mc[m])
        for _px,_py in zip(_xs,_ys):
            if pd.notna(_py): ax.annotate(f'{int(_py):,}',xy=(_px,_py),xytext=(0,6),textcoords='offset points',fontsize=6.5,color=_mc[m],ha='center',va='bottom')
    ax.set_ylabel('Redelivery Orders'); ax.set_title('C8: Monthly Redeliveries by Metro (DFW / Houston / San Antonio)',fontsize=12,fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{int(x):,}'))
    ax.legend(loc='upper left',fontsize=9,framealpha=0.85)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m')); ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(),rotation=45,ha='right',fontsize=8)
    ax.grid(axis='y',linestyle='--',alpha=0.35); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'C8_Redeliveries_Monthly_By_Metro'); plt.show(); plt.close(fig)

# ── C9: Company-wide monthly redeliveries — count bars + rate line ──────────
# v1.30.5 NEW. tbl_redel_monthly already has both columns; this is just a chart.
# Dual-axis: bars (left, absolute count) + line (right, % of tickets). Note that
# dual-axis charts can mislead if scales aren't read carefully — the count and
# rate are independently scaled.
if len(tbl_redel_monthly) > 0:
    _c9 = tbl_redel_monthly.sort_values(['delivery_year','delivery_month']).copy()
    _c9['_pd'] = pd.to_datetime(_c9['period']+'-01')
    fig, ax_l = plt.subplots(figsize=(max(12, len(_c9)*0.85), 6))
    # Bars: absolute redelivery count
    _x = np.arange(len(_c9))
    _bars = ax_l.bar(_x, _c9['redelivery_count'].values,
                     color=PALETTE['redelivery'], alpha=0.75, label='Redelivery count', width=0.7)
    ax_l.set_ylabel('Redelivery Count', color=PALETTE['redelivery'])
    ax_l.tick_params(axis='y', labelcolor=PALETTE['redelivery'])
    ax_l.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{int(x):,}'))
    # Annotate count above each bar
    _ymax_l = float(_c9['redelivery_count'].max()) if len(_c9)>0 else 0
    for _bar, _v in zip(_bars, _c9['redelivery_count'].values):
        if pd.notna(_v):
            ax_l.annotate(f'{int(_v):,}', xy=(_bar.get_x()+_bar.get_width()/2, _v),
                          xytext=(0, 3), textcoords='offset points',
                          fontsize=7, color=PALETTE['redelivery'], ha='center', va='bottom')
    # Right axis: % of tickets, line overlay
    ax_r = ax_l.twinx()
    ax_r.plot(_x, _c9['redel_pct_of_tickets'].values,
              color='#d62728', marker='D', markersize=6, linewidth=2.2,
              label='% of tickets', zorder=5)
    ax_r.set_ylabel('Redeliveries as % of Tickets', color='#d62728')
    ax_r.tick_params(axis='y', labelcolor='#d62728')
    ax_r.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.1f}%'))
    ax_r.set_ylim(bottom=0)
    # Annotate % below each marker
    for _xi, _v in zip(_x, _c9['redel_pct_of_tickets'].values):
        if pd.notna(_v):
            ax_r.annotate(f'{_v:.2f}%', xy=(_xi, _v), xytext=(0, -12),
                          textcoords='offset points', fontsize=7, color='#d62728',
                          ha='center', va='top', fontweight='bold')
    ax_l.set_xticks(_x); ax_l.set_xticklabels(_c9['period'].values, rotation=45, ha='right', fontsize=9)
    ax_l.set_title('C9: Company-Wide Monthly Redeliveries — Count & Rate', fontsize=12, fontweight='bold')
    ax_l.grid(axis='y', linestyle='--', alpha=0.3); ax_l.set_axisbelow(True)
    # Combine legends from both axes
    _h1,_l1 = ax_l.get_legend_handles_labels(); _h2,_l2 = ax_r.get_legend_handles_labels()
    ax_l.legend(_h1+_h2, _l1+_l2, loc='upper left', fontsize=9, framealpha=0.9)
    fig.tight_layout(); save_fig(fig, 'C9_Redeliveries_Monthly_Company'); plt.show(); plt.close(fig)
    print(f'C9: {len(_c9)} months plotted (count + % of tickets dual-axis)')

## Cell 22b — TECH SCORECARD: TICKETS + REDELIVERIES + LOST + OVERTIME (v1.33.0)

One row per **(tech, month)** at the *name* level (a day worked across two
warehouses counts once — this grain exists to answer "how loaded is this person,"
not "how loaded is this warehouse"). Joins:

- Tickets / visits / active days / tickets-per-active-day (headline denominator)
- Redeliveries — **originating-tech attribution** (locked decision #4)
- Lost recovery — the three definitions from Cell 17c, keyed to the **month the
  item was marked lost** (which can trail the causal visit by weeks)
- Overtime — FLSA-style weekly inference (Cell 13), matched via the payroll name map

Charts: **E1** top OT% techs (hours floor applied), **E2** company monthly OT
count-and-rate, **E3** productivity vs OT scatter (are we buying tickets with OT,
or is OT concentrated in low-output techs? — that distinction drives the fix).


In [ ]:
# v1.33.0 (N3) — Tech scorecard at (tech, month) grain
_att = df_tx[~df_tx['_unattributed']]
_sc_keys = ['techfirstname','techlastname','delivery_year','delivery_month']

_sc_base = (_att.groupby(_sc_keys, dropna=False, as_index=False)
    .agg(total_tickets=('order_num','nunique'),
         unique_patients=('record_id','nunique'),
         active_days=('completed_date','nunique'),   # name-level: a day counts once
         warehouse_count=('tech_warehouse','nunique'),
         warehouses=('tech_warehouse', lambda x: ', '.join(sorted(x.dropna().unique())))))
_sc_v = (df_visits[~df_visits['_unattributed']].groupby(_sc_keys, dropna=False, as_index=False)
    .agg(total_visits=('order_num','nunique')))
tbl_tech_scorecard_monthly = _sc_base.merge(_sc_v, on=_sc_keys, how='left')
tbl_tech_scorecard_monthly['tickets_per_active_day'] = (
    tbl_tech_scorecard_monthly['total_tickets']
    / tbl_tech_scorecard_monthly['active_days'].replace(0,np.nan)).round(3)

# ── Redeliveries (originating-tech attribution — locked decision #4) ─────────
_sc_rd = (_redel_attrib.groupby(_sc_keys, dropna=False, as_index=False)
    .agg(redelivery_count=('event_key','nunique')))
tbl_tech_scorecard_monthly = tbl_tech_scorecard_monthly.merge(_sc_rd, on=_sc_keys, how='left')
tbl_tech_scorecard_monthly['redelivery_count'] = tbl_tech_scorecard_monthly['redelivery_count'].fillna(0).astype(int)
tbl_tech_scorecard_monthly['redel_pct_of_tickets'] = (
    tbl_tech_scorecard_monthly['redelivery_count']
    / tbl_tech_scorecard_monthly['total_tickets'].replace(0,np.nan) * 100).round(2)

# ── Lost recovery (three definitions; month = month item was MARKED lost) ────
if len(tbl_tech_lost_monthly_defs) > 0:
    _pre = len(tbl_tech_scorecard_monthly)
    tbl_tech_scorecard_monthly = tbl_tech_scorecard_monthly.merge(
        tbl_tech_lost_monthly_defs, on=_sc_keys, how='left')
    assert len(tbl_tech_scorecard_monthly) == _pre, 'lost-defs merge multiplied rows'
for _c in ['lost_all_items','lost_no_pickup_items','lost_after_pickup_items']:
    if _c not in tbl_tech_scorecard_monthly.columns: tbl_tech_scorecard_monthly[_c] = 0
    tbl_tech_scorecard_monthly[_c] = tbl_tech_scorecard_monthly[_c].fillna(0).astype(int)
# NOTE: a lost item attributed to a tech-month where that tech ran zero tickets
# will not surface here (left join from the ticket base). Tech_Lost_Scorecard
# carries the complete picture.

# ── Overtime (matched via payroll name map) ──────────────────────────────────
# df_plc_map is unique on tech name; warn if two tech spellings map to one payroll
# identity (their OT would then appear on both scorecard rows).
_dup_plc = df_plc_map.groupby(['plc_first','plc_last']).size()
_n_dup_plc = int((_dup_plc > 1).sum())
if _n_dup_plc > 0:
    print(f'  *** WARNING: {_n_dup_plc} payroll identities map to >1 ticket-name spelling; '
          f'their OT appears on each spelling\'s scorecard row. Review df_plc_map.')
_sc_ot = (tbl_tech_scorecard_monthly[['techfirstname','techlastname']].drop_duplicates()
          .merge(df_plc_map, on=['techfirstname','techlastname'], how='inner')
          .merge(df_ot_monthly.rename(columns={'h_first':'plc_first','h_last':'plc_last'}),
                 on=['plc_first','plc_last'], how='inner'))
_pre = len(tbl_tech_scorecard_monthly)
tbl_tech_scorecard_monthly = tbl_tech_scorecard_monthly.merge(
    _sc_ot[['techfirstname','techlastname','delivery_year','delivery_month',
            'total_hours','ot_hours','ot_pct','max_week_hours']],
    on=_sc_keys, how='left')
assert len(tbl_tech_scorecard_monthly) == _pre, 'OT merge multiplied rows'
tbl_tech_scorecard_monthly['period'] = (
    tbl_tech_scorecard_monthly['delivery_year'].astype('Int64').astype(str) + '-' +
    tbl_tech_scorecard_monthly['delivery_month'].astype('Int64').astype(str).str.zfill(2))
tbl_tech_scorecard_monthly = tbl_tech_scorecard_monthly.sort_values(
    ['techlastname','techfirstname','delivery_year','delivery_month']).reset_index(drop=True)

# ── All-period summary (pooled sums, NOT mean-of-monthly-ratios) ─────────────
tbl_tech_scorecard = (tbl_tech_scorecard_monthly
    .groupby(['techfirstname','techlastname'], dropna=False, as_index=False)
    .agg(months_active=('delivery_month','nunique'),
         total_tickets=('total_tickets','sum'), total_visits=('total_visits','sum'),
         active_days=('active_days','sum'),
         redelivery_count=('redelivery_count','sum'),
         lost_all_items=('lost_all_items','sum'),
         lost_no_pickup_items=('lost_no_pickup_items','sum'),
         lost_after_pickup_items=('lost_after_pickup_items','sum'),
         total_hours=('total_hours','sum'), ot_hours=('ot_hours','sum'),
         warehouses=('warehouses', lambda x: ', '.join(sorted(set(w for v in x.dropna() for w in v.split(', ')))))))
tbl_tech_scorecard['tickets_per_active_day'] = (tbl_tech_scorecard['total_tickets']
    / tbl_tech_scorecard['active_days'].replace(0,np.nan)).round(3)
tbl_tech_scorecard['redel_pct_of_tickets'] = (tbl_tech_scorecard['redelivery_count']
    / tbl_tech_scorecard['total_tickets'].replace(0,np.nan) * 100).round(2)
tbl_tech_scorecard['ot_pct'] = (tbl_tech_scorecard['ot_hours']
    / tbl_tech_scorecard['total_hours'].replace(0,np.nan) * 100).round(2)
tbl_tech_scorecard = tbl_tech_scorecard.sort_values('total_tickets', ascending=False).reset_index(drop=True)

# Payroll-name OT sheet (all PCT payroll, incl. techs unmatched to tickets)
tbl_tech_ot_monthly_export = df_ot_monthly.copy()
tbl_tech_ot_monthly_export['period'] = (
    tbl_tech_ot_monthly_export['delivery_year'].astype(int).astype(str) + '-' +
    tbl_tech_ot_monthly_export['delivery_month'].astype(int).astype(str).str.zfill(2))
tbl_tech_ot_monthly_export = tbl_tech_ot_monthly_export.sort_values(
    ['h_last','h_first','delivery_year','delivery_month']).reset_index(drop=True)

print(f'Scorecard monthly: {len(tbl_tech_scorecard_monthly):,} rows | '
      f'summary: {len(tbl_tech_scorecard):,} techs | '
      f'OT-matched tech-months: {tbl_tech_scorecard_monthly["ot_hours"].notna().sum():,}')

# ── E1: Top techs by OT% (hours floor) ───────────────────────────────────────
_e1 = (tbl_tech_scorecard[tbl_tech_scorecard['total_hours'].fillna(0) >= MIN_HOURS_FOR_OT_RATE]
       .dropna(subset=['ot_pct']).sort_values('ot_pct', ascending=False).head(20).copy())
if len(_e1) > 0:
    _e1['label'] = _e1['techfirstname']+' '+_e1['techlastname']
    fig,ax = plt.subplots(figsize=(11,max(5,len(_e1)*0.45)))
    ax.barh(_e1['label'][::-1], _e1['ot_pct'][::-1], color='#d62728', alpha=0.85)
    _xmax = float(_e1['ot_pct'].max())
    for bar,(_,row) in zip(ax.patches, _e1[::-1].iterrows()):
        ax.text(bar.get_width()+_xmax*0.01, bar.get_y()+bar.get_height()/2,
                f"{row['ot_pct']:.1f}%  ({row['ot_hours']:,.0f} OT of {row['total_hours']:,.0f} hrs | {row['tickets_per_active_day'] if pd.notna(row['tickets_per_active_day']) else 0:.2f} tkt/day)",
                va='center', ha='left', fontsize=8)
    ax.set_title(f'E1: Top 20 Technicians by Overtime %% of Hours (min {MIN_HOURS_FOR_OT_RATE:.0f} hrs; inferred >40/wk)',
                 fontsize=11, fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.1f}%'))
    ax.set_xlim(right=_xmax*1.65 if _xmax>0 else 1)
    ax.grid(axis='x',linestyle='--',alpha=0.4); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'E1_OT_Pct_By_Tech'); plt.show(); plt.close(fig)

# ── E2: Company monthly OT — hours bars + rate line ──────────────────────────
if len(tbl_ot_company_monthly) > 0:
    _e2 = tbl_ot_company_monthly.sort_values(['delivery_year','delivery_month']).copy()
    _x = np.arange(len(_e2))
    fig, axl = plt.subplots(figsize=(max(12,len(_e2)*0.85), 5.5))
    _bars = axl.bar(_x, _e2['ot_hours'].values, color='#ff7f0e', alpha=0.8, width=0.7, label='OT hours')
    axl.set_ylabel('OT Hours', color='#ff7f0e'); axl.tick_params(axis='y', labelcolor='#ff7f0e')
    for _b,_v in zip(_bars,_e2['ot_hours'].values):
        axl.annotate(f'{_v:,.0f}', xy=(_b.get_x()+_b.get_width()/2,_v), xytext=(0,3),
                     textcoords='offset points', fontsize=7, color='#ff7f0e', ha='center')
    axr = axl.twinx()
    axr.plot(_x, _e2['ot_pct'].values, color='#d62728', marker='D', markersize=6, linewidth=2.2, label='OT %% of hours')
    axr.set_ylabel('OT as %% of Total Hours', color='#d62728'); axr.tick_params(axis='y', labelcolor='#d62728')
    axr.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.1f}%')); axr.set_ylim(bottom=0)
    for _xi,_v in zip(_x,_e2['ot_pct'].values):
        if pd.notna(_v): axr.annotate(f'{_v:.1f}%', xy=(_xi,_v), xytext=(0,-12),
            textcoords='offset points', fontsize=7, color='#d62728', ha='center', va='top', fontweight='bold')
    axl.set_xticks(_x); axl.set_xticklabels(_e2['period'].values, rotation=45, ha='right', fontsize=9)
    axl.set_title('E2: PCT Department Monthly Overtime — Hours & Rate (inferred >40 hrs/wk, Sun–Sat)',
                  fontsize=12, fontweight='bold')
    axl.grid(axis='y', linestyle='--', alpha=0.3); axl.set_axisbelow(True)
    _h1,_l1=axl.get_legend_handles_labels(); _h2,_l2=axr.get_legend_handles_labels()
    axl.legend(_h1+_h2,_l1+_l2, loc='upper left', fontsize=9)
    fig.tight_layout(); save_fig(fig,'E2_OT_Company_Monthly'); plt.show(); plt.close(fig)

# ── E3: Productivity vs OT scatter ───────────────────────────────────────────
_e3 = tbl_tech_scorecard[(tbl_tech_scorecard['total_hours'].fillna(0) >= MIN_HOURS_FOR_OT_RATE)
                         & tbl_tech_scorecard['tickets_per_active_day'].notna()
                         & tbl_tech_scorecard['ot_pct'].notna()].copy()
if len(_e3) >= 5:
    fig,ax = plt.subplots(figsize=(10,7))
    ax.scatter(_e3['tickets_per_active_day'], _e3['ot_pct'], s=_e3['total_tickets']/max(1,_e3['total_tickets'].max())*300+15,
               alpha=0.55, color=PALETTE['attributed'], edgecolors='white', linewidths=0.5)
    _mx, _my = _e3['tickets_per_active_day'].median(), _e3['ot_pct'].median()
    ax.axvline(_mx, color='#888', linestyle='--', linewidth=1); ax.axhline(_my, color='#888', linestyle='--', linewidth=1)
    ax.text(0.99,0.99,'High OT, high output\n(capacity-constrained?)',transform=ax.transAxes,ha='right',va='top',fontsize=8,color='#555')
    ax.text(0.01,0.99,'High OT, low output\n(routing / territory review)',transform=ax.transAxes,ha='left',va='top',fontsize=8,color='#555')
    ax.set_xlabel('Tickets per Active Day (all-period pooled)')
    ax.set_ylabel('OT %% of Hours (inferred)')
    ax.set_title(f'E3: Productivity vs Overtime by Technician (n={len(_e3)}; bubble = ticket volume;\ndashed = medians)',
                 fontsize=11, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x:.0f}%'))
    ax.grid(linestyle='--', alpha=0.3); ax.set_axisbelow(True)
    fig.tight_layout(); save_fig(fig,'E3_Productivity_vs_OT'); plt.show(); plt.close(fig)
else:
    print(f'E3 scatter skipped: only {len(_e3)} techs pass the hours floor.')


## Cell 23 — EXCEL EXPORT

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

_HF=PatternFill('solid',start_color='1F4E79'); _AF=PatternFill('solid',start_color='EBF3FB')
_HN=Font(name='Arial',bold=True,color='FFFFFF',size=10); _BN=Font(name='Arial',size=10)
_TH=Side(style='thin',color='B8CCE4'); _BD=Border(left=_TH,right=_TH,top=_TH,bottom=_TH)
_PCT={'visits_per_workhour','tickets_per_workhour','avg_min_per_visit','workload_index','dark_rate_pct',
      'pct_of_total','pct_of_dark','avg_daily_tickets_per_tech','avg_daily_visits_per_tech',
      'avg_daily_tickets','redel_per_100_visits','lost_cost_per_adc','lost_cost_per_pt_day',
      'mom_delta','mom_pct','mean_tickets','median_tickets','p75_tickets','p90_tickets','p99_tickets',
      'redel_pct_of_tickets','lost_per_100_deliveries',
      'lost_amount_pct_of_inventory','lost_count_pct_of_inventory',
      'tickets_per_active_day','visits_per_active_day',
      'tickets_per_active_day_per_tech','visits_per_active_day_per_tech',
      'utilization_pct','mom_utilization_pct','mom_legacy_metric','mom_active_metric',
      'mom_tickets_pct',
      'mtd_avg_daily_tickets','mtd_avg_daily_tickets_per_tech','mtd_avg_daily_visits_per_tech',
      'ratio_actual_vs_expected',
      'ot_pct','avg_daily_tickets_DEPRECATED'}  # v1.33.0
_NUM={'total_visits','total_tickets','total_workhours','tech_count','unique_patients',
      'redelivery_count','redelivery_items','dark_tickets','blank_tickets','dark_n','total_n',
      'lost_asset_count','ticket_count','patient_count','pickup_ticket_count',
      'deliveries','pickups','services','unattributed_count','warehouse_count','days_on_service',
      'total_inventory_count','total_inventory_amount',
      'total_deliveries','distinct_warehouses','affected_techs',
      'active_days','total_active_days','expected_tech_days','weekday_days',
      'mom_tickets','mom_tech_count','mom_active_days',
      'weekday_days_elapsed','expected_tech_days_elapsed','ticket_count','trailing_3mo_median',
      'weekday_days_full','expected_tickets_mtd_scaled',
      'lost_assets_attributed','distinct_patients','days_pickup_to_lost',
      'avg_days_pickup_to_lost','median_days_pickup_to_lost',
      'ot_hours','total_hours','reg_hours','week_hours','max_week_hours','weeks_worked',
      'unlinked_events','unlinked_items','total_days',
      'lost_all_items','lost_no_pickup_items','lost_after_pickup_items'}  # v1.33.0

def _excel_safe(df):
    if df is None or len(df.columns)==0:
        return df
    out = df.copy()
    for col in out.columns:
        s = out[col]
        # Pandas nullable extension dtypes: cast to object with None for NA
        if pd.api.types.is_extension_array_dtype(s):
            out[col] = s.astype(object).where(s.notna(), None)
        # Datetime NaT also needs to become None (openpyxl handles datetime but not NaT mixed)
        elif pd.api.types.is_datetime64_any_dtype(s):
            out[col] = s.astype(object).where(s.notna(), None)
    return out

def _cell_val(v):
    if v is None: return None
    try:
        if v is pd.NA: return None
    except Exception: pass
    # pd.isna handles NaT, NaN, pd.NA; but it ALSO returns True for arrays — guard with scalar check
    try:
        if pd.isna(v): return None
    except (TypeError, ValueError):
        pass
    return v

def _ws(wb,title,df,fcol=1):
    df = _excel_safe(df) 
    ws=wb.create_sheet(title=title); cols=list(df.columns)
    for c,col in enumerate(cols,start=1):
        cell=ws.cell(row=1,column=c,value=col)
        cell.fill=_HF; cell.font=_HN; cell.alignment=Alignment(horizontal='center',vertical='center'); cell.border=_BD
    ws.freeze_panes=ws.cell(row=2,column=fcol+1)
    for ri,row in enumerate(df.itertuples(index=False),start=2):
        fi=_AF if ri%2==0 else PatternFill()
        for ci,(val,col) in enumerate(zip(row,cols),start=1):
            val=_cell_val(val) 
            cell=ws.cell(row=ri,column=ci,value=val); cell.font=_BN; cell.border=_BD; cell.fill=fi
            if col in _PCT: cell.number_format='0.00'; cell.alignment=Alignment(horizontal='right')
            elif col in _NUM or isinstance(val,int): cell.number_format='#,##0'; cell.alignment=Alignment(horizontal='right')
            elif isinstance(val,float): cell.number_format='0.00'; cell.alignment=Alignment(horizontal='right')
            else: cell.alignment=Alignment(horizontal='left')
    for cc in ws.columns:
        w=max((len(str(c.value or '')) for c in cc),default=8)
        ws.column_dimensions[get_column_letter(cc[0].column)].width=min(45,max(10,w+2))

def _add_period(df, yr='delivery_year', mo='delivery_month'):
    df = df.copy()
    if len(df)==0:
        df['period'] = pd.Series(dtype='object')
        return df
    df['period'] = (df[yr].astype('Int64').astype(str)
                    + '-' + df[mo].astype('Int64').astype(str).str.zfill(2))
    return df

# --- Redeliveries: VP monthly, Tech monthly, Product monthly ---
_redel_vp_m = (redel_linked.groupby(['vp','delivery_year','delivery_month'],dropna=False,as_index=False)
               .agg(redelivery_count=('event_key','nunique'),
                    redelivery_items=('product','count'),
                    distinct_warehouses=('tech_warehouse','nunique'),
                    affected_techs=('techfirstname','nunique')))
_tx_vp_m = (df_tx.groupby(['vp','delivery_year','delivery_month'],dropna=False,as_index=False)
            .agg(total_tickets=('order_num','nunique')))
tbl_vp_redeliveries_monthly = _redel_vp_m.merge(_tx_vp_m, on=['vp','delivery_year','delivery_month'], how='left')
tbl_vp_redeliveries_monthly['redel_pct_of_tickets'] = (
    tbl_vp_redeliveries_monthly['redelivery_count']
    / tbl_vp_redeliveries_monthly['total_tickets'].replace(0,np.nan) * 100).round(2)
tbl_vp_redeliveries_monthly = _add_period(tbl_vp_redeliveries_monthly).sort_values(['vp','period'])

_redel_tech_m = (redel_linked[redel_linked['techfirstname'].fillna('').str.strip().ne('')
                              & redel_linked['techlastname'].fillna('').str.strip().ne('')]
                 .groupby(['techfirstname','techlastname','tech_warehouse','region','vp','state','metro',
                           'delivery_year','delivery_month'], dropna=False, as_index=False)
                 .agg(redelivery_count=('event_key','nunique'),
                      redelivery_items=('product','count')))
# v1.33.0 (H2): tickets denominator at (tech, warehouse, month) grain to match rows.
_tx_tech_m = (df_tx[~df_tx['_unattributed']]
              .groupby(['techfirstname','techlastname','tech_warehouse','delivery_year','delivery_month'],dropna=False,as_index=False)
              .agg(total_tickets=('order_num','nunique')))
tbl_tech_redeliveries_monthly = _redel_tech_m.merge(_tx_tech_m,
    on=['techfirstname','techlastname','tech_warehouse','delivery_year','delivery_month'], how='left')
tbl_tech_redeliveries_monthly['redel_pct_of_tickets'] = (
    tbl_tech_redeliveries_monthly['redelivery_count']
    / tbl_tech_redeliveries_monthly['total_tickets'].replace(0,np.nan) * 100).round(2)
tbl_tech_redeliveries_monthly = _add_period(tbl_tech_redeliveries_monthly)

tbl_redel_product_monthly = (redel_linked.groupby(['product','delivery_year','delivery_month'],dropna=False,as_index=False)
                             .agg(redelivery_count=('event_key','nunique'),
                                  warehouses=('tech_warehouse','nunique')))
tbl_redel_product_monthly = _add_period(tbl_redel_product_monthly).sort_values(['product','period'])

# --- Lost equipment: VP monthly, Total monthly, Tech-summary monthly ---
# Source: _lost_filt (already filtered to window in Cell 17). lost_year/lost_month
# are renamed to delivery_year/delivery_month for sheet-naming consistency.
tbl_lost_vp_monthly = (_lost_filt.groupby(['vp','lost_year','lost_month'],dropna=False,as_index=False)
                       .agg(lost_asset_count=('asset_tag','nunique'),
                            lost_asset_cost=('lost_cost_last_price','sum'),
                            distinct_warehouses=('tech_warehouse','nunique'),
                            lost_product_types=('product_name','nunique'))
                       .rename(columns={'lost_year':'delivery_year','lost_month':'delivery_month'}))
tbl_lost_vp_monthly = _add_period(tbl_lost_vp_monthly).sort_values(['vp','period'])

tbl_lost_total_monthly = (_lost_filt.groupby(['lost_year','lost_month'],dropna=False,as_index=False)
                          .agg(lost_asset_count=('asset_tag','nunique'),
                               lost_asset_cost=('lost_cost_last_price','sum'),
                               distinct_warehouses=('tech_warehouse','nunique'),
                               lost_product_types=('product_name','nunique'))
                          .rename(columns={'lost_year':'delivery_year','lost_month':'delivery_month'}))
tbl_lost_total_monthly = _add_period(tbl_lost_total_monthly).sort_values('period')

# Tech summary is built in Cell 18; gracefully skip if empty.
tbl_lost_tech_monthly = pd.DataFrame()
if 'tbl_lost_tech_attribution' in dir() and len(tbl_lost_tech_attribution)>0:
    _lt = tbl_lost_tech_attribution.copy()
    _lt['_ldt'] = pd.to_datetime(_lt['lost_date_raw'], errors='coerce')
    _lt['delivery_year']  = _lt['_ldt'].dt.year.astype('Int64')
    _lt['delivery_month'] = _lt['_ldt'].dt.month.astype('Int64')
    tbl_lost_tech_monthly = (_lt.groupby(['last_delivery_first','last_delivery_last','last_delivery_warehouse',
                                          'delivery_year','delivery_month'],dropna=False,as_index=False)
                             .agg(lost_asset_count=('asset_tag','nunique'),
                                  lost_asset_cost=('lost_cost_last_price','sum')))
    tbl_lost_tech_monthly = _add_period(tbl_lost_tech_monthly)

# --- Tech accountability monthly: per-tech weekday metric over time ---
# Driven from rollup_tech which already has delivery_year/delivery_month grain.
_acct_src = rollup_tech[rollup_tech['schedule_period']=='Weekday'].copy()
tbl_tech_accountability_monthly = (_acct_src.groupby(
        ['techfirstname','techlastname','tech_warehouse','region','vp','state','metro',
         'delivery_year','delivery_month'], dropna=False, as_index=False)
    .agg(total_tickets=('total_tickets','sum'),
         total_visits=('total_visits','sum'),
         total_workhours=('total_workhours','sum'),
         total_active_days=('total_active_days','sum'),
         weekday_days=('weekday_days','max')))
tbl_tech_accountability_monthly['avg_daily_tickets'] = (
    tbl_tech_accountability_monthly['total_tickets']
    / tbl_tech_accountability_monthly['weekday_days'].replace(0,np.nan)).round(3)
tbl_tech_accountability_monthly['tickets_per_active_day'] = (
    tbl_tech_accountability_monthly['total_tickets']
    / tbl_tech_accountability_monthly['total_active_days'].replace(0,np.nan)).round(3)
# v1.33.0 (H1): merge keyed on (tech, warehouse, month) — right side carries
# warehouse grain, so the old name+month keys fanned out multi-warehouse tech-months.
_pre_acct_m = len(tbl_tech_accountability_monthly)
tbl_tech_accountability_monthly = (tbl_tech_accountability_monthly
    .merge(tbl_tech_redeliveries_monthly[['techfirstname','techlastname','tech_warehouse','delivery_year','delivery_month',
                                          'redelivery_count','redelivery_items','redel_pct_of_tickets']],
           on=['techfirstname','techlastname','tech_warehouse','delivery_year','delivery_month'], how='left')
    .fillna({'redelivery_count':0,'redelivery_items':0}))
assert len(tbl_tech_accountability_monthly)==_pre_acct_m, 'H1 regression: monthly accountability merge multiplied rows'
tbl_tech_accountability_monthly = _add_period(tbl_tech_accountability_monthly)

# --- Productivity_Decomposition: company-level monthly trend decomposed ---
_decomp_src = tbl_grand_tpt.copy().sort_values(['delivery_year','delivery_month'])
_decomp_src['expected_tech_days'] = (_decomp_src['tech_count'].fillna(0) * _decomp_src['weekday_days'].fillna(0))
_decomp_src['mom_tickets'] = _decomp_src['total_tickets'].diff()
_decomp_src['mom_tickets_pct'] = _decomp_src['total_tickets'].pct_change().mul(100).round(1)
_decomp_src['mom_tech_count'] = _decomp_src['tech_count'].diff()
_decomp_src['mom_active_days'] = _decomp_src['total_active_days'].diff()
_decomp_src['mom_utilization_pct'] = _decomp_src['utilization_pct'].diff().round(1)
_decomp_src['mom_legacy_metric'] = _decomp_src['avg_daily_tickets_per_tech'].diff().round(3)
_decomp_src['mom_active_metric'] = _decomp_src['tickets_per_active_day_per_tech'].diff().round(3)
_now_yr, _now_mo = AS_OF_DATE.year, AS_OF_DATE.month
_decomp_src['is_partial_month'] = ((_decomp_src['delivery_year'].astype(int)==_now_yr) & (_decomp_src['delivery_month'].astype(int)==_now_mo))
_decomp_src['expected_tech_days_elapsed'] = (_decomp_src['tech_count'].fillna(0) * _decomp_src['weekday_days_elapsed'].fillna(0))
_decomp_src = _decomp_src.merge(df_data_freshness[['delivery_year','delivery_month','is_data_freshness_flag','ratio_actual_vs_expected','period_status']], on=['delivery_year','delivery_month'], how='left')
tbl_prod_decomposition = _decomp_src[['period','delivery_year','delivery_month','is_partial_month',
    'period_status','is_data_freshness_flag','ratio_actual_vs_expected',
    'total_tickets','tech_count','weekday_days','weekday_days_elapsed',
    'expected_tech_days','expected_tech_days_elapsed','total_active_days','utilization_pct',
    'tickets_per_active_day_per_tech',
    'avg_daily_tickets_per_tech','mtd_avg_daily_tickets_per_tech',
    'mom_tickets','mom_tickets_pct','mom_tech_count','mom_active_days','mom_utilization_pct',
    'mom_legacy_metric','mom_active_metric']].copy()
print(f'Monthly breakouts built: VP_Redel={len(tbl_vp_redeliveries_monthly):,} '
      f'Tech_Redel={len(tbl_tech_redeliveries_monthly):,} '
      f'Lost_VP={len(tbl_lost_vp_monthly):,} '
      f'Acct={len(tbl_tech_accountability_monthly):,} '
      f'Decomposition={len(tbl_prod_decomposition):,}')

tbl_lost_pickup_tech_summary = pd.DataFrame()
if 'tbl_lost_tech_attribution' in dir() and len(tbl_lost_tech_attribution) > 0 \
        and 'pickup_last_order' in tbl_lost_tech_attribution.columns:
    _linked = tbl_lost_tech_attribution[tbl_lost_tech_attribution['pickup_last_order'].notna()].copy()
    if len(_linked) > 0:
        _linked['lost_cost_last_price'] = pd.to_numeric(_linked['lost_cost_last_price'], errors='coerce').fillna(0.0)
        tbl_lost_pickup_tech_summary = (_linked.groupby(
            ['pickup_last_first','pickup_last_last','pickup_last_warehouse'],
            dropna=False, as_index=False)
            .agg(lost_assets_attributed=('asset_tag','nunique'),
                 lost_cost_attributed=('lost_cost_last_price','sum'),
                 distinct_patients=('record_id','nunique'),
                 avg_days_pickup_to_lost=('days_pickup_to_lost','mean'),
                 median_days_pickup_to_lost=('days_pickup_to_lost','median'))
            .sort_values('lost_cost_attributed', ascending=False))
        tbl_lost_pickup_tech_summary['avg_days_pickup_to_lost'] = tbl_lost_pickup_tech_summary['avg_days_pickup_to_lost'].round(1)
        tbl_lost_pickup_tech_summary['median_days_pickup_to_lost'] = tbl_lost_pickup_tech_summary['median_days_pickup_to_lost'].round(1)
print(f'Lost_Pickup_Tech_Summary: {len(tbl_lost_pickup_tech_summary):,} techs with attributed lost-asset exposure.')

try:
    wb=Workbook(); wb.remove(wb.active)

    # ── Productivity rollups ──────────────────────────────────────────────────
    _ws(wb,'Data_Freshness',df_data_freshness,fcol=1)
    _ws(wb,'Productivity_Decomposition',tbl_prod_decomposition,fcol=1)
    _ws(wb,'Tech_Productivity',rollup_tech,fcol=2)
    _ws(wb,'Tech_Accountability_Monthly',tbl_tech_accountability_monthly,fcol=2)
    _ws(wb,'Warehouse_Rollup',rollup_warehouse,fcol=1)
    _ws(wb,'State_Rollup',rollup_state,fcol=1)
    if len(rollup_metro)>0: _ws(wb,'Metro_Rollup',rollup_metro,fcol=1)
    _ws(wb,'Region_VP_Rollup',rollup_region,fcol=1)
    _ws(wb,'Tech_Accountability',tbl_tech_accountability,fcol=2)

    # ── Tickets per tech trend tables ─────────────────────────────────────────
    _ws(wb,'VP_Daily_Tickets',tbl_vp_tpt,fcol=1)
    _ws(wb,'WH_Daily_Tickets',tbl_wh_tpt,fcol=1)
    _ws(wb,'State_Daily_Tickets',tbl_state_tpt,fcol=1)
    if len(tbl_metro_tpt)>0:
        _ws(wb,'Metro_Daily_Tickets',tbl_metro_tpt,fcol=1)
    _ws(wb,'Total_Daily_Tickets',tbl_grand_tpt,fcol=1)

    # ── Redelivery tables ─────────────────────────────────────────────────────
    _ws(wb,'WH_Redeliveries',tbl_wh_redeliveries,fcol=1)
    _ws(wb,'State_Redeliveries',tbl_state_redeliveries,fcol=1)
    if len(tbl_metro_redeliveries)>0: _ws(wb,'Metro_Redeliveries',tbl_metro_redeliveries,fcol=1)
    _ws(wb,'VP_Redeliveries',tbl_vp_redeliveries,fcol=1)
    _ws(wb,'Total_Redeliveries',tbl_total_redeliveries,fcol=1)
    _ws(wb,'Tech_Redeliveries',tbl_tech_redeliveries,fcol=2)
    _ws(wb,'Redeliveries_Monthly',tbl_redel_monthly,fcol=1)
    _ws(wb,'Redeliveries_Monthly_WH',tbl_redel_monthly_wh,fcol=1)
    _ws(wb,'Redeliveries_Monthly_State',tbl_redel_monthly_state,fcol=1)
    if len(tbl_redel_monthly_metro)>0: _ws(wb,'Redeliveries_Monthly_Metro',tbl_redel_monthly_metro,fcol=1)
    _ws(wb,'Redeliveries_Product',tbl_redel_product,fcol=1)
    _ws(wb,'VP_Redeliveries_Monthly',tbl_vp_redeliveries_monthly,fcol=1)
    _ws(wb,'Tech_Redeliveries_Monthly',tbl_tech_redeliveries_monthly,fcol=2)
    _ws(wb,'Redeliveries_Product_Monthly',tbl_redel_product_monthly,fcol=1)
    if len(tbl_redel_unlinked_monthly)>0: _ws(wb,'Redel_Unlinked_Monthly',tbl_redel_unlinked_monthly,fcol=1)  # v1.33.0 (M2)

    # ── v1.33.0 Tech scorecard / OT / lost-recovery ──────────────────────────
    _ws(wb,'Tech_Scorecard',tbl_tech_scorecard,fcol=2)
    _ws(wb,'Tech_Scorecard_Monthly',tbl_tech_scorecard_monthly,fcol=2)
    if len(tbl_tech_lost_scorecard)>0: _ws(wb,'Tech_Lost_Scorecard',tbl_tech_lost_scorecard,fcol=2)
    if len(tbl_tech_ot_monthly_export)>0: _ws(wb,'Tech_OT_Monthly',tbl_tech_ot_monthly_export,fcol=2)
    _ws(wb,'OT_Company_Monthly',tbl_ot_company_monthly,fcol=1)

    # ── Lost equipment tables ─────────────────────────────────────────────────
    _ws(wb,'Lost_Equipment_By_WH',tbl_lost_by_wh,fcol=1)
    _ws(wb,'Lost_Equipment_By_State',tbl_lost_by_state,fcol=1)
    if len(tbl_lost_by_metro)>0: _ws(wb,'Lost_Equipment_By_Metro',tbl_lost_by_metro,fcol=1)
    _ws(wb,'Lost_Monthly_WH',tbl_lost_monthly_wh,fcol=1)
    _ws(wb,'Lost_Monthly_State',tbl_lost_monthly_state,fcol=1)
    if len(tbl_lost_monthly_metro)>0: _ws(wb,'Lost_Monthly_Metro',tbl_lost_monthly_metro,fcol=1)
    _ws(wb,'Lost_Equipment_By_VP',tbl_lost_by_vp,fcol=1)
    _ws(wb,'Lost_Equipment_Total',tbl_lost_total,fcol=1)
    _ws(wb,'Inventory_By_WH',tbl_inventory_by_wh,fcol=1)
    _ws(wb,'Inventory_By_State',tbl_inventory_by_state,fcol=1)
    if len(tbl_inventory_by_metro)>0: _ws(wb,'Inventory_By_Metro',tbl_inventory_by_metro,fcol=1)
    _ws(wb,'Inventory_By_VP',tbl_inventory_by_vp,fcol=1)
    if len(tbl_lost_tech_attribution)>0: _ws(wb,'Lost_Tech_Attribution',tbl_lost_tech_attribution,fcol=2)
    if len(tbl_lost_tech_summary)>0: _ws(wb,'Lost_Tech_Summary',tbl_lost_tech_summary,fcol=1)
    if len(tbl_lost_pickup_tech_summary)>0: _ws(wb,'Lost_Pickup_Tech_Summary',tbl_lost_pickup_tech_summary,fcol=1)
    _ws(wb,'Lost_Equipment_VP_Monthly',tbl_lost_vp_monthly,fcol=1)
    _ws(wb,'Lost_Equipment_Total_Monthly',tbl_lost_total_monthly,fcol=1)
    if len(tbl_lost_tech_monthly)>0: _ws(wb,'Lost_Tech_Summary_Monthly',tbl_lost_tech_monthly,fcol=1)

    # ── Performer rankings ────────────────────────────────────────────────────
    if len(tbl_top_techs)>0:
        _ws(wb,'Top_Performers_Tech',tbl_top_techs[['rank','techfirstname','techlastname','tech_warehouse','vp','state','metro','tickets_per_active_day','total_active_days','avg_daily_tickets_DEPRECATED','months_active','total_tickets','total_days']],fcol=1)
        _ws(wb,'Bottom_Performers_Tech',tbl_bot_techs[['rank','techfirstname','techlastname','tech_warehouse','vp','state','metro','tickets_per_active_day','total_active_days','avg_daily_tickets_DEPRECATED','months_active','total_tickets','total_days']],fcol=1)
    if len(tbl_top_wh)>0:
        _ws(wb,'Top_Performers_WH',tbl_top_wh[['rank','tech_warehouse','region','vp','state','metro','tickets_per_active_day_per_tech','months_active','total_tickets','total_tech_days']],fcol=1)
        _ws(wb,'Bottom_Performers_WH',tbl_bot_wh[['rank','tech_warehouse','region','vp','state','metro','tickets_per_active_day_per_tech','months_active','total_tickets','total_tech_days']],fcol=1)

    # ── Intops ────────────────────────────────────────────────────────────────
    if len(tbl_intops_monthly)>0:
        _io_cols=['period','tech_warehouse','region','vp','state','metro','delivery_year','delivery_month','schedule_period','ticket_type','reason_category','ticket_count','patient_count','roles_involved','tech_names']
        _ws(wb,'Internal_Ops_Tickets',tbl_intops_monthly[[c for c in _io_cols if c in tbl_intops_monthly.columns]],fcol=1)
        _ws(wb,'Internal_Ops_By_WH',tbl_intops_by_wh,fcol=1)
    if len(tbl_intops_tpt_vp)>0: _ws(wb,'IntOps_VP_Daily',tbl_intops_tpt_vp,fcol=1)
    if len(tbl_intops_tpt_wh)>0: _ws(wb,'IntOps_WH_Daily',tbl_intops_tpt_wh,fcol=1)

    # ── Unmatched names ───────────────────────────────────────────────────────
    if len(tbl_unmatched_candidates)>0:
        _ws(wb,'Unmatched_Name_Candidates',tbl_unmatched_candidates[['unmatched_full','unmatched_first','unmatched_last','ticket_count','patient_count','tx_warehouses','rank','candidate_name','candidate_title','candidate_dept','candidate_loc','candidate_eid','strategy','score']],fcol=1)

    path=os.path.join(OUT_DIR,f'TechWorkload_Report_{RUN_DATE}.xlsx')
    wb.save(path); print(f'Excel saved: {path}')
    print(f'  Sheets: {len(wb.sheetnames)}')
    for s in wb.sheetnames: print(f'    {s}')
except Exception as e:
    print(f'Excel export FAILED: {e}'); raise

# HR workbook (HIPAA)
if '_unmatched_ticket_detail' in dir() and len(_unmatched_ticket_detail)>0:
    try:
        wb_hr=Workbook(); wb_hr.remove(wb_hr.active)
        _ws(wb_hr,'Unmatched_Ticket_Detail',_unmatched_ticket_detail,fcol=2)
        _ws(wb_hr,'Unmatched_Name_Summary',tbl_unmatched_candidates[['unmatched_full','unmatched_first','unmatched_last','ticket_count','patient_count','tx_warehouses','rank','candidate_name','candidate_title','candidate_dept','candidate_loc','candidate_eid','strategy','score']],fcol=1)
        _ws(wb_hr,'Resolved_Recoveries',tbl_resolved_recoveries,fcol=1)
        _ws(wb_hr,'Ambiguous_Collisions',tbl_ambiguous_collisions,fcol=1)
        hr_path=os.path.join(OUT_DIR,f'Unmatched_Techs_HR_{RUN_DATE}.xlsx')
        wb_hr.save(hr_path); print(f'HR export: {hr_path}  HIPAA — internal use only.')
    except Exception as e:
        print(f'HR export FAILED: {e}')

print(f'\nRun complete: {RUN_DATE}  |  Output: {OUT_DIR}')
try:
    sql_conn.close(); print('DB connection closed.')
except Exception:
    pass